# Onde a riva — Definizione dei siti di studio

**Un’area di studio, con eventuali settori costieri.**

Input dell’area: **DEM e GeoJSON dei transetti**; l’**ortofotomosaico è facoltativo**. Puoi aggiungere un GeoJSON facoltativo dei **punti delle strutture**, con campo `struttura`, per proporre quote medie dal DEM. Il GeoJSON dei poligoni dei **settori costieri è facoltativo**: senza poligoni si rappresenta la spiaggia intera. Non occorre preparare CSV di punti o raster ritagliati.

| Comuni all’area | Specifici del settore costiero |
|---|---|
| DEM e ortofoto sorgenti; direzioni di esposizione | Transetti assegnati e relative pendenze |
| Punto di previsione di onde, surge e marea | Strutture esposte presenti e loro quote |
| Punto e serie storica ERA5 | Risultati del runup/TWL e ritagli della mappa |

Il percorso è unico: file → profili e pendenze → settori costieri → strutture esposte → ritagli → ERA5 comune → esportazione. I dati di previsione vengono scaricati dal servizio giornaliero, non moltiplicati per settore costiero.

**Il notebook è autosufficiente:** le funzioni del progetto sono incluse nella cella tecnica; non occorre clonare GitHub o pubblicare il sito. Parti in modalità dimostrativa per vedere tutti i passaggi senza credenziali. Per i tuoi dati disattiva `DEMO` e usa le celle in ordine.

Per i profili estratti dal DEM, la pendenza segue **Stockdon_runup_soluzione.ipynb**, celle 24–25: orientamento mare→terra dalla media delle prime/ultime 10 quote; distanza euclidea X/Y dal primo punto a mare; selezione prima del primo punto sopra `Z_MAX`; regressione con `np.polyfit`. Per i transetti GNSS XYZ già rilevati sulla battigia si usa invece l’intero transetto, senza selezione Zmin/Zmax; si può ricalcolare la pendenza oppure usare un campo già calcolato, scelto esplicitamente. Le formule di runup del portale restano invariate. Le pendenze misurate alimentano poi il campionamento Monte Carlo per ciascun settore (oppure per la spiaggia intera).

Il pacchetto finale non contiene token o credenziali. Il notebook non pubblica e non modifica automaticamente la repository.

## 1. Ambiente e cartella di lavoro
Esegui le tre celle in ordine. **Installa le librerie** prepara il runtime CPU; **Carica le funzioni** contiene il codice tecnico già pronto e non richiede modifiche. Può impiegare qualche minuto: attendi il messaggio finale.

In **Modalità e salvataggio** disattiva `DEMO` per i tuoi rilievi. Attiva Google Drive per riutilizzare file e cache fra sessioni; indica soltanto il nome della cartella dentro “Il mio Drive”. Le celle mostrano moduli: non occorre aprire il codice.

In [ ]:
#@title Installa le librerie
INSTALLA_DIPENDENZE = True #@param {type:"boolean"}
if INSTALLA_DIPENDENZE:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "numpy>=1.24", "pandas>=2", "matplotlib>=3.7", "folium>=0.14", "pyproj>=3.6", "cdsapi>=0.7.7,<1", "netCDF4>=1.7,<2", "scipy>=1.11,<2", "copernicusmarine==2.4.1", "rasterio>=1.3,<2", "shapely>=2,<3", "ipywidgets>=8,<9"])
print("Ambiente pronto.")

In [ ]:
#@title Carica le funzioni del progetto (cella tecnica)
# Codice condiviso con la repository, incluso per funzionare anche prima del push.
from pathlib import Path
import json, sys, tempfile, hashlib
RUNTIME = Path(tempfile.mkdtemp(prefix="ondeariva_runtime_"))
BUNDLED_SOURCES = {'forecast_service/__init__.py': '"""Calcolo e pubblicazione del portale locale, indipendente da Streamlit."""\n', 'forecast_service/site_setup.py': '"""Beach-face slope following Stockdon_runup_soluzione.ipynb, cells 24–25.\n\nThe supplied teaching notebook defines selection and orientation. Diagnostics\nreject degenerate fits; a negative slope is not converted to an absolute value.\n"""\nimport math\n\nBEACHFACE_METHOD_VERSION = \'stockdon_teaching_notebook_v1\'\n\n\ndef prepare_profile(frame, *, distance_col=\'distance\', elevation_col=\'SAMPLE_1\',\n                    x_col=\'X\', y_col=\'Y\', orientation=\'auto\'):\n    import numpy as np\n    import pandas as pd\n    if orientation not in (\'auto\',\'distance_increases_landward\',\'distance_increases_seaward\'):\n        raise ValueError(\'Orientamento non riconosciuto.\')\n    spatial = x_col is not None and y_col is not None\n    if (x_col is None) != (y_col is None):\n        raise ValueError(\'Specificare entrambe le colonne X/Y oppure nessuna.\')\n    required=[distance_col,elevation_col]+([x_col,y_col] if spatial else [])\n    missing=set(required)-set(frame.columns)\n    if missing:\n        raise ValueError(f\'Colonne mancanti: {sorted(missing)}\')\n    profile=frame.copy()\n    for name in required:\n        profile[name]=pd.to_numeric(profile[name],errors=\'raise\')\n    dropped=int(profile[elevation_col].isna().sum())\n    profile=profile.dropna(subset=[elevation_col]).sort_values(distance_col)\n    if len(profile)<2 or not np.isfinite(profile[required].to_numpy(dtype=float)).all():\n        raise ValueError(\'Servono almeno due punti con distanze, coordinate e quote finite.\')\n    if profile[distance_col].duplicated().any():\n        raise ValueError(\'Distanze duplicate nel transetto: verificare il CSV.\')\n    sea_at_end=bool(profile[elevation_col].iloc[:10].mean()>profile[elevation_col].iloc[-10:].mean())\n    ambiguous=bool(abs(profile[elevation_col].iloc[:10].mean()-profile[elevation_col].iloc[-10:].mean())<1e-12)\n    flip=sea_at_end if orientation==\'auto\' else orientation==\'distance_increases_seaward\'\n    if flip:\n        profile=profile.iloc[::-1]\n    profile=profile.reset_index(drop=True)\n    if spatial:\n        profile[\'x_mare\']=np.hypot(profile[x_col]-profile[x_col].iloc[0],profile[y_col]-profile[y_col].iloc[0])\n    else:\n        # Compatibility with 1-D profiles: declared separately in the metadata.\n        profile[\'x_mare\']=(profile[distance_col]-profile[distance_col].iloc[0]).abs()\n    profile.attrs.update(reversed=bool(flip),orientation=orientation,orientation_ambiguous=ambiguous,\n                         dropped_nan_z=dropped,distance_source=\'euclidean_xy\' if spatial else \'one_dimensional_distance\')\n    return profile\n\n\ndef fit_beachface(profile,z_min,z_max,*,elevation_col=\'SAMPLE_1\'):\n    import numpy as np\n    if not math.isfinite(z_min) or not math.isfinite(z_max) or z_min>=z_max:\n        raise ValueError(\'Definire un intervallo di quote crescente.\')\n    # Same sea-connected prefix and regression as pendenza_battigia in cell 25.\n    above=np.where(profile[elevation_col]>z_max)[0]\n    stop=int(above[0]) if len(above) else len(profile)\n    selected=(profile[elevation_col]>=z_min)&(profile.index<stop)\n    x=profile.loc[selected,\'x_mare\'].to_numpy(dtype=float)\n    z=profile.loc[selected,elevation_col].to_numpy(dtype=float)\n    if len(x)<2 or len(np.unique(x))<2:\n        raise ValueError(\'Meno di due distanze distinte prima del primo punto sopra Z_MAX.\')\n    slope,intercept=np.polyfit(x,z,1)\n    if not math.isfinite(slope) or not 0<slope<1:\n        raise ValueError(f\'Pendenza {slope:.6g} non valida: controllare orientamento e tratto di battigia.\')\n    variance=float(np.sum((z-z.mean())**2))\n    r2=1-float(np.sum((z-(slope*x+intercept))**2))/variance if variance>0 else 0.0\n    diagnostics={\'slope\':float(slope),\'n_points\':int(len(x)),\'r_squared\':r2,\'fit_intercept_m\':float(intercept),\n                 \'z_min_m\':float(z_min),\'z_max_m\':float(z_max),\'stop_index\':stop,\n                 \'first_above_zmax_found\':bool(len(above)),\'selected_x_min_m\':float(x.min()),\n                 \'selected_x_max_m\':float(x.max()),\'method\':BEACHFACE_METHOD_VERSION,**profile.attrs}\n    return diagnostics,selected\n\n\ndef analyze_transects(frame,z_min,z_max,*,id_col=\'TR_ID\',distance_col=\'distance\',elevation_col=\'SAMPLE_1\',\n                      x_col=\'X\',y_col=\'Y\',orientation=\'auto\'):\n    import pandas as pd\n    if id_col not in frame or frame[id_col].isna().any():\n        raise ValueError(\'Colonna identificativo assente o con valori mancanti.\')\n    summaries=[]; profiles={}; selections={}\n    for identifier,group in frame.groupby(id_col,sort=False):\n        identifier=str(identifier)\n        try:\n            p=prepare_profile(group,distance_col=distance_col,elevation_col=elevation_col,\n                              x_col=x_col,y_col=y_col,orientation=orientation)\n            summary,selected=fit_beachface(p,z_min,z_max,elevation_col=elevation_col)\n            summaries.append(dict(summary,transect_id=identifier,status=\'valid\',error=\'\'))\n            profiles[identifier]=p; selections[identifier]=selected\n        except (ValueError,KeyError,TypeError) as error:\n            summaries.append({\'transect_id\':identifier,\'status\':\'invalid\',\'error\':str(error),\'slope\':None})\n    return pd.DataFrame(summaries),profiles,selections\n\n\ndef slopes_from_transects(path,z_min,z_max):\n    import pandas as pd\n    frame=pd.read_csv(path)\n    if {\'TR_ID\',\'distance\',\'SAMPLE_1\',\'X\',\'Y\'}<=set(frame):\n        options={}\n    elif {\'transect_id\',\'x_m\',\'z_m\'}<=set(frame):\n        options=dict(id_col=\'transect_id\',distance_col=\'x_m\',elevation_col=\'z_m\',x_col=None,y_col=None)\n    else:\n        raise ValueError(\'CSV QGIS: TR_ID,distance,SAMPLE_1,X,Y; oppure CSV 1-D: transect_id,x_m,z_m.\')\n    summary,_,_=analyze_transects(frame,z_min,z_max,**options)\n    if summary.empty:\n        raise ValueError(\'Il CSV non contiene transetti.\')\n    invalid=summary[summary.status!=\'valid\']\n    if not invalid.empty:\n        raise ValueError(\'; \'.join(f"{r.transect_id}: {r.error}" for r in invalid.itertuples()))\n    return summary.drop(columns=[\'status\',\'error\']).to_dict(\'records\')\n', 'forecast_service/time_utils.py': '"""Pure helpers shared with the standalone Colab setup."""\nimport math\nfrom datetime import datetime,timezone\n\n\ndef iso(dt):\n    return dt.astimezone(timezone.utc).isoformat().replace(\'+00:00\',\'Z\')\n\n\ndef parse_time(value):\n    result=value if isinstance(value,datetime) else datetime.fromisoformat(str(value).replace(\'Z\',\'+00:00\'))\n    return result.replace(tzinfo=timezone.utc) if result.tzinfo is None else result.astimezone(timezone.utc)\n\n\ndef quantile(values,q):\n    ordered=sorted(values)\n    position=(len(ordered)-1)*q\n    low,high=math.floor(position),math.ceil(position)\n    return ordered[low]+(ordered[high]-ordered[low])*(position-low)\n', 'forecast_service/era5_download.py': '"""ERA5 adapter based on wave_climate/download.py, Alessio Rovere (MIT, 2026).\n\nKeep request caching, NetCDF/ZIP handling, nearest sea cell and adaptive chunks.\nOnly waves are needed here. Mean period is never renamed to peak period.\nSee LICENSE-wave-climate.txt for the upstream license.\n"""\nimport datetime as dt\nimport hashlib\nimport io\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport time\nimport zipfile\n\nERA5_TS = \'reanalysis-era5-single-levels-timeseries\'\nVARIABLES = [\'significant_height_of_combined_wind_waves_and_swell\', \'mean_wave_period\', \'mean_wave_direction\']\nSHORT = {\'swh\':\'hs_m\', \'mwp\':\'tm_s\', \'mwd\':\'direction_deg\',\n         \'significant_height_of_combined_wind_waves_and_swell\':\'hs_m\',\n         \'mean_wave_period\':\'tm_s\', \'mean_wave_direction\':\'direction_deg\'}\n\n\nclass TooLarge(Exception):\n    pass\n\n\ndef cds_client():\n    import cdsapi\n    key = os.environ.get(\'CDS_KEY\')\n    if key:\n        return cdsapi.Client(url=os.environ.get(\'CDS_URL\',\'https://cds.climate.copernicus.eu/api\'),key=key,quiet=True)\n    return cdsapi.Client(quiet=True)  # Also supports the user\'s standard ~/.cdsapirc.\n\n\ndef open_cds(path):\n    import xarray as xr\n    if zipfile.is_zipfile(path):\n        frames = []\n        with zipfile.ZipFile(path) as archive:\n            for name in archive.namelist():\n                if name.endswith(\'.nc\'):\n                    with xr.open_dataset(io.BytesIO(archive.read(name))) as ds:\n                        frames.append(ds.load())\n        if not frames:\n            raise ValueError(\'Archivio CDS senza file NetCDF.\')\n        return xr.merge(frames,compat=\'override\')\n    with xr.open_dataset(path) as ds:\n        return ds.load()\n\n\ndef ts_frame(ds):\n    import pandas as pd\n    df=ds.to_dataframe().reset_index()\n    tcol=\'valid_time\' if \'valid_time\' in df else \'time\'\n    keep={v:SHORT[v] for v in ds.data_vars if v in SHORT}\n    if \'hs_m\' not in keep.values():\n        raise ValueError(\'La risposta CDS non contiene Hs.\')\n    out=df[[tcol,*keep,*[c for c in (\'latitude\',\'longitude\') if c in df]]].rename(columns={tcol:\'time\',**keep})\n    out[\'time\']=pd.to_datetime(out[\'time\'],utc=True)\n    return out\n\n\ndef cached_fetch(client, dataset, request, cache, retries=3):\n    cache=Path(cache); cache.mkdir(parents=True,exist_ok=True)\n    key=hashlib.sha256(json.dumps({\'dataset\':dataset,**request},sort_keys=True).encode()).hexdigest()[:16]\n    path=cache/f\'era5_{key}.nc\'\n    if path.exists():\n        return path\n    partial=path.with_suffix(\'.part\')\n    for attempt in range(retries):\n        try:\n            client.retrieve(dataset,request,str(partial))\n            ds=open_cds(partial)\n            try:\n                ts_frame(ds)\n            finally:\n                ds.close()\n            partial.replace(path)\n            return path\n        except Exception as error:\n            partial.unlink(missing_ok=True)\n            message=str(error).lower()\n            if any(word in message for word in (\'too large\',\'cost limit\',\'request too big\')):\n                raise TooLarge() from error\n            if attempt==retries-1:\n                raise\n            time.sleep(30*(attempt+1))\n\n\ndef ts_range(client, lon, lat, start, end, cache, retries=3):\n    request={\'variable\':sorted(VARIABLES),\'location\':{\'longitude\':round(lon,4),\'latitude\':round(lat,4)},\n             \'date\':[f\'{start.isoformat()}/{end.isoformat()}\'],\'data_format\':\'netcdf\'}\n    try:\n        return [cached_fetch(client,ERA5_TS,request,cache,retries)]\n    except TooLarge:\n        if (end-start).days<=31:\n            raise ValueError(\'Richiesta CDS troppo grande anche dopo la suddivisione.\')\n        middle=start+dt.timedelta(days=(end-start).days//2)\n        return ts_range(client,lon,lat,start,middle,cache,retries)+ts_range(client,lon,lat,middle+dt.timedelta(days=1),end,cache,retries)\n\n\ndef nearest_sea_cell(client, lon, lat, cache, search_deg=.75):\n    # CDS time-series accepts a location. Use the gridded product for the area\n    # probe, instead of sending the upstream script\'s area to the point service.\n    request={\'product_type\':[\'reanalysis\'],\'variable\':[VARIABLES[0]],\'year\':[\'2020\'],\'month\':[\'01\'],\n             \'day\':[\'01\'],\'time\':[f\'{h:02d}:00\' for h in range(24)],\n             \'area\':[lat+search_deg,lon-search_deg,lat-search_deg,lon+search_deg],\n             \'data_format\':\'netcdf\',\'download_format\':\'unarchived\'}\n    ds=open_cds(cached_fetch(client,\'reanalysis-era5-single-levels\',request,cache))\n    try:\n        frame=ts_frame(ds)\n    finally:\n        ds.close()\n    cells=frame.groupby([\'latitude\',\'longitude\'])[\'hs_m\'].apply(lambda v:v.notna().mean()).reset_index()\n    cells=cells[cells.hs_m>.9]\n    if cells.empty:\n        raise ValueError(\'Nessuna cella marina ERA5 vicina: scegliere un punto più al largo.\')\n    delta=((cells.longitude-lon+180)%360)-180\n    distance=(delta*math.cos(math.radians(lat)))**2+(cells.latitude-lat)**2\n    cell=cells.loc[distance.idxmin()]\n    return float((cell.longitude+180)%360-180),float(cell.latitude)\n\n\ndef year_chunks(start,end,years=10):\n    if end<start or years<1:\n        raise ValueError(\'Periodo ERA5 non valido.\')\n    chunks=[]\n    current=start\n    while current<=end:\n        stop=min(dt.date(current.year+years-1,12,31),end)\n        chunks.append((current,stop))\n        current=stop+dt.timedelta(days=1)\n    return chunks\n\n\ndef download_series(location, start, end, cache, *, client=None, cell=None):\n    import pandas as pd\n    if end>dt.datetime.now(dt.timezone.utc).date()-dt.timedelta(days=6):\n        raise ValueError(\'Usare una data finale almeno sei giorni prima di oggi per ERA5.\')\n    client=client or cds_client()\n    point=location[\'wave_point\']\n    lon,lat=cell or nearest_sea_cell(client,point[\'longitude\'],point[\'latitude\'],cache)\n    frames=[]\n    for a,b in year_chunks(start,end):\n        print(f\'ERA5 {a} → {b}, cella {lat:.2f} N {lon:.2f} E (cache delle richieste attiva)\')\n        for path in ts_range(client,lon,lat,a,b,cache):\n            ds=open_cds(path)\n            try:\n                frames.append(ts_frame(ds))\n            finally:\n                ds.close()\n    frame=pd.concat(frames).drop_duplicates(\'time\',keep=\'last\').sort_values(\'time\')\n    frame=frame[(frame.time>=pd.Timestamp(start,tz=\'UTC\')) & (frame.time<pd.Timestamp(end+dt.timedelta(days=1),tz=\'UTC\'))]\n    # Reject mixed cells: a local reference must represent one known marine cell.\n    if any(c in frame and frame[c].nunique(dropna=True)>1 for c in (\'latitude\',\'longitude\')):\n        raise ValueError(\'Il CDS ha restituito più celle per la serie puntuale.\')\n    requested_count=(end-start).days*24+24\n    valid=frame[\'hs_m\'].notna()\n    info={\'dataset_id\':ERA5_TS,\'doi\':\'10.24381/1cf1ad76\',\'cell_lon\':lon,\'cell_lat\':lat,\n          \'requested_start\':str(start),\'requested_end\':str(end),\'requested_hours\':requested_count,\n          \'returned_records\':len(frame),\'valid_hs_records\':int(valid.sum()),\n          \'missing_hs_records\':requested_count-int(valid.sum()),\n          \'period_type\':\'mean_wave_period_not_peak\',\'downloaded_at\':dt.datetime.now(dt.timezone.utc).isoformat()}\n    if not valid.any():\n        raise ValueError(\'La cella ERA5 non contiene altezze d’onda valide.\')\n    return frame,info\n', 'forecast_service/climate.py': '"""Offline historical reference, reused by the two daily forecast updates."""\nfrom bisect import bisect_right\nfrom pathlib import Path\nimport csv\nimport hashlib\nimport json\nimport math\n\n\ndef build_climate(csv_path, source, *, time_column=\'time\', hs_column=\'hs_m\'):\n    from forecast_service.time_utils import iso, parse_time, quantile\n    rows = []\n    with Path(csv_path).open(encoding=\'utf-8-sig\', newline=\'\') as handle:\n        for row in csv.DictReader(handle):\n            value = float(row[hs_column])\n            if not math.isfinite(value) or value < 0:\n                raise ValueError(\'La serie storica contiene Hs mancanti o negative: pulire i dati esplicitamente.\')\n            rows.append((parse_time(row[time_column]), value))\n    rows.sort()\n    if len(rows) < 2 or len({t for t,_ in rows}) != len(rows):\n        raise ValueError(\'Servono almeno due tempi distinti, senza duplicati.\')\n    values = sorted(value for _,value in rows)\n    annual = {}\n    for time, value in rows:\n        year = str(time.year)\n        if year not in annual or value > annual[year][\'hs_m\']:\n            annual[year] = {\'hs_m\': value, \'time\': iso(time)}\n    gaps = [(b[0]-a[0]).total_seconds()/3600 for a,b in zip(rows,rows[1:])]\n    step = quantile(gaps, .5)\n    # Irregular sampling would bias an observation-count percentile.\n    if any(abs(g/step-round(g/step)) > .001 for g in gaps) or step <= 0:\n        raise ValueError(\'Campionamento non regolare: uniformare la serie prima del confronto climatologico.\')\n    return {\'schema_version\':1, \'source\':source, \'units\':\'m\', \'start\':iso(rows[0][0]), \'end\':iso(rows[-1][0]),\n            \'n_records\':len(rows), \'time_step_hours\':step,\n            \'coverage_fraction\':len(rows)/(int(round((rows[-1][0]-rows[0][0]).total_seconds()/3600/step))+1),\n            \'hs_sorted_m\':values, \'p95_m\':quantile(values,.95), \'p99_m\':quantile(values,.99),\n            \'max_hs_m\':max(values), \'annual_maxima\':annual,\n            \'input_sha256\':hashlib.sha256(Path(csv_path).read_bytes()).hexdigest(),\n            \'method\':\'Empirical CDF, all seasons and directions; annual maxima are observations, not return levels.\'}\n\n\ndef load_climate(root, relative):\n    path = (root / relative).resolve()\n    if not path.is_relative_to(root.resolve()):\n        raise ValueError(\'Il riferimento climatologico deve essere nella repository.\')\n    data = json.loads(path.read_text())\n    values = data[\'hs_sorted_m\']\n    if data[\'schema_version\'] != 1 or data[\'units\'] != \'m\' or not values or values != sorted(values) or any(not math.isfinite(v) or v < 0 for v in values):\n        raise ValueError(\'Riferimento climatologico non valido.\')\n    if len(values) != data[\'n_records\']:\n        raise ValueError(\'Conteggio della serie storica non coerente.\')\n    from forecast_service.extremes import validate_directional_extremes\n    validate_directional_extremes(data.get(\'directional_extremes\'))\n    return data\n\n\ndef compare_waves(waves, climate):\n    if not climate:\n        return waves, {\'status\':\'not_configured\', \'description\':\'Climatologia non ancora caricata per questa località.\'}\n    values = climate[\'hs_sorted_m\']\n    rows = [dict(wave, hs_percentile=100*bisect_right(values,wave[\'hs_m\'])/len(values)) for wave in waves]\n    summary = {key:value for key,value in climate.items() if key != \'hs_sorted_m\'}\n    summary[\'status\'] = \'available\'\n    summary[\'comparison_note\'] = \'Confronto indicativo fra previsione Copernicus e serie storica; eventuali bias fra prodotti non sono corretti. Il percentile non è un periodo di ritorno.\'\n    from forecast_service.extremes import compare_directional_extremes\n    rows = compare_directional_extremes(rows, climate.get(\'directional_extremes\'))\n    return rows, summary\n', 'forecast_service/exposure.py': '"""User-defined sectors of incoming waves; no nearshore wave transformation."""\nimport math\n\n\ndef is_exposed(direction, exposure):\n    if not exposure or not exposure.get(\'sectors_deg\') or not isinstance(direction, (int, float)) or not math.isfinite(direction):\n        return None\n    direction %= 360\n    return any((a <= direction <= b or b == 360 and direction == 0) if a <= b else (direction >= a or direction <= b)\n               for a, b in exposure[\'sectors_deg\'])\n\n\ndef validate_exposure(exposure):\n    if exposure is None:\n        return\n    if not isinstance(exposure, dict) or not isinstance(exposure.get(\'sectors_deg\'), list) or not exposure[\'sectors_deg\']:\n        raise ValueError(\'Esposizione: indicare almeno un settore oppure lasciare il campo non configurato.\')\n    if not isinstance(exposure.get(\'note\', \'\'), str):\n        raise ValueError(\'La nota sull’esposizione deve essere un testo.\')\n    for pair in exposure[\'sectors_deg\']:\n        if not isinstance(pair, (list, tuple)) or len(pair) != 2:\n            raise ValueError(\'Ogni settore deve contenere due angoli: da, a.\')\n        if any(isinstance(v, bool) or not isinstance(v, (int, float)) or not math.isfinite(v) for v in pair):\n            raise ValueError(\'Gli angoli di esposizione devono essere numeri finiti.\')\n        start, end = pair\n        if not 0 <= start < 360 or not 0 <= end <= 360 or start == end:\n            raise ValueError(\'Settori non validi: inizio da 0 a meno di 360, fine da 0 a 360; estremi distinti. Per tutte le direzioni usare 0,360.\')\n\n\ndef parse_exposure(text, note=\'\'):\n    """Parse clockwise incoming sectors, e.g. \'315,45; 90,180\'."""\n    if not text.strip():\n        return None\n    try:\n        sectors = [[float(value.strip()) for value in sector.split(\',\')] for sector in text.split(\';\')]\n    except ValueError as error:\n        raise ValueError(\'Usare coppie di gradi separate da virgola e settori separati da punto e virgola, es. 315,45; 90,180.\') from error\n    exposure = {\'sectors_deg\': sectors, \'note\': note}\n    validate_exposure(exposure)\n    return exposure\n', 'forecast_service/extremes.py': '"""Directional annual-maxima Gumbel fits, prepared offline during site setup.\n\nDirections are wave provenance, clockwise from true North. Statistical sectors\nform a half-open partition and are distinct from beach exposure sectors.\n"""\nimport calendar\nimport hashlib\nimport math\nfrom pathlib import Path\n\n\ndef sector_index(direction, width_deg):\n    if not isinstance(width_deg, (int, float)) or width_deg not in (30, 45, 60, 90, 120, 180, 360):\n        raise ValueError(\'Ampiezza dei settori ammessa: 30, 45, 60, 90, 120, 180 o 360 gradi.\')\n    if direction is None or not math.isfinite(direction):\n        return None\n    return int(((direction + width_deg/2) % 360)//width_deg)\n\n\ndef gumbel_return_period(height, loc, scale):\n    """Annual-sector exceedance recurrence, not hourly or conditional frequency."""\n    if not all(math.isfinite(v) for v in (height, loc, scale)) or scale <= 0:\n        raise ValueError(\'Parametri Gumbel non validi.\')\n    z=(height-loc)/scale\n    if z < -700:\n        return 1.0\n    if z > 700:\n        return None\n    exceedance=-math.expm1(-math.exp(-z))\n    value=1/exceedance\n    return value if math.isfinite(value) else None\n\n\ndef fit_gumbel(annual_maxima, *, min_years=30, bootstrap_replicates=400, seed=20261105):\n    import numpy as np\n    from scipy.stats import gumbel_r\n    if not isinstance(min_years, int) or min_years < 20:\n        raise ValueError(\'Per questa versione usare almeno 20 anni; il valore predefinito è 30.\')\n    if not isinstance(bootstrap_replicates, int) or bootstrap_replicates < 100:\n        raise ValueError(\'Servono almeno 100 repliche bootstrap.\')\n    values=np.asarray([row[\'hs_m\'] for row in annual_maxima],dtype=float)\n    n=len(values)\n    if len({row[\'year\'] for row in annual_maxima})!=n or not np.isfinite(values).all() or (values<0).any():\n        raise ValueError(\'Massimi annuali non validi o anni duplicati.\')\n    if n < min_years:\n        return {\'status\':\'insufficient_years\',\'n_years\':n,\'reason\':f\'Servono almeno {min_years} anni completi; ne sono disponibili {n}.\'}\n    if np.ptp(values)<=1e-12:\n        return {\'status\':\'constant_maxima\',\'n_years\':n,\'reason\':\'Massimi annuali costanti: non è stimabile una scala Gumbel positiva.\'}\n    loc,scale=map(float,gumbel_r.fit(values))\n    if not math.isfinite(loc) or not math.isfinite(scale) or scale<=0:\n        raise ValueError(\'Il fit Gumbel non produce parametri finiti e scala positiva.\')\n    periods=np.unique(np.r_[np.geomspace(1.01,100,80), [float(n)] if n<=100 else []])\n    central=gumbel_r.isf(1/periods,loc=loc,scale=scale)\n    rng=np.random.default_rng(seed)\n    boot=[]\n    for _ in range(bootstrap_replicates):\n        sample=rng.choice(values,size=n,replace=True)\n        if np.ptp(sample)<=1e-12:\n            continue\n        a,b=map(float,gumbel_r.fit(sample))\n        curve=gumbel_r.isf(1/periods,loc=a,scale=b)\n        if b>0 and np.isfinite(curve).all():\n            boot.append(curve)\n    if len(boot)<.9*bootstrap_replicates:\n        return {\'status\':\'unstable_bootstrap\',\'n_years\':n,\'reason\':\'Fit bootstrap troppo instabile per mostrare una curva con intervalli.\'}\n    low,high=np.quantile(boot,[.025,.975],axis=0)\n    ordered=sorted(annual_maxima,key=lambda row:row[\'hs_m\'],reverse=True)\n    return {\'status\':\'available\',\'distribution\':\'gumbel_r\',\'fit_method\':\'maximum_likelihood\',\n            \'loc_m\':loc,\'scale_m\':scale,\'n_years\':n,\'min_years\':min_years,\n            \'first_year\':min(row[\'year\'] for row in annual_maxima),\'last_year\':max(row[\'year\'] for row in annual_maxima),\n            \'return_period_years\':periods.tolist(),\'return_level_m\':central.tolist(),\n            \'ci95_low_m\':low.tolist(),\'ci95_high_m\':high.tolist(),\n            \'bootstrap_replicates\':len(boot),\'bootstrap_seed\':seed,\n            \'empirical\':[dict(row,return_period_years=(n+1)/rank) for rank,row in enumerate(ordered,1)]}\n\n\ndef build_directional_extremes(csv_path, *, time_column=\'time\', hs_column=\'hs_m\', direction_column=\'direction_deg\',\n                              sector_width_deg=45, min_years=30, bootstrap_replicates=400):\n    import numpy as np\n    import pandas as pd\n    sector_index(0,sector_width_deg)\n    path=Path(csv_path)\n    columns=pd.read_csv(path,nrows=0).columns\n    if direction_column not in columns:\n        return {\'status\':\'not_configured\',\'reason\':f\'Colonna delle direzioni {direction_column!r} assente: il fit settoriale richiede data, Hs e direzione.\'}\n    frame=pd.read_csv(path,usecols=[time_column,hs_column,direction_column]).rename(\n        columns={time_column:\'time\',hs_column:\'hs_m\',direction_column:\'direction_deg\'})\n    frame[\'time\']=pd.to_datetime(frame.time,utc=True,format=\'mixed\')\n    frame=frame.sort_values(\'time\')\n    if frame.empty or frame.time.duplicated().any():\n        raise ValueError(\'La serie degli estremi deve avere tempi distinti e non essere vuota.\')\n    if ((frame.time.dt.minute!=0)|(frame.time.dt.second!=0)|(frame.time.dt.microsecond!=0)).any():\n        return {\'status\':\'unsupported_sampling\',\'reason\':\'Per il fit settoriale usare la serie oraria ERA5 con campioni sulle ore intere.\'}\n    for name in (\'hs_m\',\'direction_deg\'):\n        frame[name]=pd.to_numeric(frame[name],errors=\'coerce\')\n    hs_valid=np.isfinite(frame.hs_m)&(frame.hs_m>=0)\n    direction_valid=np.isfinite(frame.direction_deg)&frame.direction_deg.between(0,360)\n    frame[\'valid\']=hs_valid&((frame.hs_m==0)|direction_valid)\n    frame[\'year\']=frame.time.dt.year\n    width=sector_width_deg\n    frame[\'sector\']=np.floor(((frame.direction_deg+width/2)%360)/width)\n    annual=[]; included=[]\n    for year in range(int(frame.year.min()),int(frame.year.max())+1):\n        group=frame[frame.year==year]\n        expected=8784 if calendar.isleap(year) else 8760\n        valid=int(group.valid.sum())\n        complete=valid==expected\n        annual.append({\'year\':year,\'valid_hours\':valid,\'expected_hours\':expected,\'coverage_fraction\':valid/expected,\'included\':complete})\n        if complete:\n            included.append(year)\n    sectors=[]\n    for index in range(round(360/width)):\n        center=index*width\n        sector={\'id\':f\'sector-{index}\',\'start_deg\':(center-width/2)%360,\'end_deg\':(center+width/2)%360,\n                \'center_deg\':center,\'width_deg\':width,\'annual_maxima\':[]}\n        for year in included:\n            group=frame[(frame.year==year)&frame.valid&(frame.hs_m>0)&(frame.sector==index)]\n            if not group.empty:\n                peak=group.loc[group.hs_m.idxmax()]\n                sector[\'annual_maxima\'].append({\'year\':year,\'hs_m\':float(peak.hs_m),\'time\':peak.time.isoformat().replace(\'+00:00\',\'Z\')})\n        missing=sorted(set(included)-{row[\'year\'] for row in sector[\'annual_maxima\']})\n        if missing:\n            sector[\'fit\']={\'status\':\'years_without_sector_waves\',\'n_years\':len(sector[\'annual_maxima\']),\n                           \'reason\':\'Alcuni anni completi non hanno onde positive da questo settore: escluderli condizionerebbe il tempo di ritorno. Fit non mostrato.\',\n                           \'years_without_waves\':missing}\n        else:\n            sector[\'fit\']=fit_gumbel(sector[\'annual_maxima\'],min_years=min_years,bootstrap_replicates=bootstrap_replicates,seed=20261105+index)\n        sectors.append(sector)\n    result={\'schema_version\':1,\'status\':\'available\',\'direction_convention\':\'from_true_north_clockwise\',\n            \'sector_width_deg\':width,\'sector_boundaries\':\'start_inclusive_end_exclusive\',\n            \'annual_coverage_required\':1.0,\'min_years\':min_years,\'annual_coverage\':annual,\n            \'included_years\':included,\'sectors\':sectors,\'input_sha256\':hashlib.sha256(path.read_bytes()).hexdigest(),\n            \'method\':\'Annual maximum within each directional sector in each complete UTC calendar year; stationary independent annual maxima; Gumbel MLE; percentile bootstrap resampling years.\',\n            \'sampling_note\':\'Only complete hourly calendar years with valid Hs and direction (direction may be missing for calm Hs=0) are used. No-event years are not silently removed.\',\n            \'references\':[\'https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.gumbel_r.html\',\n                          \'https://confluence.ecmwf.int/spaces/FCST/pages/111155338/What+is+the+direction+convention+for+wave+fields\']}\n    validate_directional_extremes(result)\n    return result\n\n\ndef validate_directional_extremes(extremes):\n    if not extremes or extremes.get(\'status\')!=\'available\':\n        return\n    width=extremes[\'sector_width_deg\']\n    sector_index(0,width)\n    if extremes[\'schema_version\']!=1 or extremes[\'direction_convention\']!=\'from_true_north_clockwise\':\n        raise ValueError(\'Formato o convenzione dei settori degli estremi non riconosciuti.\')\n    sectors=extremes[\'sectors\']\n    if len(sectors)!=round(360/width):\n        raise ValueError(\'La partizione dei settori degli estremi non copre tutte le direzioni.\')\n    for index,sector in enumerate(sectors):\n        if sector[\'id\']!=f\'sector-{index}\' or sector[\'center_deg\']!=index*width or sector[\'width_deg\']!=width:\n            raise ValueError(\'Settori degli estremi incoerenti.\')\n        if sector[\'start_deg\']!=(index*width-width/2)%360 or sector[\'end_deg\']!=(index*width+width/2)%360:\n            raise ValueError(\'Estremi dei settori statistici incoerenti.\')\n        fit=sector[\'fit\']\n        if fit[\'status\']!=\'available\':\n            continue\n        if fit[\'n_years\']<20 or fit[\'n_years\']!=len(sector[\'annual_maxima\']):\n            raise ValueError(\'Numero di anni del fit Gumbel non valido.\')\n        if not math.isfinite(fit[\'loc_m\']) or not math.isfinite(fit[\'scale_m\']) or fit[\'scale_m\']<=0:\n            raise ValueError(\'Parametri Gumbel non validi.\')\n        periods=fit[\'return_period_years\']\n        arrays=[fit[key] for key in (\'return_level_m\',\'ci95_low_m\',\'ci95_high_m\')]\n        if not periods or len(set(periods))!=len(periods) or periods!=sorted(periods) or periods[0]<=1:\n            raise ValueError(\'Tempi di ritorno della curva non validi.\')\n        if any(len(a)!=len(periods) or any(not math.isfinite(v) for v in a) for a in arrays):\n            raise ValueError(\'Curve Gumbel non valide.\')\n        if any(a>b for a,b in zip(fit[\'ci95_low_m\'],fit[\'ci95_high_m\'])):\n            raise ValueError(\'Intervalli Gumbel non validi.\')\n\n\ndef compare_directional_extremes(waves, extremes):\n    if not extremes or extremes.get(\'status\')!=\'available\':\n        return waves\n    out=[]\n    for wave in waves:\n        index=sector_index(wave.get(\'direction_deg\'),extremes[\'sector_width_deg\'])\n        sector=extremes[\'sectors\'][index] if index is not None else None\n        fit=sector[\'fit\'] if sector else None\n        period=gumbel_return_period(wave[\'hs_m\'],fit[\'loc_m\'],fit[\'scale_m\']) if fit and fit[\'status\']==\'available\' else None\n        out.append(dict(wave,wave_sector_id=sector[\'id\'] if sector else None,hs_return_period_years=period))\n    return out\n', 'forecast_service/sealevel_reference.py': '"""Fixed historical Copernicus SSH reference, prepared once per site/cell/year."""\nimport calendar\nimport copy\nfrom datetime import datetime, timedelta, timezone\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\n\nDATASET_ID=\'cmems_mod_med_phy-ssh_anfc_4.2km-2D_PT1H-m\'\n\n\ndef _time(value):\n    value=datetime.fromisoformat(str(value).replace(\'Z\',\'+00:00\'))\n    return value.replace(tzinfo=timezone.utc) if value.tzinfo is None else value.astimezone(timezone.utc)\n\n\ndef _iso(value):\n    return value.astimezone(timezone.utc).isoformat().replace(\'+00:00\',\'Z\')\n\n\ndef validate_reference(reference, *, location=None, dataset_id=None, selected_point=None):\n    if reference is None:\n        return\n    if reference.get(\'schema_version\')!=1 or reference.get(\'kind\')!=\'copernicus_historical_mean\':\n        raise ValueError(\'Riferimento storico del livello marino non riconosciuto.\')\n    year=reference[\'year\']\n    if isinstance(year,bool) or not isinstance(year,int) or not 1900<=year<=9998:\n        raise ValueError(\'Anno del riferimento marino non valido.\')\n    expected=8784 if calendar.isleap(year) else 8760\n    start=datetime(year,1,1,tzinfo=timezone.utc)\n    if reference[\'n_records\']!=expected or _time(reference[\'start\'])!=start or _time(reference[\'end\'])!=datetime(year+1,1,1,tzinfo=timezone.utc)-timedelta(hours=1):\n        raise ValueError(\'Il riferimento deve usare un intero anno orario completo.\')\n    if not math.isfinite(reference[\'mean_ssh_m\']) or reference.get(\'variable\')!=\'zos\' or not reference.get(\'dataset_version\'):\n        raise ValueError(\'Valore o provenienza del riferimento SSH non validi.\')\n    if reference.get(\'data_mode\') not in (\'study\',\'demo\'):\n        raise ValueError(\'Modalità del riferimento marino non valida.\')\n    if reference.get(\'local_datum_status\')!=\'not_validated\':\n        raise ValueError(\'La media del modello non certifica il riferimento delle quote locali.\')\n    for field in (\'requested_point\',\'selected_point\'):\n        point=reference[field]\n        if not all(math.isfinite(point[key]) for key in (\'latitude\',\'longitude\')):\n            raise ValueError(\'Coordinate del riferimento marino non valide.\')\n    if dataset_id is not None and reference[\'dataset_id\']!=dataset_id:\n        raise ValueError(\'Riferimento storico e previsione usano dataset SSH diversi: ricalcolare il riferimento.\')\n    if location is not None:\n        point=location.get(\'sealevel_point\',location[\'wave_point\'])\n        if any(abs(point[key]-reference[\'requested_point\'][key])>1e-8 for key in (\'latitude\',\'longitude\')):\n            raise ValueError(\'Il punto del livello marino è cambiato: ricalcolare il riferimento storico per questa località.\')\n        if location.get(\'mean_reference_m\') is not None and abs(location[\'mean_reference_m\']-reference[\'mean_ssh_m\'])>1e-9:\n            raise ValueError(\'Media SSH e riferimento salvato nella località non coincidono.\')\n    if selected_point is not None and any(abs(selected_point[key]-reference[\'selected_point\'][key])>1e-5 for key in (\'latitude\',\'longitude\')):\n        raise ValueError(\'La cella SSH prevista differisce da quella del riferimento storico: ricalcolare il riferimento.\')\n\n\ndef build_reference(records, year, requested_point, selected_point, *, dataset_id=DATASET_ID, dataset_version,\n                    data_mode=\'study\'):\n    expected=8784 if calendar.isleap(year) else 8760\n    samples=sorted((_time(row[\'time\']),float(row[\'ssh_m\'])) for row in records)\n    start=datetime(year,1,1,tzinfo=timezone.utc)\n    if len(samples)!=expected or any(time!=start+timedelta(hours=i) or not math.isfinite(value) for i,(time,value) in enumerate(samples)):\n        raise ValueError(\'SSH storica incompleta: servono tutte le ore dell’anno, senza duplicati o valori mancanti.\')\n    monthly={}\n    for month in range(1,13):\n        values=[value for time,value in samples if time.month==month]\n        monthly[str(month)]={\'mean_ssh_m\':math.fsum(values)/len(values),\'n_records\':len(values)}\n    canonical=json.dumps([[_iso(time),value] for time,value in samples],allow_nan=False,separators=(\',\',\':\'))\n    reference={\'schema_version\':1,\'kind\':\'copernicus_historical_mean\',\'year\':year,\'dataset_id\':dataset_id,\n               \'dataset_version\':dataset_version,\'variable\':\'zos\',\'requested_point\':dict(requested_point),\n               \'selected_point\':dict(selected_point),\'start\':_iso(start),\'end\':_iso(samples[-1][0]),\n               \'n_records\':expected,\'mean_ssh_m\':math.fsum(value for _,value in samples)/expected,\n               \'monthly\':monthly,\'data_mode\':data_mode,\'local_datum_status\':\'not_validated\',\n               \'method\':\'Arithmetic mean of all hourly SSH values in one complete UTC calendar year; fixed for future forecasts.\',\n               \'input_sha256\':hashlib.sha256(canonical.encode()).hexdigest()}\n    validate_reference(reference)\n    return reference\n\n\ndef apply_reference(location, reference):\n    result=copy.deepcopy(location)\n    result.update(mean_reference_m=reference[\'mean_ssh_m\'],sea_level_reference=copy.deepcopy(reference),\n                  vertical_reference_status=\'provisional_model_mean\')\n    validate_location_reference(result)\n    return result\n\n\ndef validate_location_reference(location, *, dataset_id=None, selected_point=None):\n    reference=location.get(\'sea_level_reference\')\n    if reference is None:\n        return\n    if location.get(\'mean_reference_m\') is None:\n        raise ValueError(\'Il riferimento storico richiede la media SSH fissa nella configurazione.\')\n    validate_reference(reference,location=location,dataset_id=dataset_id,selected_point=selected_point)\n\n\ndef download_reference(location, year, username, password, *, dataset_id=DATASET_ID, dataset_version=None, cache=None):\n    """Only called at site setup; no historical download in forecast updates."""\n    import copernicusmarine\n    import numpy as np\n    import pandas as pd\n    if year>=datetime.now(timezone.utc).year:\n        raise ValueError(\'Scegliere un anno di calendario già concluso.\')\n    point=location.get(\'sealevel_point\',location[\'wave_point\'])\n    cache=Path(cache) if cache else None\n    key=hashlib.sha256(json.dumps([dataset_id,dataset_version,year,point],sort_keys=True).encode()).hexdigest()[:16]\n    target=cache/f\'ssh_reference_{year}_{key}.json\' if cache else None\n    if target and target.exists():\n        reference=json.loads(target.read_text())\n        validate_reference(reference,location=location,dataset_id=dataset_id)\n        return reference\n    if not username or not password:\n        raise ValueError(\'Il calcolo storico SSH richiede le credenziali Copernicus Marine.\')\n    if not dataset_version:\n        catalogue=copernicusmarine.describe(dataset_id=dataset_id,disable_progress_bar=True)\n        dataset=next(d for product in catalogue.products for d in product.datasets if d.dataset_id==dataset_id)\n        dataset_version=max(version.label for version in dataset.versions)\n    start=datetime(year,1,1,tzinfo=timezone.utc)\n    ds=copernicusmarine.open_dataset(dataset_id=dataset_id,dataset_version=dataset_version,variables=[\'zos\'],\n        username=username,password=password,start_datetime=_iso(start),\n        end_datetime=_iso(datetime(year+1,1,1,tzinfo=timezone.utc)-timedelta(hours=1)),\n        minimum_latitude=point[\'latitude\'],maximum_latitude=point[\'latitude\'],\n        minimum_longitude=point[\'longitude\'],maximum_longitude=point[\'longitude\'],coordinates_selection_method=\'nearest\')\n    try:\n        array=ds[\'zos\']\n        if array.attrs.get(\'units\',\'m\') not in (\'m\',\'metre\',\'meter\'):\n            raise ValueError(\'La SSH storica deve essere espressa in metri.\')\n        for dim in list(array.dims):\n            if dim!=\'time\':\n                if array.sizes[dim]!=1: raise ValueError(\'Selezionare una sola cella SSH.\')\n                array=array.isel({dim:0})\n        selected={key:float(np.asarray(ds[key].values).reshape(-1)[0]) for key in (\'latitude\',\'longitude\')}\n        times=pd.to_datetime(ds.time.values,utc=True)\n        records=[{\'time\':time.isoformat(),\'ssh_m\':float(value)} for time,value in zip(times,np.asarray(array.values).reshape(-1))]\n    finally:\n        ds.close()\n    reference=build_reference(records,year,point,selected,dataset_id=dataset_id,dataset_version=dataset_version)\n    if target:\n        cache.mkdir(parents=True,exist_ok=True)\n        temporary=target.with_suffix(\'.tmp\');temporary.write_text(json.dumps(reference,ensure_ascii=False,allow_nan=False,indent=2)+\'\\n\');temporary.replace(target)\n    return reference\n', 'forecast_service/site_package.py': '"""Portable study-site packages exported by the standalone Colab notebook."""\nimport copy\nimport json\nfrom pathlib import Path\nimport re\nimport tempfile\nimport zipfile\n\n\ndef validate_payload(payload):\n    slug=payload.get(\'slug\',\'\')\n    if not re.fullmatch(\'[a-z0-9]+(?:-[a-z0-9]+)*\',slug):\n        raise ValueError(\'Identificativo della località non valido.\')\n    if payload.get(\'schema_version\')!=1 or payload.get(\'data_mode\') not in (\'study\',\'demo\'):\n        raise ValueError(\'Formato del pacchetto non riconosciuto.\')\n    if payload.get(\'data_mode\')!=\'demo\' and any(payload.get(\'provenance\',{}).get(key)==\'demo\' for key in (\'profile_data_mode\',\'climate_data_mode\')):\n        raise ValueError(\'Dati dimostrativi dichiarati come dati di studio.\')\n    location=payload[\'location\']\n    from forecast_service.sealevel_reference import validate_location_reference\n    validate_location_reference(location)\n    if payload.get(\'data_mode\')!=\'demo\' and (location.get(\'sea_level_reference\') or {}).get(\'data_mode\')==\'demo\':\n        raise ValueError(\'Riferimento SSH dimostrativo dichiarato come dato di studio.\')\n    from forecast_service.exposure import validate_exposure\n    validate_exposure(location.get(\'wave_exposure\'))\n    if not location.get(\'slopes\') or any(not isinstance(s,(int,float)) or not 0<s<1 for s in location[\'slopes\']):\n        raise ValueError(\'Pendenze non valide.\')\n    if location.get(\'slope_sampling\') is not None:\n        from forecast_service.slope_sampling import validate_sampling\n        validate_sampling(location[\'slopes\'],location[\'slope_sampling\'],[r[\'slope\'] for r in location.get(\'transects\',[])],unit_id=\'whole\')\n    if not isinstance(payload.get(\'thresholds\'), list):\n        raise ValueError(\'Soglie mancanti.\')\n    if location.get(\'climatology_file\') not in (None,f\'config/climatology/{slug}.json\'):\n        raise ValueError(\'Percorso climatologico non valido nel pacchetto.\')\n    from forecast_service.sectors import resolve_sectors, validate_mappings\n    resolve_sectors(location, payload[\'thresholds\'])\n    validate_mappings(location, slug)\n    json.dumps(payload,allow_nan=False)\n    return slug\n\n\ndef write_site_package(output,payload,*,climatology=None,attachments=None,cartography=None):\n    slug=validate_payload(payload)\n    if bool(climatology) != bool(payload[\'location\'].get(\'climatology_file\')):\n        raise ValueError(\'Riferimento climatologico e configurazione non coerenti.\')\n    if climatology:\n        from forecast_service.climate import load_climate\n        with tempfile.TemporaryDirectory() as folder:\n            root=Path(folder)\n            (root/\'climatology.json\').write_text(json.dumps(climatology,allow_nan=False))\n            load_climate(root,\'climatology.json\')\n    expected=cartography_paths(payload[\'location\'],slug)\n    if set(cartography or {})!=set(expected):\n        raise ValueError(\'Allegati cartografici e configurazione non coerenti.\')\n    validate_cartography(cartography or {}, payload[\'location\'])\n    output=Path(output); output.parent.mkdir(parents=True,exist_ok=True)\n    temporary=output.with_suffix(\'.zip.tmp\')\n    try:\n        with zipfile.ZipFile(temporary,\'w\',compression=zipfile.ZIP_DEFLATED) as archive:\n            archive.writestr(\'site.json\',json.dumps(payload,ensure_ascii=False,allow_nan=False,indent=2)+\'\\n\')\n            if climatology:\n                archive.writestr(\'climatology.json\',json.dumps(climatology,ensure_ascii=False,allow_nan=False,separators=(\',\',\':\'))+\'\\n\')\n            for name,path in (cartography or {}).items():\n                archive.write(path,name)\n            for name,path in (attachments or {}).items():\n                if not re.fullmatch(\'[A-Za-z0-9_.-]+\',name) or name.startswith(\'.\'):\n                    raise ValueError(\'Nome allegato non valido.\')\n                archive.write(path,\'diagnostics/\'+name)\n        temporary.replace(output)\n    finally:\n        temporary.unlink(missing_ok=True)\n    return output\n\n\ndef import_site_package(path,root,*,replace=False,allow_demo=False):\n    from forecast_service.pipeline import load_configuration\n    from forecast_service.climate import load_climate\n    root=Path(root)\n    locations,impacts=load_configuration(root)\n    with zipfile.ZipFile(path) as archive:\n        members=archive.infolist()\n        if len(members)>1000 or sum(m.file_size for m in members)>100*1024*1024:\n            raise ValueError(\'Pacchetto troppo grande.\')\n        if len({m.filename for m in members})!=len(members):\n            raise ValueError(\'Nomi duplicati nel pacchetto.\')\n        for member in members:\n            name=member.filename\n            if name not in (\'site.json\',\'climatology.json\') and not re.fullmatch(r\'diagnostics/[A-Za-z0-9_][A-Za-z0-9_.-]*\',name) and not re.fullmatch(r\'cartography/[a-z0-9]+(?:-[a-z0-9]+)*/(?:grid\\.npz|orthophoto\\.png)\',name):\n                raise ValueError(\'Percorso non ammesso nel pacchetto.\')\n        payload=json.loads(archive.read(\'site.json\'))\n        slug=validate_payload(payload)\n        if payload[\'data_mode\']==\'demo\' and not allow_demo:\n            raise ValueError(\'Pacchetto dimostrativo: usare --allow-demo soltanto per una prova locale.\')\n        if slug in locations[\'locations\'] and not replace:\n            raise ValueError(\'Località già presente: usare --replace per aggiornarla intenzionalmente.\')\n        updates={}\n        local=copy.deepcopy(payload[\'location\'])\n        existing=locations[\'locations\'].get(slug,{})\n        if \'sea_level_reference\' not in local and \'mean_reference_m\' not in local and existing.get(\'sea_level_reference\'):\n            from forecast_service.sealevel_reference import apply_reference\n            local=apply_reference(local,existing[\'sea_level_reference\'])\n            payload.setdefault(\'provenance\',{})[\'retained_existing_sea_level_reference\']=True\n        payload[\'location\']=local\n        validate_payload(payload)\n        expected=cartography_paths(local,slug)\n        actual={m.filename for m in members if m.filename.startswith(\'cartography/\')}\n        if actual!=set(expected):\n            raise ValueError(\'Allegati cartografici e configurazione non coerenti.\')\n        with tempfile.TemporaryDirectory() as folder:\n            paths={}\n            for name,destination in expected.items():\n                raw=archive.read(name)\n                target=Path(folder)/name; target.parent.mkdir(parents=True,exist_ok=True); target.write_bytes(raw)\n                paths[name]=target; updates[destination]=raw\n            validate_cartography(paths,local)\n        if local.get(\'sea_level_reference\'):\n\n            ref=local[\'sea_level_reference\']\n            updates[f"config/study_sites/{slug}/sea_level_reference_{ref[\'year\']}.json"]=(json.dumps(ref,ensure_ascii=False,allow_nan=False,indent=2)+\'\\n\').encode()\n        if local.get(\'climatology_file\'):\n            raw=archive.read(\'climatology.json\')\n            with tempfile.TemporaryDirectory() as folder:\n                climate_root=Path(folder)\n                (climate_root/\'climatology.json\').write_bytes(raw)\n                load_climate(climate_root,\'climatology.json\')\n            updates[local[\'climatology_file\']]=raw\n        elif \'climatology.json\' in archive.namelist():\n            raise ValueError(\'Climatologia nel pacchetto ma non nella configurazione.\')\n        locations[\'locations\'][slug]=local\n        impacts[\'locations\'][slug]={\'name\':local[\'name\'],\'thresholds\':payload[\'thresholds\']}\n        for name,contents in [(\'locations.json\',locations),(\'impact_thresholds.json\',impacts)]:\n            updates[\'config/\'+name]=(json.dumps(contents,ensure_ascii=False,allow_nan=False,indent=2)+\'\\n\').encode()\n        updates[f\'config/study_sites/{slug}/site.json\']=(json.dumps(payload,ensure_ascii=False,allow_nan=False,indent=2)+\'\\n\').encode()\n        for member in members:\n            if member.filename.startswith(\'diagnostics/\'):\n                updates[f\'config/study_sites/{slug}/\'+member.filename]=archive.read(member)\n    # Validate merged configuration before touching any existing file.\n    with tempfile.TemporaryDirectory() as folder:\n        check=Path(folder); (check/\'config\').mkdir()\n        for name in (\'locations.json\',\'impact_thresholds.json\'):\n            (check/\'config\'/name).write_bytes(updates[\'config/\'+name])\n        load_configuration(check)\n    old={name:(root/name).read_bytes() if (root/name).exists() else None for name in updates}\n    try:\n        for name,contents in updates.items():\n            destination=root/name; destination.parent.mkdir(parents=True,exist_ok=True)\n            temporary=destination.with_name(destination.name+\'.import.tmp\')\n            temporary.write_bytes(contents); temporary.replace(destination)\n    except Exception:\n        for name,contents in old.items():\n            destination=root/name\n            if contents is None:\n                destination.unlink(missing_ok=True)\n            else:\n                destination.write_bytes(contents)\n        raise\n    return slug\n\n\ndef cartography_paths(location,slug):\n    from forecast_service.sectors import mapping_entries\n    paths={}\n    for key,mapping in mapping_entries(location):\n        paths[f\'cartography/{key}/grid.npz\']=mapping[\'grid_file\']\n        if mapping.get(\'orthophoto_file\'):\n            paths[f\'cartography/{key}/orthophoto.png\']=mapping[\'orthophoto_file\']\n    return paths\n\n\ndef validate_cartography(paths,location):\n    from forecast_service.inundation import load_grid\n    from forecast_service.inundation import orthophoto_bytes\n    from forecast_service.sectors import mapping_entries\n    configurations=dict(mapping_entries(location))\n    for name,path in paths.items():\n        if name.endswith(\'/grid.npz\'):\n            z,_,_,_=load_grid(path)\n            ortho=paths.get(name.replace(\'grid.npz\',\'orthophoto.png\'))\n            if ortho:\n                unit=name.split(\'/\')[1]\n                orthophoto_bytes(ortho,z.shape,configurations[unit])\n', 'forecast_service/sectors.py': '"""Optional beach subdivisions; forcing belongs to the parent location."""\nimport math\nimport re\n\n\ndef valid_id(value):\n    return isinstance(value, str) and re.fullmatch(r\'[a-z0-9]+(?:-[a-z0-9]+)*\', value) is not None\n\n\ndef validate_thresholds(thresholds, *, allow_empty=False):\n    if not isinstance(thresholds, list) or (not thresholds and not allow_empty):\n        raise ValueError(\'Soglie del settore mancanti.\')\n    try:\n        ids = [t[\'id\'] for t in thresholds]\n        heights = [t[\'elevation_m\'] for t in thresholds]\n        if len(set(ids)) != len(ids) or any(not valid_id(i.replace(\'_\', \'-\')) for i in ids):\n            raise ValueError(\'Identificativi delle soglie non validi.\')\n        if any(not isinstance(h, (int, float)) or not math.isfinite(h) for h in heights) or heights != sorted(heights):\n            raise ValueError(\'Quote delle soglie non valide.\')\n        if any(t[\'season\'] not in (\'summer\', \'all_year\') or not t.get(\'description\') for t in thresholds):\n            raise ValueError(\'Descrizione o stagione delle soglie non valida.\')\n    except (KeyError, TypeError) as error:\n        raise ValueError(\'Soglie del settore non valide.\') from error\n\n\ndef resolve_sectors(location, thresholds):\n    """Derive slopes from assigned transects, never from their order in a list."""\n    from forecast_service.exposure import validate_exposure\n    sectors = location.get(\'sectors\', {})\n    if not isinstance(sectors, dict):\n        raise ValueError(\'I settori devono essere un dizionario.\')\n    transects = location.get(\'transects\') or []\n    by_id = {}\n    for row in transects:\n        key = str(row[\'transect_id\'])\n        if key in by_id:\n            raise ValueError(\'Identificativo transetto duplicato.\')\n        by_id[key] = row\n    assigned = set()\n    resolved = {}\n    allowed = {\'name\', \'transect_ids\', \'thresholds\', \'wave_exposure\', \'mapping\', \'notes\', \'summer_season\', \'slopes\', \'slope_sampling\'}\n    for key, sector in sectors.items():\n        if not valid_id(key) or key == \'whole\' or not isinstance(sector, dict) or not sector.get(\'name\'):\n            raise ValueError(\'Identificativo o nome del settore non valido (whole è riservato).\')\n        if set(sector) - allowed:\n            raise ValueError(\'Campi del settore non ammessi: \' + \', \'.join(sorted(set(sector)-allowed)))\n        if not isinstance(sector.get(\'transect_ids\'), list):\n            raise ValueError(\'transect_ids deve essere una lista di identificativi.\')\n        ids = [str(i) for i in sector[\'transect_ids\']]\n        if not ids or len(set(ids)) != len(ids) or set(ids)-by_id.keys():\n            raise ValueError(f\'Transetti mancanti, duplicati o sconosciuti nel settore {key}.\')\n        if assigned.intersection(ids):\n            raise ValueError(\'Uno stesso transetto è assegnato a più settori.\')\n        if \'wave_exposure\' in sector and sector[\'wave_exposure\'] != location.get(\'wave_exposure\'):\n            raise ValueError(\'L’esposizione alle onde è comune all’area: non può variare tra settori costieri.\')\n        assigned.update(ids)\n        selected = [by_id[i] for i in ids]\n        slopes = [row.get(\'slope\') for row in selected]\n        if any(row.get(\'status\', \'valid\') != \'valid\' for row in selected) or not slopes or any(isinstance(s, bool) or not isinstance(s, (int, float)) or not math.isfinite(s) or not 0 < s < 1 for s in slopes):\n            raise ValueError(f\'Pendenze dei transetti non valide nel settore {key}.\')\n        if sector.get(\'slope_sampling\') is not None:\n            from forecast_service.slope_sampling import validate_sampling\n            validate_sampling(sector.get(\'slopes\', []), sector[\'slope_sampling\'], slopes, unit_id=key)\n            slopes = sector[\'slopes\']\n        else:\n            if \'slopes\' in sector:\n                raise ValueError(\'Pendenze campionate del settore senza metadati di campionamento.\')\n            slopes = sorted(set(slopes))\n        merged = {k: v for k, v in location.items() if k not in (\'sectors\', \'mapping\', \'slope_sampling\')}\n        merged.update(sector, slopes=slopes, transects=selected, thresholds=sector.get(\'thresholds\', thresholds))\n        merged[\'wave_exposure\'] = location.get(\'wave_exposure\')\n        validate_thresholds(merged[\'thresholds\'], allow_empty=True)\n        validate_exposure(merged.get(\'wave_exposure\'))\n        season = merged.get(\'summer_season\')\n        if season:\n            from datetime import datetime\n            dates = [season.get(\'start_month_day\'), season.get(\'end_month_day\')]\n            if any(dates) and not all(dates):\n                raise ValueError(\'Definire entrambi gli estremi della stagione.\')\n            for value in dates:\n                if value:\n                    datetime.strptime(\'2000-\'+value, \'%Y-%m-%d\')\n        resolved[key] = merged\n    if sectors and assigned != set(by_id):\n        raise ValueError(\'Ogni transetto incluso deve appartenere a un settore: \' + \', \'.join(sorted(set(by_id)-assigned)))\n    return resolved\n\n\ndef mapping_entries(location):\n    """Stable identifiers for optional per-beach/per-sector cartography."""\n    if location.get(\'mapping\'):\n        yield \'whole\', location[\'mapping\']\n    for key, sector in (location.get(\'sectors\') or {}).items():\n        if sector.get(\'mapping\'):\n            yield key, sector[\'mapping\']\n\n\ndef validate_mappings(location, slug):\n    for key, mapping in mapping_entries(location):\n        prefix = f\'config/study_sites/{slug}/cartography/{key}/\'\n        if not isinstance(mapping, dict) or mapping.get(\'grid_file\') != prefix+\'grid.npz\':\n            raise ValueError(\'Percorso della griglia cartografica non valido.\')\n        if mapping.get(\'orthophoto_file\') not in (None, prefix+\'orthophoto.png\'):\n            raise ValueError(\'Percorso dell’ortofoto non valido.\')\n        for field in (\'survey_credit\', \'survey_date\', \'license\', \'vertical_reference\'):\n            if not isinstance(mapping.get(field), str) or not mapping[field].strip():\n                raise ValueError(f\'Cartografia: compilare {field}.\')\n        if mapping.get(\'vertical_alignment_confirmed\') is not True:\n            raise ValueError(\'Confermare il raccordo verticale DEM/livello marino prima di attivare la mappa.\')\n        if not isinstance(mapping.get(\'dem_offset_m\'), (int, float)) or not math.isfinite(mapping[\'dem_offset_m\']):\n            raise ValueError(\'Correzione verticale del DEM non valida.\')\n', 'forecast_service/inundation.py': '"""Static, sea-connected potential reach. Not hydrodynamics or flood depth.\n\nPrepared grids are bounded, north-up EPSG:3857 rasters (bounds in WGS84). Flood the full grid\nbefore clipping to the display polygon; sector edges are not artificial walls.\nThe one-off raster preparation lives separately to keep production lightweight.\n"""\nfrom collections import deque\nimport hashlib\nimport math\nfrom pathlib import Path\nimport struct\nimport zipfile\nimport zlib\n\nimport numpy as np\n\nMAX_CELLS = 1024 * 1024\nMAX_ORTHOPHOTO_CELLS = 4096 * 4096\nMETHOD = \'connected_static_twl_filtered_p75_p95_v1\'\n\n\ndef png_bytes(rgba, *, max_cells=MAX_CELLS):\n    rgba = np.asarray(rgba, dtype=np.uint8)\n    height, width, channels = rgba.shape\n    if channels != 4 or height * width > max_cells:\n        raise ValueError(\'Dimensioni PNG non valide.\')\n    def chunk(kind, data):\n        return struct.pack(\'>I\', len(data)) + kind + data + struct.pack(\'>I\', zlib.crc32(kind+data) & 0xffffffff)\n    pixels = b\'\'.join(b\'\\0\'+row.tobytes() for row in rgba)\n    return b\'\\x89PNG\\r\\n\\x1a\\n\' + chunk(b\'IHDR\', struct.pack(\'>IIBBBBB\', width, height, 8, 6, 0, 0, 0)) + chunk(b\'IDAT\', zlib.compress(pixels)) + chunk(b\'IEND\', b\'\')\n\n\ndef orthophoto_bytes(path, grid_shape, mapping):\n    raw = Path(path).read_bytes()\n    shape = mapping.get(\'orthophoto_shape\', grid_shape)\n    if len(raw) > 12*1024*1024 or raw[:8] != b\'\\x89PNG\\r\\n\\x1a\\n\' or len(raw) < 24:\n        raise ValueError(\'Ortofoto preparata non valida o troppo grande.\')\n    width, height = struct.unpack(\'>II\', raw[16:24])\n    if not width or not height or max(width, height) > 4096 or width*height > MAX_ORTHOPHOTO_CELLS or [height, width] != list(shape):\n        raise ValueError(\'Ortofoto preparata non coerente con le dimensioni dichiarate.\')\n    return raw\n\n\ndef load_grid(path):\n    # Bound compressed-array expansion before numpy allocates anything.\n    with zipfile.ZipFile(path) as archive:\n        if len(archive.namelist()) != 4 or set(archive.namelist()) != {\'z.npy\', \'seeds.npy\', \'display_mask.npy\', \'bounds.npy\'} or sum(m.file_size for m in archive.infolist()) > 16*1024*1024:\n            raise ValueError(\'Griglia cartografica non valida o troppo grande.\')\n    with np.load(path, allow_pickle=False) as data:\n        z = data[\'z\']; seeds = data[\'seeds\']; display = data[\'display_mask\']; bounds = data[\'bounds\']\n    if z.ndim != 2 or not z.size or z.size > MAX_CELLS or max(z.shape) > 1024 or z.dtype.kind != \'f\' or np.isinf(z).any():\n        raise ValueError(\'DEM preparato non valido.\')\n    if seeds.shape != z.shape or display.shape != z.shape or seeds.dtype != bool or display.dtype != bool:\n        raise ValueError(\'Maschere del DEM non coerenti.\')\n    if not seeds.any() or not np.isfinite(z[seeds]).all() or not display.any():\n        raise ValueError(\'Mancano celle valide di partenza verso mare o area da rappresentare.\')\n    if bounds.shape != (4,) or not np.isfinite(bounds).all():\n        raise ValueError(\'Estensione geografica non valida.\')\n    west, south, east, north = bounds.tolist()\n    if not -180 <= west < east <= 180 or not -90 <= south < north <= 90:\n        raise ValueError(\'Estensione geografica non valida.\')\n    return z, seeds, display, [[south, west], [north, east]]\n\n\ndef connected_mask(z, seeds, level):\n    """Eight-neighbour connectivity, as r.lake\'s 3x3 neighbourhood."""\n    if not math.isfinite(level):\n        raise ValueError(\'Quota di propagazione non valida.\')\n    eligible = np.isfinite(z) & (z < level)\n    filled = seeds & eligible\n    queue = deque(zip(*np.nonzero(filled)))\n    height, width = z.shape\n    while queue:\n        row, col = queue.popleft()\n        for r in range(max(0, row-1), min(height, row+2)):\n            for c in range(max(0, col-1), min(width, col+2)):\n                if eligible[r, c] and not filled[r, c]:\n                    filled[r, c] = True\n                    queue.append((r, c))\n    return filled\n\n\ndef build_map(unit, bulletin, mapping, root, slug, unit_id):\n    from forecast_service.ensemble import STATISTICS_METHOD\n    from forecast_service.exposure import is_exposed\n    from forecast_service.time_utils import parse_time\n    start = parse_time(bulletin.get(\'display_from\', bulletin[\'processing_started_at\'])).replace(minute=0, second=0, microsecond=0)\n    from datetime import timedelta\n    end = start + timedelta(hours=bulletin[\'display_hours\'])\n    if bulletin.get(\'display_to\'):\n        end = parse_time(bulletin[\'display_to\'])\n    rows = [r for r in unit[\'hourly\'] if start <= parse_time(r[\'time\']) < end]\n    expected = [start+timedelta(hours=i) for i in range(bulletin[\'display_hours\'])]\n    if [parse_time(r[\'time\']) for r in rows] != expected or any(r.get(\'statistics_method\') != STATISTICS_METHOD or not all(isinstance(r.get(f\'twl_p{q}_m\'), (int, float)) and math.isfinite(r[f\'twl_p{q}_m\']) for q in (75, 95)) for r in rows):\n        raise ValueError(\'La mappa richiede tutte le 96 ore e percentili filtrati validi.\')\n    grid_path = Path(root)/mapping[\'grid_file\']\n    z, seeds, display, bounds = load_grid(grid_path)\n    prefix = f\'maps/{slug}/{unit_id}\'\n    assets = {}\n    layers = []\n    for q, color in [(75, (35, 129, 188, 115)), (95, (107, 197, 232, 90))]:\n        peak = max(rows, key=lambda r: r[f\'twl_p{q}_m\'])\n        level = peak[f\'twl_p{q}_m\']\n        if not (z[seeds] < level).any():\n            raise ValueError(\'Quota prevista sotto tutte le celle di partenza: spostare i punti di avvio verso mare nel DEM.\')\n        mask = connected_mask(z, seeds, level) & display\n        rgba = np.zeros((*z.shape, 4), dtype=np.uint8)\n        rgba[mask] = color\n        filename = f\'{prefix}-p{q}.png\'\n        assets[filename] = png_bytes(rgba)\n        layers.append({\'percentile\': q, \'level_m\': level, \'peak_time\': peak[\'time\'], \'image\': \'data/\'+filename, \'cells\': int(mask.sum())})\n    ortho = None\n    if mapping.get(\'orthophoto_file\'):\n        image = orthophoto_bytes(Path(root)/mapping[\'orthophoto_file\'], z.shape, mapping)\n        filename = prefix+\'-orthophoto.png\'; assets[filename] = image; ortho = \'data/\'+filename\n    terrain, terrain_scale = None, None\n    if not ortho:\n        from forecast_service.terrain_background import terrain_background\n        image, terrain_scale = terrain_background(z, display)\n        filename = prefix+\'-terrain.png\'; assets[filename] = image; terrain = \'data/\'+filename\n    exposure = [is_exposed(r.get(\'direction_deg\'), unit.get(\'wave_exposure\')) if r.get(\'hs_m\', 0) > 0 else None for r in rows]\n    return {\'status\': \'available\', \'method\': METHOD, \'data_mode\': bulletin[\'mode\'], \'period_from\': start.isoformat(), \'period_to\': end.isoformat(),\n            \'bounds\': bounds, \'layers\': layers, \'orthophoto\': ortho, \'terrain\': terrain, \'terrain_scale\': terrain_scale,\n            \'survey_credit\': mapping[\'survey_credit\'], \'survey_date\': mapping[\'survey_date\'], \'license\': mapping[\'license\'],\n            \'vertical_reference\': mapping[\'vertical_reference\'], \'dem_offset_m\': mapping[\'dem_offset_m\'],\n            \'grid_sha256\': hashlib.sha256(grid_path.read_bytes()).hexdigest(),\n            \'hours_non_frontal\': exposure.count(False), \'hours_direction_unknown\': exposure.count(None),\n            \'interpretation\': \'Estensione statica potenziale, non profondità né probabilità di allagamento. Il runup è intermittente; nessuna trasformazione direzionale o simulazione di sormonto, portate e durata.\'}, assets\n\n\ndef write_maps(bulletin, output, root):\n    """An unavailable optional map must not suppress the otherwise valid forecast."""\n    for slug, location in bulletin[\'locations\'].items():\n        for unit_id, unit in [(\'whole\', location), *location.get(\'sectors\', {}).items()]:\n            mapping = unit.get(\'mapping\')\n            if not mapping:\n                continue\n            try:\n                metadata, assets = build_map(unit, bulletin, mapping, root, slug, unit_id)\n            except (ValueError, OSError, KeyError, zipfile.BadZipFile) as error:\n                unit[\'inundation_map\'] = {\'status\': \'unavailable\', \'reason\': str(error)}\n                continue\n            for name, raw in assets.items():\n                destination = Path(output)/name\n                destination.parent.mkdir(parents=True, exist_ok=True)\n                destination.write_bytes(raw)\n            unit[\'inundation_map\'] = metadata\n', 'forecast_service/mapping_setup.py': '"""One-off GeoTIFF preparation for lightweight static-map publication."""\nfrom pathlib import Path\nimport json\n\nimport numpy as np\n\nfrom forecast_service.inundation import png_bytes, load_grid, MAX_ORTHOPHOTO_CELLS\nfrom forecast_service.sectors import valid_id, validate_mappings\n\n\nclass SeedSelectionError(ValueError):\n    """Keep per-transect diagnostics available when no coastal cell is found."""\n    def __init__(self, message, report):\n        super().__init__(message)\n        self.report = report\n\n\ndef select_sea_seeds(z, transform, *, seed_points=None, seed_profiles=None,\n                     max_advance_m=5.0, max_rise_m=0.5):\n    """Choose first valid prepared cell on the initial sea-to-land profile.\n\n    Manual points are strict and never relocated. Automatic profiles map an ID\n    to ordered [lon, lat, distance_from_first_valid_m, source_elevation_m]\n    samples. Elevations must already use the map\'s vertical correction.\n    Never search in 2-D or jump beyond a source terrain barrier to find a seed.\n    """\n    import math\n    from pyproj import Transformer\n    from rasterio.transform import rowcol, xy\n    if seed_points is not None and seed_profiles is not None:\n        raise ValueError(\'Usare punti manuali oppure profili per l’avvio automatico, non entrambi.\')\n    if not seed_points and not seed_profiles:\n        raise ValueError(\'Indicare punti o profili di partenza verso mare.\')\n    if not all(isinstance(x,(int,float)) and math.isfinite(x) and x>=0 for x in (max_advance_m,max_rise_m)):\n        raise ValueError(\'Distanza e rialzo massimi di avvio devono essere finiti e non negativi.\')\n    to_map=Transformer.from_crs(4326,3857,always_xy=True)\n    to_wgs=Transformer.from_crs(3857,4326,always_xy=True)\n    height,width=z.shape\n    seeds=np.zeros(z.shape,dtype=bool)\n    report=[]\n    def cell(point):\n        if len(point)!=2 or not np.isfinite(point).all() or not -180<=point[0]<=180 or not -85<point[1]<85:\n            raise ValueError(\'Coordinate di avvio: servono longitudine e latitudine WGS84 valide.\')\n        x,y=to_map.transform(*point)\n        row,col=rowcol(transform,x,y)\n        if not 0<=row<height or not 0<=col<width or not np.isfinite(z[row,col]):\n            return None\n        return int(row),int(col)\n    if seed_points is not None:\n        for i,point in enumerate(seed_points,1):\n            selected=cell(point)\n            if selected is None:\n                raise ValueError(f\'Punto a mare manuale {i} fuori dalla griglia preparata o su NoData: scegliere una cella valida; coordinate {point}.\')\n            row,col=selected;seeds[row,col]=True\n            report.append(dict(transect_id=f\'manuale-{i}\',status=\'manuale\',advance_m=0.,\n                               requested_point=list(point),selected_point=list(point),grid_elevation_m=float(z[row,col])))\n        return seeds,report\n    if not isinstance(seed_profiles,dict) or not seed_profiles:\n        raise ValueError(\'Profili di avvio automatico non validi.\')\n    for identifier,samples in seed_profiles.items():\n        samples=np.asarray(samples,dtype=float)\n        if samples.ndim!=2 or samples.shape[1]!=4 or not len(samples) or not np.isfinite(samples).all():\n            raise ValueError(f\'Transetto {identifier}: campioni di avvio non validi.\')\n        if abs(samples[0,2])>1e-6 or (np.diff(samples[:,2])<0).any() or (samples[:,2]<0).any():\n            raise ValueError(f\'Transetto {identifier}: le distanze devono partire da zero e crescere verso terra.\')\n        reference=float(samples[0,3]);selected=None\n        for lon,lat,distance,elevation in samples:\n            # Strict prefix: a low cell beyond a crest is not a valid shortcut.\n            if distance>max_advance_m+1e-8 or elevation>reference+max_rise_m+1e-8:\n                break\n            candidate=cell([lon,lat])\n            if candidate is None:\n                continue\n            row,col=candidate\n            if float(z[row,col])>reference+max_rise_m+1e-8:\n                continue\n            selected=(row,col,lon,lat,distance)\n            break\n        if selected is None:\n            report.append(dict(transect_id=str(identifier),status=\'nessuna_cella_costiera_valida\',advance_m=None,\n                               requested_point=samples[0,:2].tolist(),selected_point=None,grid_elevation_m=None))\n            continue\n        row,col,lon,lat,distance=selected\n        center=xy(transform,row,col)\n        seeds[row,col]=True\n        report.append(dict(transect_id=str(identifier),status=\'iniziale\' if distance<1e-8 else \'avanzato_sul_transetto\',\n                           advance_m=float(distance),requested_point=samples[0,:2].tolist(),\n                           selected_point=[float(lon),float(lat)],grid_cell_center=list(to_wgs.transform(*center)),\n                           grid_elevation_m=float(z[row,col])))\n    failed=[r[\'transect_id\'] for r in report if r[\'selected_point\'] is None]\n    if failed:\n        raise SeedSelectionError(\'Nessuna cella costiera valida per i transetti \'+\', \'.join(failed)+\n            f\' entro {max_advance_m:g} m e {max_rise_m:g} m di rialzo. Controllare la diagnostica o PUNTI_MARE_MANUALI.\',report)\n    return seeds,report\n\n\ndef prepare_mapping(dem_path, output, *, slug, unit_id=\'whole\', orthophoto_path=None,\n                    display_polygon=None, seed_points=None, survey_credit=\'\', survey_date=\'\',\n                    license=\'\', vertical_reference=\'\', dem_offset_m=0.0,\n                    vertical_alignment_confirmed=False, max_dimension=768, clip_to_polygon=False, clip_buffer_m=100,\n                    orthophoto_resolution_m=None, seed_profiles=None,\n                    seed_max_advance_m=5.0, seed_max_rise_m=0.5):\n    """Prepare a local map from a full-area DEM; preserve originals outside Git.\n\n    Polygon and sea-side seed points use WGS84 lon/lat. Offset is ADDED once to\n    DEM heights to match the TWL reference, independently of profile corrections.\n    Full DEM extent is the connectivity domain, polygon is only a display mask.\n    """\n    import rasterio\n    from rasterio.warp import calculate_default_transform, reproject, Resampling, transform_bounds, transform_geom, transform as reproject_coordinates\n    from rasterio.transform import from_bounds, rowcol\n    from rasterio.features import geometry_mask\n    if not valid_id(slug) or not valid_id(unit_id) or not isinstance(max_dimension, int) or not 32 <= max_dimension <= 1024:\n        raise ValueError(\'Identificativi o dimensione della griglia non validi.\')\n    prefix = f\'config/study_sites/{slug}/cartography/{unit_id}/\'\n    metadata = dict(grid_file=prefix+\'grid.npz\', orthophoto_file=prefix+\'orthophoto.png\' if orthophoto_path else None,\n                    survey_credit=survey_credit, survey_date=survey_date, license=license, vertical_reference=vertical_reference,\n                    dem_offset_m=float(dem_offset_m), vertical_alignment_confirmed=vertical_alignment_confirmed)\n    validate_mappings({\'mapping\': metadata} if unit_id == \'whole\' else {\'sectors\': {unit_id: {\'mapping\': metadata}}}, slug)\n    if not seed_points and not seed_profiles:\n        raise ValueError(\'Indicare punti di partenza verso mare in celle valide del DEM (lon, lat).\')\n    with rasterio.open(dem_path) as dem:\n        if not dem.crs or dem.count != 1:\n            raise ValueError(\'DEM: GeoTIFF con una banda, CRS e quote in metri richiesti.\')\n        if dem.units[0] not in (None, \'\', \'m\', \'metre\', \'meter\', \'metres\', \'meters\'):\n            raise ValueError(\'Convertire le quote del DEM in metri.\')\n        left, bottom, right, top = transform_bounds(dem.crs, \'EPSG:3857\', *dem.bounds)\n        native_transform, width, height = calculate_default_transform(dem.crs, \'EPSG:3857\', dem.width, dem.height, *dem.bounds)\n        if clip_to_polygon:\n            if not display_polygon:\n                raise ValueError(\'Poligono necessario per il clipping automatico.\')\n            from forecast_service.geospatial_setup import buffered_map_bounds\n            crop = buffered_map_bounds(display_polygon, clip_buffer_m)\n            left, bottom, right, top = max(left,crop[0]), max(bottom,crop[1]), min(right,crop[2]), min(top,crop[3])\n            if left >= right or bottom >= top:\n                raise ValueError(\'Il settore non interseca il DEM.\')\n            width = max(1, int(np.ceil((right-left)/abs(native_transform.a))))\n            height = max(1, int(np.ceil((top-bottom)/abs(native_transform.e))))\n        west, south, east, north = transform_bounds(\'EPSG:3857\', \'EPSG:4326\', left, bottom, right, top)\n        if not -180 <= west < east <= 180 or not -90 <= south < north <= 90 or east-west > .2 or north-south > .2:\n            raise ValueError(\'Area troppo grande: attivare clip_to_polygon o suddividere in settori più piccoli.\')\n        factor = min(1, max_dimension/max(width, height))\n        width, height = max(1, round(width*factor)), max(1, round(height*factor))\n        transform = from_bounds(left, bottom, right, top, width, height)\n        z = np.full((height, width), np.nan, dtype=np.float32)\n        # Never average away a seawall. Resolution limits still need field review.\n        reproject(rasterio.band(dem, 1), z, src_transform=dem.transform, src_crs=dem.crs,\n                  src_nodata=dem.nodata, dst_transform=transform, dst_crs=\'EPSG:3857\', dst_nodata=np.nan,\n                  resampling=Resampling.nearest)\n        z = z*float(dem.scales[0])+float(dem.offsets[0])\n        z += metadata[\'dem_offset_m\']\n    if not display_polygon or display_polygon.get(\'type\') not in (\'Polygon\', \'MultiPolygon\'):\n        raise ValueError(\'Definire il poligono WGS84 dell’area da rappresentare.\')\n    display = geometry_mask([transform_geom("EPSG:4326", "EPSG:3857", display_polygon)], out_shape=z.shape, transform=transform, invert=True)\n    seeds, seed_report = select_sea_seeds(z, transform, seed_points=seed_points, seed_profiles=seed_profiles,\n                                        max_advance_m=seed_max_advance_m, max_rise_m=seed_max_rise_m)\n    metadata[\'seed_selection\'] = seed_report\n    metadata[\'seed_selection_limits\'] = dict(max_advance_m=seed_max_advance_m,max_rise_m=seed_max_rise_m)\n    output = Path(output); output.mkdir(parents=True, exist_ok=True)\n    grid = output/\'grid.npz\'\n    np.savez_compressed(grid, z=z, seeds=seeds, display_mask=display, bounds=np.array([west, south, east, north]))\n    load_grid(grid)\n    assets = {f\'cartography/{unit_id}/grid.npz\': grid}\n    if orthophoto_path:\n        ortho_width, ortho_height = width, height\n        ortho_transform = transform\n        if orthophoto_resolution_m is not None:\n            import math\n            from pyproj import Geod\n            if not isinstance(orthophoto_resolution_m, (int, float)) or not math.isfinite(orthophoto_resolution_m) or orthophoto_resolution_m <= 0:\n                raise ValueError(\'Risoluzione ortofoto: indicare metri/pixel positivi.\')\n            geod = Geod(ellps=\'WGS84\')\n            ground_width = geod.inv(west, (south+north)/2, east, (south+north)/2)[2]\n            ground_height = geod.inv((west+east)/2, south, (west+east)/2, north)[2]\n            ortho_width = max(1, math.ceil(ground_width/orthophoto_resolution_m))\n            ortho_height = max(1, math.ceil(ground_height/orthophoto_resolution_m))\n            if max(ortho_width, ortho_height) > 4096 or ortho_width*ortho_height > MAX_ORTHOPHOTO_CELLS:\n                raise ValueError(\'Ritaglio ortofoto oltre 4096 pixel per asse: aumentare il passo in metri/pixel o ridurre l’area.\')\n            ortho_transform = from_bounds(left, bottom, right, top, ortho_width, ortho_height)\n            metadata[\'orthophoto_ground_resolution_m\'] = [ground_width/ortho_width, ground_height/ortho_height]\n        with rasterio.open(orthophoto_path) as ortho:\n            if not ortho.crs or ortho.count not in (1, 3, 4) or any(dtype != \'uint8\' for dtype in ortho.dtypes):\n                raise ValueError(\'Ortofoto: GeoTIFF georeferenziato a 8 bit, grigio/RGB/RGBA.\')\n            rgba = np.zeros((ortho_height, ortho_width, 4), dtype=np.uint8)\n            bands = [1, 1, 1] if ortho.count == 1 else [1, 2, 3]\n            for channel, band in enumerate(bands):\n                reproject(rasterio.band(ortho, band), rgba[:, :, channel], src_transform=ortho.transform,\n                          src_crs=ortho.crs, dst_transform=ortho_transform, dst_crs=\'EPSG:3857\', resampling=Resampling.bilinear)\n            # Source mask honours alpha and NoData, without reading the whole mosaic.\n            from rasterio.vrt import WarpedVRT\n            from rasterio.enums import ColorInterp\n            with WarpedVRT(ortho, crs=\'EPSG:3857\', transform=ortho_transform, width=ortho_width, height=ortho_height,\n                           add_alpha=ColorInterp.alpha not in ortho.colorinterp) as vrt:\n                valid = vrt.dataset_mask()\n            ortho_display = geometry_mask([transform_geom(\'EPSG:4326\', \'EPSG:3857\', display_polygon)],\n                                          out_shape=(ortho_height, ortho_width), transform=ortho_transform, invert=True)\n            rgba[:, :, 3] = np.where(ortho_display, valid, 0)\n            raw = png_bytes(rgba, max_cells=MAX_ORTHOPHOTO_CELLS)\n            if len(raw) > 12*1024*1024:\n                raise ValueError(\'Ortofoto preparata oltre 12 MiB: aumentare il passo o ridurre il ritaglio.\')\n            image = output/\'orthophoto.png\'; image.write_bytes(raw)\n            assets[f\'cartography/{unit_id}/orthophoto.png\'] = image\n        metadata[\'orthophoto_shape\'] = [ortho_height, ortho_width]\n        metadata[\'orthophoto_target_resolution_m\'] = orthophoto_resolution_m\n    metadata[\'clip_to_polygon\'] = bool(clip_to_polygon)\n    metadata[\'clip_buffer_m\'] = clip_buffer_m if clip_to_polygon else None\n    metadata[\'prepared_crs\'] = \'EPSG:3857\'\n    metadata[\'prepared_grid_shape\'] = list(z.shape)\n    metadata[\'dem_source_name\'] = Path(dem_path).name\n    metadata[\'orthophoto_source_name\'] = Path(orthophoto_path).name if orthophoto_path else None\n    metadata[\'processing_note\'] = \'DEM ricampionato al vicino più prossimo; verificare opere strette e risoluzione. Connessione nel dominio locale con margine, ritaglio sul poligono solo in uscita.\'\n    (output/\'mapping.json\').write_text(json.dumps(metadata, ensure_ascii=False, indent=2)+\'\\n\')\n    return metadata, assets\n', 'forecast_service/geospatial_setup.py': '"""One-off profiles from DEM + GeoJSON; the beach-face fit remains unchanged."""\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\n\n\ndef reproject_geometry(geometry, source_crs, target_crs):\n    import numpy as np\n    from pyproj import Transformer\n    from shapely import transform\n    converter = Transformer.from_crs(source_crs, target_crs, always_xy=True)\n    return transform(geometry, lambda xy: np.column_stack(converter.transform(xy[:, 0], xy[:, 1])))\n\n\ndef read_geojson(path, *, crs=\'EPSG:4326\', target_crs=\'EPSG:32632\', id_field=\'TR_ID\', polygon=False, name_field=\'name\'):\n    from pyproj import CRS\n    from shapely.geometry import shape\n    from shapely import line_merge\n    data = json.loads(Path(path).read_text())\n    if data.get(\'type\') != \'FeatureCollection\' or not data.get(\'features\'):\n        raise ValueError(\'Serve un GeoJSON FeatureCollection non vuoto.\')\n    source_crs = CRS.from_user_input(crs)\n    declared = data.get(\'crs\', {}).get(\'properties\', {}).get(\'name\')\n    if declared and not source_crs.equals(CRS.from_user_input(declared), ignore_axis_order=True):\n        raise ValueError(\'Il CRS dichiarato nel GeoJSON differisce dal CRS indicato nel notebook.\')\n    results = {}\n    for feature in data[\'features\']:\n        properties = feature.get(\'properties\') or {}\n        identifier = properties.get(id_field)\n        if identifier is None or str(identifier).strip() == \'\':\n            raise ValueError(f\'Identificativo assente: verificare il campo {id_field}.\')\n        identifier = str(identifier)\n        if identifier in results:\n            raise ValueError(f\'Identificativo GeoJSON duplicato: {identifier}.\')\n        geometry = shape(feature[\'geometry\'])\n        geometry = reproject_geometry(geometry, source_crs, target_crs)\n        if not polygon and geometry.geom_type == \'MultiLineString\':\n            geometry = line_merge(geometry)\n        expected = (\'Polygon\', \'MultiPolygon\') if polygon else (\'LineString\',)\n        if geometry.geom_type not in expected or geometry.is_empty or not geometry.is_valid or not all(math.isfinite(v) for v in geometry.bounds):\n            raise ValueError(f\'Geometria non valida per {identifier}: servono poligoni validi o linee continue.\')\n        results[identifier] = dict(geometry=geometry, name=str(properties.get(name_field) or identifier))\n    return results\n\n\ndef extract_dem_profiles(dem_path, transects_path, *, profile_crs=\'EPSG:32632\', geojson_crs=\'EPSG:4326\', id_field=\'TR_ID\', step_m=None):\n    """Sample original DEM cells, including NoData as NaN; never interpolate holes.\n\n    Distance follows the line for ordering only. The existing fitting function\n    subsequently recomputes straight-line distance from the seaward first point.\n    GeoJSON Z is deliberately ignored: all heights come from the DEM.\n    """\n    import numpy as np\n    import pandas as pd\n    import rasterio\n    from pyproj import CRS, Transformer\n    metric_crs = CRS.from_user_input(profile_crs)\n    if not metric_crs.is_projected or not all(abs(axis.unit_conversion_factor-1) < 1e-9 for axis in metric_crs.axis_info[:2]):\n        raise ValueError(\'Il CRS dei profili deve essere proiettato in metri.\')\n    lines = read_geojson(transects_path, crs=geojson_crs, target_crs=metric_crs, id_field=id_field)\n    records, diagnostics = [], []\n    with rasterio.open(dem_path) as dem:\n        if dem.count != 1 or not dem.crs or dem.units[0] not in (None, \'\', \'m\', \'metre\', \'meter\', \'metres\', \'meters\'):\n            raise ValueError(\'DEM: una banda, CRS e quote in metri richiesti.\')\n        to_metric = Transformer.from_crs(dem.crs, metric_crs, always_xy=True)\n        to_dem = Transformer.from_crs(metric_crs, dem.crs, always_xy=True)\n        center = (dem.width/2, dem.height/2)\n        pixel = [to_metric.transform(*(dem.transform*p)) for p in (center, (center[0]+1, center[1]), (center[0], center[1]+1))]\n        native_step = min(math.dist(pixel[0], point) for point in pixel[1:])\n        spacing = native_step if step_m is None or step_m == 0 else float(step_m)\n        if not math.isfinite(spacing) or spacing <= 0:\n            raise ValueError(\'Passo di campionamento non valido.\')\n        scale, offset = float(dem.scales[0]), float(dem.offsets[0])\n        if not math.isfinite(scale) or not math.isfinite(offset):\n            raise ValueError(\'Scala o offset del DEM non validi.\')\n        for identifier, item in lines.items():\n            line = item[\'geometry\']\n            if not math.isfinite(line.length) or line.length <= 0:\n                raise ValueError(f\'Transetto {identifier} di lunghezza nulla.\')\n            count = math.ceil(line.length/spacing)+1\n            if count > 100000 or len(records)+count > 1000000:\n                raise ValueError(\'Troppi campioni: aumentare il passo o ridurre l’area.\')\n            distances = np.append(np.arange(0, line.length, spacing), line.length)\n            points = [line.interpolate(float(distance)) for distance in distances]\n            xy = [(point.x, point.y) for point in points]\n            native_xy = [to_dem.transform(*point) for point in xy]\n            missing, outside = 0, 0\n            for distance, point, sample_xy, sample in zip(distances, xy, native_xy, dem.sample(native_xy, indexes=1, masked=True)):\n                row, col = dem.index(*sample_xy)\n                is_outside = not 0 <= row < dem.height or not 0 <= col < dem.width\n                invalid = is_outside or bool(np.ma.getmaskarray(sample)[0]) or not np.isfinite(sample[0])\n                value = float(\'nan\') if invalid else float(sample[0])*scale+offset\n                missing += int(invalid); outside += int(is_outside)\n                records.append(dict(TR_ID=identifier, distance=float(distance), SAMPLE_1=value, X=point[0], Y=point[1]))\n            chord = math.dist(xy[0], xy[-1])\n            diagnostics.append(dict(transect_id=identifier, length_m=float(line.length), n_points=len(distances),\n                                    missing_z=missing, outside_dem=outside, step_m=spacing,\n                                    curved_profile=chord == 0 or line.length > chord*1.01))\n        metadata = dict(method=\'dem_native_cell_sample_v1\', dem_source_name=Path(dem_path).name,\n                        transects_source_name=Path(transects_path).name, profile_crs=metric_crs.to_string(),\n                        geojson_crs=CRS.from_user_input(geojson_crs).to_string(), dem_crs=dem.crs.to_string(), step_m=spacing,\n                        dem_scale=scale, dem_offset=offset,\n                        transects_sha256=hashlib.sha256(Path(transects_path).read_bytes()).hexdigest())\n    return pd.DataFrame(records), pd.DataFrame(diagnostics), metadata\n\n\ndef sector_assignment_choices(areas_path, profiles, selections, included_ids, *, profile_crs=\'EPSG:32632\',\n                              geojson_crs=\'EPSG:4326\', id_field=\'sector_id\', name_field=\'name\', x_col=\'X\', y_col=\'Y\'):\n    """Inspect ambiguous assignments before asking the user through widgets."""\n    from shapely.geometry import LineString\n    areas = read_geojson(areas_path, crs=geojson_crs, target_crs=profile_crs, id_field=id_field, polygon=True, name_field=name_field)\n    records = []\n    for key in included_ids:\n        key = str(key)\n        beachface = LineString(profiles[key].loc[selections[key], [x_col, y_col]].to_numpy())\n        point = beachface.interpolate(.5, normalized=True)\n        candidates = [identifier for identifier, item in areas.items() if item[\'geometry\'].covers(point)]\n        allowed = [identifier for identifier, item in areas.items() if item[\'geometry\'].intersects(beachface)]\n        records.append(dict(transect_id=key, automatic=candidates[0] if len(candidates)==1 else None, allowed=allowed))\n    return records, {key:item[\'name\'] for key,item in areas.items()}\n\n\ndef sectors_from_beachfaces(areas_path, profiles, selections, included_ids, *, profile_crs=\'EPSG:32632\',\n                           geojson_crs=\'EPSG:4326\', id_field=\'sector_id\', name_field=\'name\',\n                           x_col=\'X\', y_col=\'Y\', manual=None):\n    """Assign using the midpoint of the fitted beach face, not the inland line end.\n\n    Boundaries, overlapping areas and unmatched points require an explicit\n    decision. Do not clip profiles: removing their crest would change the fit.\n    """\n    from shapely.geometry import LineString, mapping\n    from pyproj import Transformer\n    from forecast_service.sectors import valid_id\n    areas = read_geojson(areas_path, crs=geojson_crs, target_crs=profile_crs, id_field=id_field, polygon=True, name_field=name_field)\n    if any(not valid_id(key) or key == \'whole\' for key in areas):\n        raise ValueError(\'Gli ID dei settori devono usare minuscole, numeri e trattini; whole è riservato.\')\n    included = [str(key) for key in included_ids]\n    manual = manual or {}\n    if set(manual)-set(included):\n        raise ValueError(\'Assegnazioni manuali per transetti non inclusi.\')\n    sectors = {key: {\'name\': item[\'name\'], \'transect_ids\': []} for key, item in areas.items()}\n    report = []\n    for key in included:\n        selected = profiles[key].loc[selections[key], [x_col, y_col]]\n        beachface = LineString(selected.to_numpy())\n        point = beachface.interpolate(.5, normalized=True)\n        candidates = [identifier for identifier, item in areas.items() if item[\'geometry\'].covers(point)]\n        chosen = manual.get(key)\n        if chosen is not None:\n            if chosen not in areas or not areas[chosen][\'geometry\'].intersects(beachface):\n                raise ValueError(f\'Assegnazione manuale non coerente con la battigia del transetto {key}.\')\n        elif len(candidates) != 1:\n            raise ValueError(f\'Transetto {key}: punto centrale della battigia in {len(candidates)} settori. Correggere i poligoni o indicare ASSEGNAZIONI_MANUALI.\')\n        else:\n            chosen = candidates[0]\n        sectors[chosen][\'transect_ids\'].append(key)\n        report.append(dict(transect_id=key, sector_id=chosen, method=\'manual\' if key in manual else \'beachface_midpoint\'))\n    empty = [key for key, item in sectors.items() if not item[\'transect_ids\']]\n    if empty:\n        raise ValueError(\'Settori senza transetti validi: \'+\', \'.join(empty))\n    polygons = {key: mapping(reproject_geometry(item[\'geometry\'], profile_crs, \'EPSG:4326\')) for key, item in areas.items()}\n    import pandas as pd\n    return sectors, polygons, pd.DataFrame(report)\n\n\ndef buffered_map_bounds(polygon, buffer_m):\n    """A local calculation domain in Web Mercator, with a ground-metre buffer."""\n    from shapely.geometry import shape\n    from pyproj import Transformer\n    if not isinstance(buffer_m, (int, float)) or not math.isfinite(buffer_m) or not 10 <= buffer_m <= 2000:\n        raise ValueError(\'Margine di clipping: scegliere da 10 a 2000 metri.\')\n    geometry = shape(polygon)\n    if geometry.geom_type not in (\'Polygon\', \'MultiPolygon\') or geometry.is_empty or not geometry.is_valid:\n        raise ValueError(\'Poligono di clipping non valido.\')\n    center = geometry.centroid\n    if not -180 <= center.x <= 180 or not 0 <= center.y <= 84:\n        raise ValueError(\'Poligono WGS84 fuori dal dominio previsto per il setup mediterraneo.\')\n    zone = min(60, int((center.x+180)//6)+1)\n    local_crs = f\'EPSG:{32600+zone}\'\n    local = reproject_geometry(geometry, \'EPSG:4326\', local_crs).buffer(buffer_m)\n    return reproject_geometry(local, local_crs, \'EPSG:3857\').bounds\n', 'forecast_service/gnss_setup.py': '"""Measured XYZ beach-face transects: no DEM sampling or elevation selection."""\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\n\nimport numpy as np\n\nGNSS_OLS_METHOD = \'gnss_full_beachface_ols_v1\'\nGNSS_ATTRIBUTE_METHOD = \'gnss_slope_attribute_v1\'\n\n\ndef _number(value, label):\n    if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value):\n        raise ValueError(f\'{label}: serve un numero finito.\')\n    return float(value)\n\n\ndef _load(path):\n    data = json.loads(Path(path).read_text())\n    if data.get(\'type\') != \'FeatureCollection\' or not data.get(\'features\'):\n        raise ValueError(\'GNSS: serve un GeoJSON FeatureCollection non vuoto.\')\n    if any(f.get(\'type\')!=\'Feature\' or not isinstance(f.get(\'properties\'),dict) for f in data[\'features\']):\n        raise ValueError(\'GNSS: elementi o attributi GeoJSON non validi.\')\n    return data\n\n\ndef inspect_gnss(path):\n    """Fields for dropdowns; a declared CRS is read without silently overriding it."""\n    from pyproj import CRS\n    data = _load(path)\n    fields = sorted({key for f in data[\'features\'] for key in f[\'properties\']})\n    numeric = [key for key in fields if all(isinstance(f[\'properties\'].get(key),(int,float)) and\n        not isinstance(f[\'properties\'][key],bool) and math.isfinite(f[\'properties\'][key]) for f in data[\'features\'])]\n    declared = data.get(\'crs\',{}).get(\'properties\',{}).get(\'name\')\n    return dict(fields=fields,numeric_fields=numeric,count=len(data[\'features\']),\n        crs=CRS.from_user_input(declared).to_string() if declared else \'EPSG:4326\',\n        crs_declared=bool(declared),sha256=hashlib.sha256(Path(path).read_bytes()).hexdigest())\n\n\ndef extract_gnss_profiles(path, *, geojson_crs, profile_crs=\'EPSG:32632\', id_field=\'transect_id\',\n                          slope_field=None, r2_field=None, z_correction_m=0.):\n    """Preserve every measured XYZ vertex. Transform XY only, never GNSS Z."""\n    import pandas as pd\n    from pyproj import CRS, Transformer\n    data = _load(path)\n    source = CRS.from_user_input(geojson_crs)\n    target = CRS.from_user_input(profile_crs)\n    if not target.is_projected or not all(abs(a.unit_conversion_factor-1)<1e-9 for a in target.axis_info[:2]):\n        raise ValueError(\'GNSS: il CRS dei profili deve essere proiettato in metri.\')\n    declared = data.get(\'crs\',{}).get(\'properties\',{}).get(\'name\')\n    if declared and not source.equals(CRS.from_user_input(declared),ignore_axis_order=True):\n        raise ValueError(\'Il CRS dichiarato nel GeoJSON GNSS differisce da quello selezionato.\')\n    correction = _number(z_correction_m,\'Correzione quote GNSS\')\n    converter = Transformer.from_crs(source.to_2d(),target.to_2d(),always_xy=True)\n    records, diagnostics, references = [], [], {}\n    for feature in data[\'features\']:\n        props = feature[\'properties\']; identifier = props.get(id_field)\n        if isinstance(identifier,bool) or identifier is None or not str(identifier).strip():\n            raise ValueError(f\'GNSS: identificativo assente nel campo {id_field}.\')\n        key = str(identifier)\n        if key in references:\n            raise ValueError(f\'GNSS: identificativo duplicato {key}.\')\n        geometry = feature.get(\'geometry\') or {}\n        if geometry.get(\'type\') != \'LineString\':\n            raise ValueError(f\'GNSS {key}: serve una LineString XYZ, una per transetto.\')\n        vertices = np.asarray(geometry.get(\'coordinates\'),dtype=float)\n        if vertices.ndim!=2 or vertices.shape[0]<2 or vertices.shape[1]<3 or not np.isfinite(vertices[:,:3]).all():\n            raise ValueError(f\'GNSS {key}: servono almeno due vertici XYZ con quote finite in metri.\')\n        if len(vertices)>100000 or len(records)+len(vertices)>1000000:\n            raise ValueError(\'GNSS: troppi punti nel file.\')\n        x,y = converter.transform(vertices[:,0],vertices[:,1])\n        xy = np.column_stack([x,y])\n        if not np.isfinite(xy).all():\n            raise ValueError(f\'GNSS {key}: coordinate trasformate non valide; verificare il CRS.\')\n        distances = np.r_[0,np.cumsum(np.linalg.norm(np.diff(xy,axis=0),axis=1))]\n        if distances[-1]<=0 or (np.diff(distances)<=0).any():\n            raise ValueError(f\'GNSS {key}: punti consecutivi coincidenti o lunghezza nulla.\')\n        reference = {}\n        if slope_field:\n            beta = _number(props.get(slope_field),f\'GNSS {key}, campo {slope_field}\')\n            if not 0<beta<1:raise ValueError(f\'GNSS {key}: la pendenza nel campo {slope_field} deve essere fra 0 e 1 (m/m).\')\n            reference[\'provided_slope\'] = beta\n        if r2_field:\n            r2 = _number(props.get(r2_field),f\'GNSS {key}, campo {r2_field}\')\n            if not 0<=r2<=1:raise ValueError(f\'GNSS {key}: R² nel campo {r2_field} deve essere fra 0 e 1.\')\n            reference[\'provided_r_squared\'] = r2\n        references[key] = reference\n        for distance,(px,py),pz in zip(distances,xy,vertices[:,2]):\n            records.append(dict(TR_ID=key,distance=float(distance),X=float(px),Y=float(py),SAMPLE_1=float(pz)+correction))\n        chord = float(np.linalg.norm(xy[-1]-xy[0]))\n        diagnostics.append(dict(transect_id=key,n_points=len(vertices),missing_z=0,length_m=float(distances[-1]),\n            curved_profile=chord==0 or distances[-1]>chord*1.01,**reference))\n    metadata = dict(method=\'gnss_measured_xyz_v1\',transects_source_name=Path(path).name,\n        transects_sha256=hashlib.sha256(Path(path).read_bytes()).hexdigest(),geojson_crs=source.to_string(),\n        profile_crs=target.to_string(),id_field=id_field,slope_field=slope_field,r2_field=r2_field,\n        z_source=\'geometry_xyz\',z_correction_m=correction,z_units=\'m\',references=references)\n    return pd.DataFrame(records),pd.DataFrame(diagnostics),metadata\n\n\ndef analyze_gnss_transects(frame, *, references=None, slope_source=\'xyz\', orientation=\'auto\'):\n    """Fit all measured vertices against Euclidean XY distance from the sea end.\n\n    Short GNSS profiles use non-overlapping end groups for orientation, unlike\n    the DEM teaching workflow whose first/last 10-point rule stays unchanged.\n    An optional supplied beta is an explicit alternative, never mixed silently.\n    """\n    import pandas as pd\n    if slope_source not in (\'xyz\',\'attribute\') or orientation not in (\'auto\',\'distance_increases_landward\',\'distance_increases_seaward\'):\n        raise ValueError(\'Sorgente pendenza o orientamento GNSS non valido.\')\n    summaries,profiles,selections = [],{},{}\n    references = references or {}\n    for key,group in frame.groupby(\'TR_ID\',sort=False):\n        key = str(key)\n        try:\n            p = group.sort_values(\'distance\').copy()\n            if len(p)<2 or not np.isfinite(p[[\'distance\',\'X\',\'Y\',\'SAMPLE_1\']].to_numpy(dtype=float)).all():\n                raise ValueError(\'Servono almeno due punti GNSS con coordinate e quote finite.\')\n            edge = min(10,max(1,len(p)//2))\n            difference = float(p.SAMPLE_1.iloc[:edge].mean()-p.SAMPLE_1.iloc[-edge:].mean())\n            flip = difference>0 if orientation==\'auto\' else orientation==\'distance_increases_seaward\'\n            if flip:p=p.iloc[::-1]\n            p=p.reset_index(drop=True)\n            p[\'x_mare\']=np.hypot(p.X-p.X.iloc[0],p.Y-p.Y.iloc[0])\n            x,z = p.x_mare.to_numpy(),p.SAMPLE_1.to_numpy()\n            if len(np.unique(x))<2:raise ValueError(\'Servono almeno due distanze GNSS distinte.\')\n            fitted, fitted_intercept = np.polyfit(x,z,1)\n            reference = references.get(key,{})\n            if slope_source==\'attribute\' and \'provided_slope\' not in reference:\n                raise ValueError(\'Selezionare un campo pendenza per usare i valori forniti.\')\n            beta = float(fitted) if slope_source==\'xyz\' else float(reference[\'provided_slope\'])\n            if not math.isfinite(beta) or not 0<beta<1:raise ValueError(\'Pendenza GNSS non valida; verificare i punti e l’orientamento mare–terra.\')\n            intercept = float(fitted_intercept) if slope_source==\'xyz\' else float(z.mean()-beta*x.mean())\n            variance = float(np.sum((z-z.mean())**2))\n            r2 = 1-float(np.sum((z-(beta*x+intercept))**2))/variance if variance>0 else 0.\n            attrs = dict(reversed=bool(flip),orientation=orientation,orientation_ambiguous=abs(difference)<1e-12,\n                dropped_nan_z=0,distance_source=\'euclidean_xy\')\n            p.attrs.update(attrs)\n            summary = dict(transect_id=key,status=\'valid\',error=\'\',slope=beta,n_points=len(p),r_squared=r2,\n                fit_intercept_m=intercept,z_min_m=None,z_max_m=None,stop_index=len(p),first_above_zmax_found=False,\n                selected_x_min_m=float(x.min()),selected_x_max_m=float(x.max()),\n                method=GNSS_OLS_METHOD if slope_source==\'xyz\' else GNSS_ATTRIBUTE_METHOD,**attrs,**reference)\n            if \'provided_slope\' in reference:summary[\'slope_difference_from_field\']=beta-reference[\'provided_slope\']\n            summaries.append(summary);profiles[key]=p;selections[key]=pd.Series(True,index=p.index)\n        except (ValueError,KeyError,TypeError) as error:\n            summaries.append(dict(transect_id=key,status=\'invalid\',error=str(error),slope=None))\n    return pd.DataFrame(summaries),profiles,selections\n\n\ndef gnss_map_profiles(dem_path,transects_path,profiles,*,profile_crs,geojson_crs,id_field):\n    """DEM-only coastal starts, oriented using GNSS, independent of GNSS datum.\n\n    The dense DEM samples are for map seeds only; never feed them back to the\n    GNSS fit, sector assignment, or Monte Carlo source slopes.\n    """\n    from forecast_service.geospatial_setup import extract_dem_profiles\n    from forecast_service.site_setup import prepare_profile\n    sampled,report,metadata = extract_dem_profiles(dem_path,transects_path,profile_crs=profile_crs,\n        geojson_crs=geojson_crs,id_field=id_field)\n    prepared = {}\n    for key,p in profiles.items():\n        prepared[key] = prepare_profile(sampled[sampled.TR_ID==key],orientation=\n            \'distance_increases_seaward\' if p.attrs[\'reversed\'] else \'distance_increases_landward\')\n        # The coastal search follows the ordered line, including bends. Using\n        # Euclidean radius here would admit a later return leg into the prefix.\n        # This column is only for map seeds, never the GNSS slope regression.\n        prepared[key][\'x_mare\'] = (prepared[key][\'distance\']-prepared[key][\'distance\'].iloc[0]).abs()\n        prepared[key].attrs[\'distance_source\'] = \'along_line_map_seeds_only\'\n    return prepared,report,metadata\n', 'forecast_service/gnss_forms.py': '"""Field selectors for XYZ transects and source-dependent slope controls."""\nfrom forecast_service.gnss_setup import inspect_gnss\n\n\nclass GNSSFieldEditor:\n    def __init__(self,path):\n        import ipywidgets as w\n        self.schema = inspect_gnss(path)\n        fields = self.schema[\'fields\']; numeric=self.schema[\'numeric_fields\']\n        def default(names,options):return next((key for key in names if key in options),None)\n        self.id = w.Dropdown(options=[(\'Seleziona…\',None)]+[(key,key) for key in fields],\n            value=default([\'transect_id\',\'TR_ID\',\'id\',\'ID\'],fields),layout=w.Layout(width=\'100%\'))\n        self.slope = w.Dropdown(options=[(\'Nessuno\',None)]+[(key,key) for key in numeric],\n            value=default([\'tan_beta\',\'slope\',\'beta\'],numeric),layout=w.Layout(width=\'100%\'))\n        self.r2 = w.Dropdown(options=[(\'Nessuno\',None)]+[(key,key) for key in numeric],\n            value=default([\'r2\',\'r_squared\',\'R2\'],numeric),layout=w.Layout(width=\'100%\'))\n        self.method = w.Dropdown(options=[(\'Ricalcola su tutti i punti XYZ\',\'xyz\'),(\'Usa il campo pendenza selezionato\',\'attribute\')],\n            value=\'xyz\',layout=w.Layout(width=\'100%\'))\n        self.crs = w.Text(value=self.schema[\'crs\'],layout=w.Layout(width=\'100%\'))\n        self.correction = w.FloatText(value=0.,layout=w.Layout(width=\'100%\'))\n        self.reference = w.Text(value=\'Riferimento delle quote GNSS da verificare\',layout=w.Layout(width=\'100%\'))\n        pairs=[(\'Identificativo del transetto\',self.id),(\'Sorgente della pendenza\',self.method),\n            (\'Campo pendenza β, in m/m (facoltativo nel ricalcolo)\',self.slope),(\'Campo R² fornito, per confronto (facoltativo)\',self.r2),\n            (\'CRS delle coordinate GNSS (letto dal file, se dichiarato)\',self.crs),\n            (\'Correzione da sommare alle sole quote GNSS (m)\',self.correction),(\'Riferimento verticale delle quote GNSS\',self.reference)]\n        self.view=w.VBox([w.HTML(\'Le quote sono la <b>terza coordinata Z dei vertici</b>, in metri. Non si ricavano dal DEM. Con il ricalcolo, i campi β e R² forniti restano valori di confronto.\')]+\n            [w.VBox([w.HTML(label),field]) for label,field in pairs],layout=w.Layout(width=\'100%\',max_width=\'850px\'))\n\n    def read(self):\n        import math\n        if self.id.value is None:raise ValueError(\'Seleziona il campo identificativo dei transetti GNSS.\')\n        if self.method.value==\'attribute\' and self.slope.value is None:raise ValueError(\'Seleziona il campo della pendenza GNSS.\')\n        if not self.crs.value.strip() or not self.reference.value.strip() or not math.isfinite(self.correction.value):\n            raise ValueError(\'Completa CRS, riferimento e correzione delle quote GNSS.\')\n        return dict(id_field=self.id.value,slope_field=self.slope.value,r2_field=self.r2.value,\n            slope_source=self.method.value,geojson_crs=self.crs.value.strip(),z_correction_m=float(self.correction.value),\n            vertical_reference=self.reference.value.strip())\n\n\nclass ProfileFitEditor:\n    def __init__(self,*,gnss=False,z_min=0.,z_max=1.,orientation=\'auto\'):\n        import ipywidgets as w\n        self.gnss=gnss\n        self.orientation=w.Dropdown(options=[(\'Automatico (controllare il grafico)\',\'auto\'),\n            (\'Il primo punto è a mare\',\'distance_increases_landward\'),(\'L’ultimo punto è a mare\',\'distance_increases_seaward\')],\n            value=orientation,layout=w.Layout(width=\'100%\'))\n        fields=[w.HTML(\'<b>Orientamento del transetto</b>\'),self.orientation]\n        self.z_min,self.z_max=None,None\n        if gnss:\n            fields.insert(0,w.HTML(\'Transetti GNSS già sulla battigia: si usano <b>tutti i vertici</b>, senza Zmin e Zmax.\'))\n        else:\n            self.z_min=w.FloatText(value=float(z_min),description=\'Zmin (m)\')\n            self.z_max=w.FloatText(value=float(z_max),description=\'Zmax (m)\')\n            fields=[w.HTML(\'<b>Tratto di battigia da selezionare nel profilo DEM</b>\'),self.z_min,self.z_max]+fields\n        self.view=w.VBox(fields,layout=w.Layout(width=\'100%\',max_width=\'850px\'))\n\n    def read(self):\n        import math\n        lo=None if self.gnss else float(self.z_min.value)\n        hi=None if self.gnss else float(self.z_max.value)\n        if not self.gnss and (not math.isfinite(lo) or not math.isfinite(hi) or lo>=hi):\n            raise ValueError(\'Zmin e Zmax devono essere finite e crescenti.\')\n        return dict(z_min=lo,z_max=hi,orientation=self.orientation.value)\n', 'forecast_service/structure_forms.py': '"""Editable exposed structures for the one-off notebook, per coastal sector."""\nimport copy\nimport html\n\n\nclass StructureEditor:\n    def __init__(self, units, defaults, *, by_unit=None):\n        import ipywidgets as widgets\n        if by_unit is not None and set(by_unit) != set(units):\n            raise ValueError("Le proposte delle strutture devono corrispondere alle unità costiere.")\n        self.units = units\n        self.rows = {}\n        self.widgets = widgets\n        panels = []\n        for key, label in units.items():\n            self.rows[key] = []\n            box = widgets.VBox()\n            for threshold in (defaults if by_unit is None else by_unit[key]):\n                self._add(key, box, threshold)\n            button = widgets.Button(description=\'Aggiungi struttura\', icon=\'plus\')\n            button.on_click(lambda _, key=key, box=box: self._add(key, box))\n            panels.append(widgets.VBox([widgets.HTML(f\'<b>{html.escape(label)}</b>\'), box, button]))\n        self.view = widgets.VBox(panels)\n\n    def _add(self, key, box, threshold=None):\n        import uuid\n        w = self.widgets\n        threshold = threshold or dict(id=\'struttura-\'+uuid.uuid4().hex[:10], short_label=\'Nuova struttura\', elevation_m=1., season=\'all_year\')\n        present = w.Checkbox(value=True, description=\'Presente\', indent=False, layout=w.Layout(width=\'110px\'))\n        name = w.Text(value=threshold.get(\'short_label\', threshold.get(\'description\', \'\')), description=\'Struttura\')\n        elevation = w.FloatText(value=float(threshold[\'elevation_m\']), description=\'Quota (m)\')\n        season = w.Dropdown(options=[(\'Tutto l’anno\', \'all_year\'), (\'Stagione estiva\', \'summer\')], value=threshold[\'season\'], description=\'Attività\')\n        row = dict(id=threshold[\'id\'], present=present, name=name, elevation=elevation, season=season,\n                   survey_statistics=copy.deepcopy(threshold.get(\'survey_statistics\')))\n        present.observe(lambda change: [setattr(field, \'disabled\', not change[\'new\']) for field in (name, elevation, season)], names=\'value\')\n        self.rows[key].append(row)\n        fields = [present, name, elevation, season]\n        stats = row[\'survey_statistics\']\n        if stats:\n            fields.append(w.HTML(f"Quote dai punti DEM: media {stats[\'mean_elevation_m\']:.2f} m; "\n                f"minimo {stats[\'min_elevation_m\']:.2f} m; massimo {stats[\'max_elevation_m\']:.2f} m; "\n                f"{stats[\'n_valid\']} punti validi, {stats[\'n_excluded\']} esclusi."))\n        box.children = (*box.children, w.VBox(fields))\n\n    def read(self):\n        from forecast_service.sectors import validate_thresholds\n        result = {}\n        for key, rows in self.rows.items():\n            thresholds = []\n            for row in rows:\n                if not row[\'present\'].value:\n                    continue\n                label = row[\'name\'].value.strip()\n                if not label:\n                    raise ValueError(f\'Nome della struttura mancante in {self.units[key]}.\')\n                thresholds.append(dict(id=row[\'id\'], short_label=label,\n                    description=\'Possibile interessamento: \'+label.lower(),\n                    elevation_m=float(row[\'elevation\'].value), season=row[\'season\'].value))\n                if row[\'survey_statistics\'] is not None:\n                    stats=copy.deepcopy(row[\'survey_statistics\'])\n                    stats[\'selected_elevation_source\']=\'dem_point_mean\' if thresholds[-1][\'elevation_m\']==stats[\'mean_elevation_m\'] else \'manual_override\'\n                    thresholds[-1][\'survey_statistics\']=stats\n            thresholds.sort(key=lambda value: value[\'elevation_m\'])\n            validate_thresholds(thresholds, allow_empty=True)\n            result[key] = copy.deepcopy(thresholds)\n        return result\n', 'forecast_service/structure_points.py': '"""Propose exposed-structure elevations from original DEM cells at mapped points.\n\n    Each valid point has equal weight. Means group by coastal sector and the\n    normalized \'struttura\' label, not by all DEM cells in a sector. Geometry Z\n    never supplies a height. Keep every invalid observation in the audit table.\n"""\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport re\nimport unicodedata\n\n\ndef structure_label(value):\n    if not isinstance(value, str):\n        raise ValueError(\'Il campo struttura deve contenere un nome testuale.\')\n    label = \' \'.join(unicodedata.normalize(\'NFKC\', value).split())\n    if not label:\n        raise ValueError(\'Nome struttura mancante.\')\n    return label, label.casefold()\n\n\ndef thresholds_from_structure_points(dem_path, points_path, coastal_polygons, *,\n                                    geojson_crs=\'EPSG:4326\', label_field=\'struttura\',\n                                    elevation_correction_m=0., sector_field=\'settore_costiero\', season_field=\'stagione\'):\n    import numpy as np\n    import pandas as pd\n    import rasterio\n    from pyproj import CRS\n    from shapely.geometry import shape\n    from shapely.errors import ShapelyError\n    from forecast_service.geospatial_setup import reproject_geometry\n    if not coastal_polygons or not math.isfinite(elevation_correction_m):\n        raise ValueError(\'Settori costieri o correzione delle quote non validi.\')\n    polygons = {key: shape(value) for key, value in coastal_polygons.items()}\n    if any(poly.geom_type not in (\'Polygon\', \'MultiPolygon\') or poly.is_empty or not poly.is_valid for poly in polygons.values()):\n        raise ValueError(\'Poligoni dei settori costieri non validi.\')\n    raw = Path(points_path).read_bytes()\n    data = json.loads(raw)\n    if data.get(\'type\') != \'FeatureCollection\' or not data.get(\'features\') or len(data[\'features\']) > 100000:\n        raise ValueError(\'Serve un GeoJSON FeatureCollection di punti non vuoto (massimo 100.000 punti).\')\n    source_crs = CRS.from_user_input(geojson_crs)\n    declared = data.get(\'crs\', {}).get(\'properties\', {}).get(\'name\')\n    if declared and not source_crs.equals(CRS.from_user_input(declared), ignore_axis_order=True):\n        raise ValueError(\'Il CRS dichiarato nei punti differisce dal CRS indicato nel notebook.\')\n    observations = []\n    season_names = {\'summer\':\'summer\', \'estiva\':\'summer\', \'all_year\':\'all_year\', \'annuale\':\'all_year\'}\n    with rasterio.open(dem_path) as dem:\n        if dem.count != 1 or not dem.crs or dem.units[0] not in (None, \'\', \'m\', \'metre\', \'meter\', \'metres\', \'meters\'):\n            raise ValueError(\'DEM delle strutture: una banda, CRS e quote in metri richiesti.\')\n        scale, offset = float(dem.scales[0]), float(dem.offsets[0])\n        if not all(math.isfinite(v) for v in (scale, offset)):\n            raise ValueError(\'Scala o offset del DEM non validi.\')\n        for index, feature in enumerate(data[\'features\'], 1):\n            if not isinstance(feature, dict):\n                feature = {}\n            properties = feature.get(\'properties\') or {}\n            if not isinstance(properties, dict):\n                properties = {}\n            row = dict(point_index=index, feature_id=feature.get(\'id\', properties.get(\'id\', index)),\n                       structure=None, structure_key=None, sector_id=None, elevation_m=None,\n                       dem_elevation_m=None, dem_row=None, dem_col=None,\n                       season=None, season_assumed=False, status=\'invalid_geometry\', reason=\'Serve una geometria Point valida.\')\n            try:\n                label, normalized = structure_label(properties.get(label_field))\n                row.update(structure=label, structure_key=normalized)\n            except ValueError as error:\n                row.update(status=\'missing_label\', reason=str(error)); observations.append(row); continue\n            try:\n                if not isinstance(feature.get(\'geometry\'), dict):\n                    observations.append(row); continue\n                point = shape(feature[\'geometry\'])\n                if point.geom_type != \'Point\' or point.is_empty or not point.is_valid:\n                    observations.append(row); continue\n                point = reproject_geometry(point, source_crs, \'EPSG:4326\')\n                if not all(math.isfinite(v) for v in (point.x, point.y)):\n                    observations.append(row); continue\n            except (ValueError, TypeError, KeyError, ShapelyError):\n                observations.append(row); continue\n            candidates = [key for key, poly in polygons.items() if poly.covers(point)]\n            explicit = properties.get(sector_field)\n            if explicit is not None and str(explicit).strip():\n                chosen = str(explicit)\n                if chosen not in candidates:\n                    row.update(status=\'invalid_sector\', reason=\'Il settore_costiero dichiarato non contiene il punto.\')\n                    observations.append(row); continue\n            elif len(candidates) != 1:\n                row.update(status=\'outside_sectors\' if not candidates else \'ambiguous_sector\', reason=f\'Il punto appartiene a {len(candidates)} settori costieri.\')\n                observations.append(row); continue\n            else:\n                chosen = candidates[0]\n            row[\'sector_id\'] = chosen\n            native = reproject_geometry(point, \'EPSG:4326\', dem.crs)\n            r, c = dem.index(native.x, native.y)\n            row.update(dem_row=r, dem_col=c)\n            if not 0 <= r < dem.height or not 0 <= c < dem.width:\n                row.update(status=\'outside_dem\', reason=\'Punto fuori dall’estensione del DEM.\')\n                observations.append(row); continue\n            sample = next(dem.sample([(native.x, native.y)], indexes=1, masked=True))\n            if bool(np.ma.getmaskarray(sample)[0]) or not np.isfinite(sample[0]):\n                row.update(status=\'nodata\', reason=\'Quota DEM mancante; nessuno zero o riempimento automatico.\')\n                observations.append(row); continue\n            value = float(sample[0])*scale+offset\n            if not math.isfinite(value+float(elevation_correction_m)):\n                row.update(status=\'invalid_elevation\', reason=\'Quota trasformata non finita.\')\n                observations.append(row); continue\n            row.update(dem_elevation_m=value, elevation_m=value+float(elevation_correction_m))\n            season = properties.get(season_field)\n            if season is None or season == \'\':\n                row.update(season=\'all_year\', season_assumed=True)\n            elif not isinstance(season, str) or season.casefold().strip() not in season_names:\n                row.update(status=\'invalid_season\', reason=\'Stagione non valida: usare estiva/summer oppure annuale/all_year.\')\n                observations.append(row); continue\n            else:\n                row[\'season\'] = season_names[season.casefold().strip()]\n            row.update(status=\'valid\', reason=\'\'); observations.append(row)\n        dem_crs = dem.crs.to_string()\n    report = pd.DataFrame(observations)\n    proposals = {key: [] for key in coastal_polygons}\n    summaries = []\n    grouped = report[report.sector_id.notna() & report.structure_key.notna()].groupby([\'sector_id\', \'structure_key\'], sort=True)\n    for (unit, key), group in grouped:\n        valid = group[group.status == \'valid\']\n        seasons = set(valid.season)\n        if len(seasons) > 1:\n            raise ValueError(f\'Stagioni discordanti per {group.structure.iloc[0]} nel settore {unit}: correggere i punti.\')\n        label = group.structure.iloc[0]\n        mean = float(valid.elevation_m.mean()) if len(valid) else None\n        stats = dict(method=\'arithmetic_mean_dem_points_v1\', original_structure=label,\n                     mean_elevation_m=mean, min_elevation_m=float(valid.elevation_m.min()) if len(valid) else None,\n                     max_elevation_m=float(valid.elevation_m.max()) if len(valid) else None,\n                     n_valid=int(len(valid)), n_excluded=int(len(group)-len(valid)),\n                     n_dem_cells=int(len(valid[[\'dem_row\', \'dem_col\']].drop_duplicates())),\n                     season_assumed=bool(valid.season_assumed.any()), elevation_correction_m=float(elevation_correction_m))\n        summaries.append(dict(sector_id=unit, structure=label, **stats))\n        if mean is not None:\n            slug = re.sub(\'[^a-z0-9]+\', \'-\', unicodedata.normalize(\'NFKD\', key).encode(\'ascii\', \'ignore\').decode()).strip(\'-\')[:35] or \'struttura\'\n            identifier = slug+\'-\'+hashlib.sha256(key.encode()).hexdigest()[:8]\n            proposals[unit].append(dict(id=identifier, short_label=label, description=\'Possibile interessamento: \'+label.lower(),\n                                        elevation_m=mean, season=next(iter(seasons)), survey_statistics=stats))\n    for values in proposals.values():\n        values.sort(key=lambda item: (item[\'elevation_m\'], item[\'id\']))\n    metadata = dict(method=\'arithmetic_mean_dem_points_v1\', points_source_name=Path(points_path).name,\n                    points_sha256=hashlib.sha256(raw).hexdigest(), geojson_crs=source_crs.to_string(), label_field=label_field,\n                    dem_source_name=Path(dem_path).name, dem_crs=dem_crs,\n                    dem_scale=scale, dem_offset=offset, elevation_correction_m=float(elevation_correction_m),\n                    polygons_sha256=hashlib.sha256(json.dumps(coastal_polygons, sort_keys=True).encode()).hexdigest(),\n                    n_points=len(report), n_valid=int((report.status == \'valid\').sum()), n_excluded=int((report.status != \'valid\').sum()))\n    columns = [\'sector_id\', \'structure\', \'method\', \'original_structure\', \'mean_elevation_m\',\n               \'min_elevation_m\', \'max_elevation_m\', \'n_valid\', \'n_excluded\', \'n_dem_cells\',\n               \'season_assumed\', \'elevation_correction_m\']\n    return proposals, report, pd.DataFrame(summaries, columns=columns), metadata\n', 'forecast_service/contour_preview.py': '"""Display-only outer reach lines. Never fill cells or change terrain heights.\n\nThe connected wet mask remains the scientific/geometric result. This optional\nsetup preview omits internal island outlines and simplifies the external\nwet/dry interface. It is not applied to published forecasts or impact metrics.\n"""\nimport hashlib\nimport math\nfrom pathlib import Path\n\nimport numpy as np\n\nMETHOD = \'external_reach_lines_display_only_v1\'\nMAX_CELLS = 1024 * 1024\nMAX_SEGMENTS = 200000\n\n\ndef _boolean(value, shape, name):\n    a = np.asarray(value)\n    if a.dtype != bool or a.shape != shape:\n        raise ValueError(f\'{name}: serve una maschera booleana con le dimensioni della griglia.\')\n    return a\n\n\ndef _lines(geometry):\n    from shapely.geometry import MultiLineString\n    if geometry.is_empty:\n        return MultiLineString([])\n    if geometry.geom_type in (\'LineString\', \'MultiLineString\'):\n        return geometry\n    result = []\n    if hasattr(geometry, \'geoms\'):\n        for part in geometry.geoms:\n            line = _lines(part)\n            if line.geom_type == \'LineString\' and not line.is_empty:\n                result.append(line)\n            elif line.geom_type == \'MultiLineString\':\n                result.extend(line.geoms)\n    return MultiLineString(result)\n\n\ndef _vertices(geometry):\n    if geometry.is_empty:\n        return 0\n    if geometry.geom_type == \'LineString\':\n        return len(geometry.coords)\n    return sum(_vertices(part) for part in geometry.geoms)\n\n\ndef reach_lines(reached, valid, *, pixel_size_m=(1., 1.), display_mask=None, tolerance_m=1.):\n    """Return original/simplified outer interfaces in local ground metres.\n\n    x increases by column; y decreases by row, origin at the top-left corner.\n    Dry components enclosed inside the reached area are not drawn. Interfaces\n    against NoData, raster edges or an artificial display crop are not invented.\n    Classification uses the whole grid before applying the display mask.\n    """\n    from scipy import ndimage\n    from shapely.geometry import MultiLineString, shape, mapping\n    from shapely import line_merge\n    from shapely.ops import unary_union\n    from affine import Affine\n    from rasterio.features import shapes\n\n    reached = np.asarray(reached)\n    if reached.ndim != 2 or not reached.size or reached.size > MAX_CELLS or reached.dtype != bool:\n        raise ValueError(\'Griglia di risalita non valida o oltre 1024 × 1024 celle.\')\n    valid = _boolean(valid, reached.shape, \'Copertura valida\')\n    display = np.ones(reached.shape, bool) if display_mask is None else _boolean(display_mask, reached.shape, \'Area da mostrare\')\n    if (reached & ~valid).any():\n        raise ValueError(\'La risalita contiene celle senza dato altimetrico.\')\n    if len(pixel_size_m) != 2 or not np.isfinite(pixel_size_m).all() or min(pixel_size_m) <= 0:\n        raise ValueError(\'Dimensioni delle celle in metri non valide.\')\n    if not isinstance(tolerance_m, (int, float)) or not math.isfinite(tolerance_m) or not 0 <= tolerance_m <= 20:\n        raise ValueError(\'Tolleranza del contorno: scegliere un valore fra 0 e 20 metri.\')\n    dx, dy = map(float, pixel_size_m)\n    dry = ~reached\n    border = np.zeros(reached.shape, bool)\n    border[0, :] = dry[0, :]; border[-1, :] = dry[-1, :]\n    border[:, 0] = dry[:, 0]; border[:, -1] = dry[:, -1]\n    external = ndimage.binary_propagation(border, structure=np.ones((3, 3), bool), mask=dry)\n    segments = []\n    # Vertical edges between left/right cells, with both cells observed.\n    left_wet = reached[:, :-1] & external[:, 1:] & display[:, :-1]\n    right_wet = reached[:, 1:] & external[:, :-1] & display[:, 1:]\n    edges = (left_wet | right_wet) & valid[:, :-1] & valid[:, 1:]\n    rr, cc = np.nonzero(edges)\n    if len(rr) > MAX_SEGMENTS:\n        raise ValueError(\'Contorno troppo frammentato: ridurre la risoluzione della griglia cartografica e riprovare.\')\n    segments.extend([((float((c+1)*dx), float(-r*dy)),\n                      (float((c+1)*dx), float(-(r+1)*dy))) for r, c in zip(rr, cc)])\n    # Horizontal edges between upper/lower cells.\n    upper_wet = reached[:-1, :] & external[1:, :] & display[:-1, :]\n    lower_wet = reached[1:, :] & external[:-1, :] & display[1:, :]\n    edges = (upper_wet | lower_wet) & valid[:-1, :] & valid[1:, :]\n    rr, cc = np.nonzero(edges)\n    if len(segments)+len(rr) > MAX_SEGMENTS:\n        raise ValueError(\'Contorno troppo frammentato: ridurre la risoluzione della griglia cartografica e riprovare.\')\n    segments.extend([((float(c*dx), float(-(r+1)*dy)),\n                      (float((c+1)*dx), float(-(r+1)*dy))) for r, c in zip(rr, cc)])\n    original = line_merge(MultiLineString(segments)) if segments else MultiLineString([])\n    simplified = original.simplify(float(tolerance_m), preserve_topology=True) if tolerance_m else original\n    # Do not draw simplified chords through NoData or outside the display area.\n    domain = valid & display\n    polygons = [shape(g) for g, v in shapes(domain.astype(\'uint8\'), mask=domain,\n                transform=Affine(dx, 0, 0, 0, -dy, 0)) if v == 1]\n    domain_geometry = unary_union(polygons)\n    original = _lines(original.intersection(domain_geometry))\n    simplified = _lines(simplified.intersection(domain_geometry))\n    _, islands = ndimage.label(dry & ~external, structure=np.ones((3, 3), bool))\n    report = dict(method=METHOD, display_only=True, reached_cells=int(reached.sum()),\n                  filled_cells=0, mask_modified=False, elevations_modified=False,\n                  internal_components_not_drawn=int(islands), interface_segments=len(segments),\n                  pixel_size_ground_m=[dx, dy], tolerance_m=float(tolerance_m),\n                  original_vertices=_vertices(original), simplified_vertices=_vertices(simplified),\n                  original_length_m=float(original.length), simplified_length_m=float(simplified.length),\n                  empty=bool(original.is_empty),\n                  interpretation=\'Solo limite cartografico: non ricostruisce terreno né propagazione dietro false barriere del DSM.\')\n    return original, simplified, report\n\n\ndef preview_from_grid(grid_path, *, level_m, tolerance_m=1., display_polygon=None):\n    """Use the existing connected_mask unchanged, then prepare WGS84 lines."""\n    from pyproj import Transformer, Geod\n    from shapely import transform\n    from shapely.geometry import mapping, shape\n    from forecast_service.inundation import load_grid, connected_mask\n\n    if not isinstance(level_m, (int, float)) or not math.isfinite(level_m):\n        raise ValueError(\'Quota di prova non valida.\')\n    z, seeds, display, bounds = load_grid(grid_path)\n    reached = connected_mask(z, seeds, float(level_m))\n    (south, west), (north, east) = bounds\n    height, width = z.shape\n    ground = Geod(ellps=\'WGS84\')\n    dx = ground.inv(west, (south+north)/2, east, (south+north)/2)[2]/width\n    dy = ground.inv((west+east)/2, south, (west+east)/2, north)[2]/height\n    original, simplified, report = reach_lines(reached, np.isfinite(z), pixel_size_m=(dx, dy),\n                                                display_mask=display, tolerance_m=tolerance_m)\n    to_merc = Transformer.from_crs(4326, 3857, always_xy=True)\n    to_wgs = Transformer.from_crs(3857, 4326, always_xy=True)\n    left, bottom = to_merc.transform(west, south); right, top = to_merc.transform(east, north)\n    sx, sy = (right-left)/(width*dx), (top-bottom)/(height*dy)\n    clip = None\n    if display_polygon is not None:\n        clip = transform(shape(display_polygon), lambda xy: np.column_stack(to_merc.transform(xy[:, 0], xy[:, 1])))\n    def geographic(geometry):\n        merc = transform(geometry, lambda xy: np.column_stack((left+xy[:, 0]*sx, top+xy[:, 1]*sy)))\n        if clip is not None:\n            merc = _lines(merc.intersection(clip))\n        geo = transform(merc, lambda xy: np.column_stack(to_wgs.transform(xy[:, 0], xy[:, 1])))\n        return dict(type=\'FeatureCollection\', features=[] if geo.is_empty else [dict(type=\'Feature\',\n                    properties=dict(display_only=True, level_m=float(level_m)), geometry=mapping(geo))])\n    report.update(level_m=float(level_m), grid_sha256=hashlib.sha256(Path(grid_path).read_bytes()).hexdigest(),\n                  bounds_wgs84=bounds, bounds_3857=[left, bottom, right, top],\n                  seeds_below_level=int((seeds & (z < level_m)).sum()))\n    return geographic(original), geographic(simplified), report\n', 'forecast_service/slope_sampling.py': '"""Reproducible slope sensitivity samples, prepared once from measured transects.\n\nThese weights describe a chosen representation of beach variability, not the\nprobability of a forecast, an empirical model, or an inundation event.\n"""\nimport hashlib\nimport json\nimport math\nfrom statistics import NormalDist\n\nimport numpy as np\n\n\nMETHODS = (\'measured\', \'linear_range\', \'truncated_normal\')\n\n\ndef _digest(values):\n    return hashlib.sha256(json.dumps(values, separators=(\',\', \':\'), allow_nan=False).encode()).hexdigest()\n\n\ndef _source(values):\n    values = list(values)\n    if not values or any(isinstance(v, bool) or not isinstance(v, (int, float)) or not math.isfinite(v) or not 0 < v < 1 for v in values):\n        raise ValueError(\'Servono pendenze misurate finite, comprese fra 0 e 1.\')\n    return sorted(float(v) for v in values)\n\n\ndef sample_slopes(source, *, method=\'measured\', count=200, seed=42, unit_id=\'whole\'):\n    """Fit underlying normal μ/σ (population std), truncate, then sample by CDF.\n\n    Keep repeated measured observations when fitting the distribution. A single\n    transect or zero variance produces one fixed slope, never invented spread.\n    """\n    values = _source(source)\n    if method not in METHODS or isinstance(count, bool) or not isinstance(count, int) or not 2 <= count <= 2000:\n        raise ValueError(\'Metodo di campionamento o numero di campioni non valido (2–2000).\')\n    if isinstance(seed, bool) or not isinstance(seed, int) or not 0 <= seed < 2**32 or not isinstance(unit_id, str) or not unit_id:\n        raise ValueError(\'Seme casuale o identificativo del settore non valido.\')\n    lo, hi = values[0], values[-1]\n    mean, std = float(np.mean(values)), float(np.std(values, ddof=0))\n    if method == \'measured\' or lo == hi:\n        samples = sorted(set(values))\n    elif method == \'linear_range\':\n        samples = np.linspace(lo, hi, count).tolist()\n    else:\n        distribution = NormalDist(mean, std)\n        lower, upper = distribution.cdf(lo), distribution.cdf(hi)\n        # Stable sector-specific stream: unrelated sector order cannot change it.\n        salt = int.from_bytes(hashlib.sha256(unit_id.encode()).digest()[:8], \'little\')\n        rng = np.random.Generator(np.random.PCG64(np.random.SeedSequence([seed, salt])))\n        probabilities = rng.uniform(lower, upper, count)\n        samples = sorted(set(distribution.inv_cdf(float(p)) for p in probabilities))\n        if any(not lo <= value <= hi for value in samples):\n            raise ValueError(\'Campioni fuori dai limiti misurati: controllare la precisione delle pendenze.\')\n    metadata = dict(sampling_version=1, rng=\'PCG64\', method=method, requested_count=count, effective_count=len(samples), seed=seed, unit_id=unit_id,\n        source_count=len(values), source_min=lo, source_max=hi, source_mean=mean, source_std=std,\n        std_ddof=0, degenerate=lo == hi, source_sha256=_digest(values), samples_sha256=_digest(samples),\n        interpretation=\'Sensibilità alla variabilità delle pendenze; non probabilità di allagamento o affidabilità dei modelli.\')\n    return samples, metadata\n\n\ndef validate_sampling(samples, metadata, source, *, unit_id):\n    """Reject stale or edited samples after changing transect assignments."""\n    values = _source(source)\n    checked = _source(samples)\n    if not isinstance(metadata, dict):\n        raise ValueError(\'Metadati del campionamento delle pendenze mancanti.\')\n    count, seed = metadata.get(\'requested_count\'), metadata.get(\'seed\')\n    valid_options = (metadata.get(\'sampling_version\')==1 and metadata.get(\'rng\')==\'PCG64\' and\n        metadata.get(\'method\') in METHODS and isinstance(count,int) and not isinstance(count,bool) and 2<=count<=2000 and\n        isinstance(seed,int) and not isinstance(seed,bool) and 0<=seed<2**32)\n    stats = dict(source_min=values[0],source_max=values[-1],source_mean=float(np.mean(values)),source_std=float(np.std(values,ddof=0)))\n    valid_stats = all(isinstance(metadata.get(key),(int,float)) and math.isfinite(metadata[key]) and\n        math.isclose(metadata[key],value,rel_tol=1e-12,abs_tol=1e-15) for key,value in stats.items())\n    valid = (valid_options and valid_stats and metadata.get(\'unit_id\')==unit_id and metadata.get(\'std_ddof\')==0 and\n        metadata.get(\'source_sha256\')==_digest(values) and metadata.get(\'source_count\')==len(values) and\n        metadata.get(\'samples_sha256\')==_digest(checked) and metadata.get(\'effective_count\')==len(checked) and\n        list(samples)==checked and len(set(checked))==len(checked) and values[0]<=checked[0]<=checked[-1]<=values[-1] and\n        metadata.get(\'degenerate\')==(values[0]==values[-1]))\n    if valid:\n        if metadata[\'method\']==\'measured\' or values[0]==values[-1]:\n            valid = checked==sorted(set(values))\n        elif metadata[\'method\']==\'linear_range\':\n            valid = len(checked)==count and np.allclose(checked,np.linspace(values[0],values[-1],count),rtol=1e-12,atol=1e-15)\n        else:\n            valid = 1<=len(checked)<=count\n    # Do not regenerate Monte Carlo during validation: the saved realization is\n    # authoritative even when Colab and production use different Python versions.\n    if not valid:\n        raise ValueError(\'Campioni delle pendenze non coerenti: rieseguire il campionamento dopo le modifiche ai transetti.\')\n', 'forecast_service/sector_forms.py': '"""Notebook boxes for ambiguous assignments and public coastal-sector names."""\nimport html\n\n\nclass SectorAssignmentEditor:\n    def __init__(self, records, names):\n        import ipywidgets as w\n        self.rows = {}\n        panels = []\n        for record in records:\n            if record[\'automatic\'] is not None:\n                continue\n            key = record[\'transect_id\']\n            options = [(\'Scegli il settore…\', None)] + [(names[k]+\' [\'+k+\']\', k) for k in record[\'allowed\']]\n            field = w.Dropdown(options=options, value=None, layout=w.Layout(width=\'100%\'))\n            self.rows[key] = field\n            panels.append(w.VBox([w.HTML(\'<b>Transetto \'+html.escape(key)+\'</b> · assegnazione da confermare\'), field]))\n        if not panels:\n            panels.append(w.HTML(\'Tutti i transetti hanno un’assegnazione automatica univoca. Puoi eseguire la cella successiva.\'))\n        self.view = w.VBox(panels, layout=w.Layout(width=\'100%\', max_width=\'850px\'))\n\n    def read(self):\n        missing = [key for key,field in self.rows.items() if field.value is None]\n        if missing:\n            raise ValueError(\'Scegli un settore per i transetti \'+\', \'.join(missing)+\'. Se non ci sono opzioni, verifica i poligoni.\')\n        return {key:field.value for key,field in self.rows.items()}\n\n\nclass SectorNameEditor:\n    def __init__(self, names):\n        import ipywidgets as w\n        self.rows = {key:w.Text(value=name, layout=w.Layout(width=\'100%\')) for key,name in names.items()}\n        self.view = w.VBox([w.VBox([w.HTML(\'Nome pubblico · ID \'+html.escape(key)),field])\n            for key,field in self.rows.items()],layout=w.Layout(width=\'100%\',max_width=\'850px\'))\n\n    def read(self):\n        result = {key:field.value.strip() for key,field in self.rows.items()}\n        if not all(result.values()):\n            raise ValueError(\'Inserisci un nome per ciascun settore costiero.\')\n        return result\n', 'forecast_service/terrain_background.py': '"""Colour the existing prepared DEM for display only; leave NoData transparent."""\nimport numpy as np\n\nfrom forecast_service.inundation import png_bytes\n\nCOLORS = [\'#f5f4e1\', \'#d3e0af\', \'#98b179\', \'#b29568\', \'#7f644b\']\n\n\ndef terrain_background(z, display):\n    z = np.asarray(z)\n    valid = np.isfinite(z) & np.asarray(display, dtype=bool)\n    if not valid.any():\n        raise ValueError(\'Nessuna quota DEM valida nell’area da mostrare.\')\n    low, high = np.quantile(z[valid], [.02, .98]).tolist()\n    if high <= low:\n        low, high = float(z[valid].min()), float(z[valid].max())\n    rgba = np.zeros((*z.shape, 4), dtype=np.uint8)\n    scale = np.zeros(z.shape, dtype=float)\n    if high > low:\n        scale[valid] = np.clip((z[valid]-low)/(high-low), 0., 1.)\n    rgb = np.array([[int(color[i:i+2],16) for i in (1,3,5)] for color in COLORS])\n    for channel in range(3):\n        rgba[:,:,channel][valid] = np.interp(scale[valid], np.linspace(0,1,len(COLORS)),rgb[:,channel]).round().astype(np.uint8)\n    rgba[:,:,3][valid] = 255\n    return png_bytes(rgba), dict(min_m=low,max_m=high,colors=COLORS,\n        label=\'Quota del terreno (m)\',scale_note=\'Colori saturati sotto il 2° e sopra il 98° percentile delle quote; le quote di calcolo non cambiano.\')\n', 'forecast_service/demo_survey.py': '"""Small, explicitly synthetic inputs to exercise the same study-area workflow."""\nfrom pathlib import Path\nimport json\n\n\ndef create_demo_survey(output, latitude, longitude, profile_crs):\n    import numpy as np\n    import rasterio\n    from rasterio.transform import from_origin\n    from pyproj import Transformer\n    from forecast_service.geospatial_setup import reproject_geometry\n    from shapely.geometry import LineString, mapping\n    output = Path(output); output.mkdir(parents=True, exist_ok=True)\n    to_metric = Transformer.from_crs(4326, profile_crs, always_xy=True)\n    x0, y0 = to_metric.transform(longitude, latitude)\n    width, height = 90, 40\n    x0, y0 = round(x0), round(y0)\n    distance = np.arange(width)*1.\n    z = np.empty((height, width), dtype=np.float32)\n    for row in range(height):\n        z[row] = (.045 if row < height/2 else .065)*distance-.3\n        z[row, (distance >= 40) & (distance <= 56)] = .4\n    transform = from_origin(x0, y0, 1, 1)\n    dem = output/\'dem_DEMO.tif\'\n    with rasterio.open(dem, \'w\', driver=\'GTiff\', height=height, width=width, count=1, dtype=\'float32\', crs=profile_crs, transform=transform) as dst:\n        dst.write(z, 1)\n    ortho = output/\'ortofoto_DEMO.tif\'\n    with rasterio.open(ortho, \'w\', driver=\'GTiff\', height=height, width=width, count=3, dtype=\'uint8\', crs=profile_crs, transform=transform) as dst:\n        dst.write(np.full((3, height, width), 175, dtype=np.uint8))\n    features = []\n    for identifier, row, reverse in [(\'T1\', 10, False), (\'T2\', 30, True)]:\n        xy = [(x0+.5, y0-row-.5), (x0+width-.5, y0-row-.5)]\n        line = LineString(xy[::-1] if reverse else xy)\n        features.append(dict(type=\'Feature\', properties={\'TR_ID\': identifier}, geometry=mapping(reproject_geometry(line, profile_crs, \'EPSG:4326\'))))\n    lines = output/\'transetti_DEMO.geojson\'\n    lines.write_text(json.dumps(dict(type=\'FeatureCollection\', features=features)))\n    return dem, ortho, lines\n', 'forecast_service/ensemble.py': '"""Statistical screening of model/slope results; scientific model outputs stay raw."""\nfrom math import isfinite\nfrom statistics import fmean, pstdev\n\nSTATISTICS_METHOD = \'sigma_clip_3_population_v1\'\nSTATISTICS_METADATA = {\n    \'method\': STATISTICS_METHOD,\n    \'grouping\': \'location_or_sector_hour\',\n    \'ensemble_unit\': \'model_slope\',\n    \'sigma_multiplier\': 3,\n    \'sigma_ddof\': 0,\n    \'passes\': 1,\n    \'bounds_inclusive\': True,\n    \'raw_csv_preserved\': True,\n}\n\n\ndef sigma_clip(values):\n    """One pass, population sigma; values exactly at mean ±3σ are retained."""\n    values = list(values)\n    if any(not isfinite(value) for value in values):\n        raise ValueError(\'Il filtro richiede risultati finiti.\')\n    if not values:\n        return [], None, None\n    mean = fmean(values)\n    sigma = pstdev(values, mu=mean)\n    included = [True] * len(values) if sigma == 0 else [abs(value-mean) <= 3*sigma for value in values]\n    return included, mean, sigma\n', 'forecast_service/LICENSE-wave-climate.txt': 'MIT License\n\nCopyright (c) 2026 Alessio Rovere\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n', 'config/study_sites/borghetto-santo-spirito/sea_level_reference_2025.json': '{\n  "schema_version": 1,\n  "kind": "copernicus_historical_mean",\n  "year": 2025,\n  "dataset_id": "cmems_mod_med_phy-ssh_anfc_4.2km-2D_PT1H-m",\n  "dataset_version": "202511",\n  "variable": "zos",\n  "requested_point": {\n    "latitude": 44.08,\n    "longitude": 8.33\n  },\n  "selected_point": {\n    "latitude": 44.0625,\n    "longitude": 8.333333015441895\n  },\n  "start": "2025-01-01T00:00:00Z",\n  "end": "2025-12-31T23:00:00Z",\n  "n_records": 8760,\n  "mean_ssh_m": -0.4032032064388128,\n  "monthly": {\n    "1": {\n      "mean_ssh_m": -0.410685404639785,\n      "n_records": 744\n    },\n    "2": {\n      "mean_ssh_m": -0.5115411359226191,\n      "n_records": 672\n    },\n    "3": {\n      "mean_ssh_m": -0.4807945041263441,\n      "n_records": 744\n    },\n    "4": {\n      "mean_ssh_m": -0.46933154022222223,\n      "n_records": 720\n    },\n    "5": {\n      "mean_ssh_m": -0.4296025681586021,\n      "n_records": 744\n    },\n    "6": {\n      "mean_ssh_m": -0.4312678409444445,\n      "n_records": 720\n    },\n    "7": {\n      "mean_ssh_m": -0.3726959052822581,\n      "n_records": 744\n    },\n    "8": {\n      "mean_ssh_m": -0.3369342917688172,\n      "n_records": 744\n    },\n    "9": {\n      "mean_ssh_m": -0.3413011670013889,\n      "n_records": 720\n    },\n    "10": {\n      "mean_ssh_m": -0.36464155797715053,\n      "n_records": 744\n    },\n    "11": {\n      "mean_ssh_m": -0.31705964612500004,\n      "n_records": 720\n    },\n    "12": {\n      "mean_ssh_m": -0.38133004919354835,\n      "n_records": 744\n    }\n  },\n  "data_mode": "study",\n  "local_datum_status": "not_validated",\n  "method": "Arithmetic mean of all hourly SSH values in one complete UTC calendar year; fixed for future forecasts.",\n  "input_sha256": "c5f2271ef0e2de0c95d91b074a032294af6ee9c1e46074d74c122dfacf6eaef2"\n}\n'}
for relative, source in BUNDLED_SOURCES.items():
    target = RUNTIME / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding="utf-8")
for module_name in list(sys.modules):
    if module_name=="forecast_service" or module_name.startswith("forecast_service."):
        del sys.modules[module_name]
sys.path.insert(0, str(RUNTIME))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import folium
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass
from IPython.display import display, FileLink
from pyproj import CRS, Transformer
from forecast_service.site_setup import analyze_transects, BEACHFACE_METHOD_VERSION
from forecast_service.era5_download import download_series
from forecast_service.climate import build_climate
from forecast_service.exposure import parse_exposure
from forecast_service.extremes import build_directional_extremes
from forecast_service.sealevel_reference import download_reference, validate_reference, apply_reference
from forecast_service.site_package import write_site_package
print("Metodo pendenze:", BEACHFACE_METHOD_VERSION)

In [ ]:
#@title Modalità e salvataggio
DEMO = False #@param {type:"boolean"}
USA_GOOGLE_DRIVE = True #@param {type:"boolean"}
CARTELLA_DRIVE = 'OndeARiva_setup' #@param {type:"string"}
if USA_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive") / CARTELLA_DRIVE
else:
    BASE = Path("/content/ondeariva_dati") if Path("/content").exists() else RUNTIME / "dati"
BASE.mkdir(parents=True, exist_ok=True)
print("Modalità:", "DIMOSTRATIVA — profili e onde sintetici" if DEMO else "SITO DI STUDIO — dati dell’utente")
print("Cartella:", BASE)
print("La cartella del runtime è temporanea. Per conservare la cache usa Drive oppure scarica il pacchetto finale.")

## 2. Area di studio e dati marini comuni
Le coordinate sono in gradi WGS84. Il punto sulla mappa identifica il sito; quello offshore serve a scaricare onde Copernicus ed ERA5. La cella marina ERA5 verrà verificata durante il download. Il punto del livello marino può essere distinto da quello delle onde.

I valori iniziali sono quelli provvisori di Borghetto; possono essere sostituiti liberamente. Il campo `ID_SITO` deve usare lettere minuscole, numeri e trattini.

In [ ]:
#@title Identità e punti di estrazione
ID_SITO = 'borghetto-santo-spirito' #@param {type:"string"}
NOME_SITO = 'Borghetto Santo Spirito (SV)' #@param {type:"string"}
TRATTO_DI_COSTA = 'Riviera di Ponente · Mar Ligure' #@param {type:"string"}
#@markdown **Posizione della località sulla mappa** · latitudine e longitudine WGS84.
LAT_SITO = 44.109 #@param {type:"number"}
LON_SITO = 8.242 #@param {type:"number"}
#@markdown **Punto al largo** · coordinate per onde e serie storica.
LAT_ONDE = 44.08 #@param {type:"number"}
LON_ONDE = 8.33 #@param {type:"number"}
#@markdown **Livello marino** · lascia attiva la casella se usa lo stesso punto delle onde.
LIVELLO_STESSO_PUNTO = True #@param {type:"boolean"}
LAT_LIVELLO = 44.08 #@param {type:"number"}
LON_LIVELLO = 8.33 #@param {type:"number"}
import re
if not re.fullmatch(r"[a-z0-9]+(?:-[a-z0-9]+)*", ID_SITO):
    raise ValueError("ID_SITO non valido.")
for lat, lon in [(LAT_SITO,LON_SITO),(LAT_ONDE,LON_ONDE),(LAT_LIVELLO,LON_LIVELLO)]:
    if not np.isfinite([lat,lon]).all() or not 30<=lat<=46.5 or not -18.2<=lon<=37:
        raise ValueError("Le coordinate devono essere nel dominio mediterraneo.")
WORK = BASE / ID_SITO
WORK.mkdir(parents=True, exist_ok=True)
SITE_INPUT = dict(name=NOME_SITO, area=TRATTO_DI_COSTA,
    map_point=dict(latitude=LAT_SITO,longitude=LON_SITO),
    wave_point=dict(latitude=LAT_ONDE,longitude=LON_ONDE))
MAPPA = folium.Map(location=[LAT_SITO,LON_SITO],zoom_start=12,tiles="OpenStreetMap")
folium.Marker([LAT_SITO,LON_SITO],tooltip="Sito costiero").add_to(MAPPA)
folium.CircleMarker([LAT_ONDE,LON_ONDE],radius=7,color="#006d77",tooltip="Punto onde richiesto").add_to(MAPPA)
folium.LatLngPopup().add_to(MAPPA)
display(MAPPA)
print("Clicca sulla mappa per leggere coordinate, poi riportale nei campi e riesegui questa cella.")

### Direzioni di esposizione — comuni all’intera area
Indica le **direzioni da cui provengono le onde**, non quelle verso cui si muovono: 0° = Nord, 90° = Est, 180° = Sud, 270° = Ovest, in senso orario rispetto al Nord geografico. I settori devono riferirsi al punto offshore scelto e alla spiaggia rappresentata, tenendo conto di costa, promontori e opere.

Scrivi coppie `da,a`, separate da `;` per più settori. Esempi solo di formato: `90,180` comprende Est→Sud; `315,45` attraversa il Nord; `0,360` include tutte le direzioni. Gli estremi sono inclusi. **Lascia vuoto finché non hai definito l’esposizione del sito**: nessun settore di esempio viene applicato automaticamente.

Questi intervalli di provenienza sono comuni a tutti i settori costieri: non sono poligoni della spiaggia. Il portale avverte quando la direzione al largo è fuori dagli intervalli indicati. Questa verifica non simula rifrazione, diffrazione o riparo e non azzera né corregge automaticamente onde, runup o TWL. Anche il confronto storico ERA5 resta riferito a tutte le direzioni.

In [ ]:
#@title Settori di provenienza delle onde che espongono la spiaggia
SETTORI_PROVENIENZA = '' #@param {type:"string"}
NOTE_ESPOSIZIONE = '' #@param {type:"string"}
ESPOSIZIONE = parse_exposure(SETTORI_PROVENIENZA, NOTE_ESPOSIZIONE)
if ESPOSIZIONE is None:
    print("Esposizione non definita: il portale mostrerà un’avvertenza.")
else:
    display(pd.DataFrame(ESPOSIZIONE["sectors_deg"], columns=["Da (gradi)", "A (gradi, senso orario)"]))
    fig, ax = plt.subplots(figsize=(5, 5), subplot_kw={"projection": "polar"})
    ax.set_theta_zero_location("N"); ax.set_theta_direction(-1)
    ax.set_xticks(np.deg2rad(np.arange(0, 360, 45)), ["N", "NE", "E", "SE", "S", "SO", "O", "NO"])
    ax.set_yticks([])
    for start, end in ESPOSIZIONE["sectors_deg"]:
        span = 360 if end-start == 360 else (end-start) % 360
        angles = np.deg2rad(np.linspace(start, start+span, 181))
        ax.fill_between(angles, 0, 1, color="#527d78", alpha=.35)
    ax.set_title("Direzioni di provenienza che espongono la spiaggia")
    plt.show()
    print("Schema orientativo: non è una simulazione della propagazione verso riva.")

### Stagione e parametri di calcolo — comuni all’area
Stagione, rugosità e raccordo del livello marino si impostano una volta. Le strutture esposte saranno definite nel passaggio successivo, separatamente per ogni settore costiero. Indica inizio/fine della stagione in formato `MM-GG`; lascia entrambi vuoti se ancora sconosciuti. La correzione del livello marino raccorda il riferimento del modello a quello locale; non va usata per correggere automaticamente il DEM.

In [ ]:
#@title Stagione, metodi e riferimento delle quote
INIZIO_ESTATE = '05-01' #@param {type:"string"}
FINE_ESTATE = '09-30' #@param {type:"string"}
QUOTA_OMBRELLONI_M = 0.7
QUOTA_STRUTTURE_STAGIONALI_M = 0.9
QUOTA_CHIOSCHI_M = 1.1
QUOTA_PASSEGGIATA_M = 1.4
RUGOSITA_M = 0.001 #@param {type:"number"}
OFFSET_LIVELLO_M = 0.0 #@param {type:"number"}
NOTE_SITO = 'Parametri di studio; quote e raccordo verticale da validare.' #@param {type:"string"}
import datetime as dt
if bool(INIZIO_ESTATE)!=bool(FINE_ESTATE):
    raise ValueError("Indica entrambe le date estive oppure nessuna.")
for value in (INIZIO_ESTATE,FINE_ESTATE):
    if value: dt.datetime.strptime("2000-"+value,"%Y-%m-%d")
heights=[QUOTA_OMBRELLONI_M,QUOTA_STRUTTURE_STAGIONALI_M,QUOTA_CHIOSCHI_M,QUOTA_PASSEGGIATA_M]
if not np.isfinite(heights).all() or heights!=sorted(set(heights)):
    raise ValueError("Le quattro quote devono essere finite e strettamente crescenti.")
if not np.isfinite([RUGOSITA_M,OFFSET_LIVELLO_M]).all() or RUGOSITA_M<0:
    raise ValueError("Rugosità o correzione del livello non valide.")
thresholds=[]
for identifier,height,label,season in zip(
    ["prima_fila_ombrelloni","strutture_turistiche_stagionali","strutture_turistiche_fisse","infrastrutture_fisse"],heights,
    ["Prima fila di ombrelloni","Strutture turistiche stagionali","Chioschi e strutture fisse","Passeggiata a mare"],
    ["summer","summer","all_year","all_year"]):
    thresholds.append(dict(id=identifier,elevation_m=float(height),short_label=label,description="Possibile interessamento: "+label.lower(),season=season))
print("I valori iniziali delle strutture vengono proposti nel modulo successivo, dove possono variare per settore costiero.")

## 3. File dell’area: obbligatori e facoltativi
**Obbligatori per il percorso con mappe:** DEM GeoTIFF e GeoJSON dei transetti. Scegli **DEM + linee** per estrarre le quote dal raster, oppure **GNSS (LineString XYZ)** per usare i punti e le quote misurate sulla battigia. Il DEM resta necessario per le mappe e per le eventuali quote delle strutture. **Facoltativi:** ortofoto GeoTIFF, poligoni dei settori costieri e punti delle strutture con campo `struttura`.

Inserisci il percorso Drive di ogni file disponibile. Se lasci vuoto un file obbligatorio, Colab chiede di caricarlo. Per un file facoltativo, lascia sia il percorso vuoto sia la casella disattivata per saltarlo; attiva la casella per caricarlo dal computer. Senza ortofoto, le mappe mostrano il DEM a colori e OpenStreetMap dove mancano quote. Senza settori si usa la spiaggia intera.

**Impostazioni dei file:** quote in metri; CRS dei GeoJSON e campo identificativo devono corrispondere ai dati esportati da QGIS. `PASSO_CAMPIONAMENTO_M=0` segue il pixel DEM. Nel percorso DEM le quote provengono dal raster; nel percorso GNSS dalla terza coordinata Z del GeoJSON, in metri. Il GNSS ha un proprio riferimento verticale e una correzione separata. Non viene eseguita una conversione automatica fra quote ellissoidiche, ortometriche o livello medio mare. Per i GNSS, dopo la lettura del file scegli i campi nei menu: ID, pendenza β (facoltativa nel ricalcolo) e R² fornito (facoltativo, per confronto). Il CRS dichiarato viene proposto automaticamente e deve essere confermato.

I punti delle strutture vanno sul terreno o sulla quota d’accesso, non sui tetti. La cartella locale `notebooks/dati originali/` non viene trasferita automaticamente su Colab.

In [ ]:
#@title File dell’area di studio
SORGENTE_PENDENZE = 'DEM + linee' #@param ["DEM + linee", "GNSS (LineString XYZ)"]
FILE_DEM_PROFILI = '' #@param {type:"string"}
FILE_ORTOFOTO_COMUNE = '' #@param {type:"string"}
USA_ORTOFOTO = False #@param {type:"boolean"}
FILE_TRANSETTI_GEOJSON = '' #@param {type:"string"}
FILE_AREE_GEOJSON = '' #@param {type:"string"}
FILE_PUNTI_STRUTTURE = '' #@param {type:"string"}
USA_PUNTI_STRUTTURE = False #@param {type:"boolean"}
USA_AREE_GEOJSON = False #@param {type:"boolean"}
print("Percorsi impostati. Esegui la cella successiva per leggere i dati.")

In [ ]:
#@title Sistema di coordinate e lettura dei file
#@markdown **CRS di lavoro:** deve essere proiettato in metri. Serve a entrambi i percorsi.
CRS_PROFILI = 'EPSG:32632' #@param {type:"string"}
#@markdown **Poligoni e punti delle strutture, se presenti:** campi e CRS dei relativi file.
COL_ID_AREE = 'sector_id' #@param {type:"string"}
COL_NOME_AREE = 'name' #@param {type:"string"}
CRS_GEOJSON_AREE = 'EPSG:4326' #@param {type:"string"}
COL_STRUTTURA = 'struttura' #@param {type:"string"}
CRS_GEOJSON_STRUTTURE = 'EPSG:4326' #@param {type:"string"}
#@markdown **Solo per DEM + linee:** questi campi definiscono ID, CRS delle linee e passo del campionamento. Il GNSS avrà i suoi menu dopo la lettura del file.
COL_ID_GEOJSON = 'TR_ID' #@param {type:"string"}
CRS_GEOJSON_TRANSETTI = 'EPSG:4326' #@param {type:"string"}
PASSO_CAMPIONAMENTO_M = 0.0 #@param {type:"number"}
#@markdown **Quote del DEM:** la correzione si applica ai profili DEM e ai punti delle strutture. Le quote GNSS hanno impostazioni separate.
CORREZIONE_QUOTE_DEM_M = 0.0 #@param {type:"number"}
RIFERIMENTO_QUOTE = 'Livello medio mare, da verificare' #@param {type:"string"}
COL_ID, COL_DIST, COL_Z, COL_X, COL_Y = "TR_ID", "distance", "SAMPLE_1", "X", "Y"
crs = CRS.from_user_input(CRS_PROFILI)
if not crs.is_projected or not all(abs(a.unit_conversion_factor-1)<1e-9 for a in crs.axis_info[:2]):
    raise ValueError("Usa un CRS di lavoro proiettato in metri.")

def scegli_file(percorso, messaggio):
    if percorso:
        source = Path(percorso)
    else:
        print(messaggio)
        from google.colab import files
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError("Seleziona un solo file per questo passaggio.")
        name, content = next(iter(uploaded.items()))
        source = WORK / Path(name).name
        source.write_bytes(content)
    if not source.is_file():
        raise ValueError(f"File non trovato: {source}")
    return source

AREAS_SOURCE = None
STRUCTURES_SOURCE = None
if DEMO:
    from forecast_service.demo_survey import create_demo_survey
    DEM_SOURCE, ORTHO_SOURCE, GEO_TRANSECTS_SOURCE = create_demo_survey(WORK/"input_DEMO", LAT_SITO, LON_SITO, CRS_PROFILI)
    print("DEM, immagine e transetti sintetici: nessun rilievo reale.")
else:
    DEM_SOURCE = scegli_file(FILE_DEM_PROFILI, "1/2 · Carica il DEM GeoTIFF dell’intera area")
    ORTHO_SOURCE = scegli_file(FILE_ORTOFOTO_COMUNE, "Facoltativo · Carica l’ortofotomosaico GeoTIFF") if FILE_ORTOFOTO_COMUNE or USA_ORTOFOTO else None
    GEO_TRANSECTS_SOURCE = scegli_file(FILE_TRANSETTI_GEOJSON, "2/2 · Carica il GeoJSON delle linee dei transetti")
    if USA_AREE_GEOJSON or FILE_AREE_GEOJSON:
        AREAS_SOURCE = scegli_file(FILE_AREE_GEOJSON, "Facoltativo · Carica i poligoni dei settori costieri")
    if USA_PUNTI_STRUTTURE or FILE_PUNTI_STRUTTURE:
        STRUCTURES_SOURCE = scegli_file(FILE_PUNTI_STRUTTURE, "Facoltativo · Carica il GeoJSON dei punti delle strutture")
PROFILE_SOURCE_KIND = "DEM + linee" if DEMO else SORGENTE_PENDENZE
GNSS_FIELD_EDITOR = None
GNSS_FIELDS = None
if PROFILE_SOURCE_KIND == "GNSS (LineString XYZ)":
    from forecast_service.gnss_forms import GNSSFieldEditor
    try:
        from google.colab import output as colab_output
        colab_output.enable_custom_widget_manager()
    except ImportError:
        pass
    GNSS_FIELD_EDITOR = GNSSFieldEditor(GEO_TRANSECTS_SOURCE)
    print("Transetti GNSS:", GNSS_FIELD_EDITOR.schema["count"], "· conferma i campi qui sotto, poi esegui Leggi i profili.")
    display(GNSS_FIELD_EDITOR.view)
else:
    print("Percorso DEM: le quote saranno estratte dal raster. Esegui Leggi i profili.")

In [ ]:
#@title Leggi i profili dalla sorgente scelta
if PROFILE_SOURCE_KIND != ("DEM + linee" if DEMO else SORGENTE_PENDENZE):
    raise ValueError("Sorgente cambiata: riesegui la lettura dei file prima di proseguire.")
if GNSS_FIELD_EDITOR:
    from forecast_service.gnss_setup import extract_gnss_profiles
    GNSS_FIELDS = GNSS_FIELD_EDITOR.read()
    options={key:GNSS_FIELDS[key] for key in ["geojson_crs","id_field","slope_field","r2_field","z_correction_m"]}
    profili,diagnostica_estrazione,GEO_PREPROCESSING = extract_gnss_profiles(GEO_TRANSECTS_SOURCE,profile_crs=CRS_PROFILI,**options)
    GEO_PREPROCESSING.update(slope_source=GNSS_FIELDS["slope_source"],vertical_reference=GNSS_FIELDS["vertical_reference"])
    PROFILE_SOURCE = f"{GEO_TRANSECTS_SOURCE.name} · vertici GNSS XYZ"
    PROFILE_VERTICAL_REFERENCE = GNSS_FIELDS["vertical_reference"]
    PROFILE_Z_CORRECTION = GNSS_FIELDS["z_correction_m"]
    print("Usati tutti i vertici GNSS originali: nessun campionamento delle quote dal DEM, nessuna selezione Zmin/Zmax.")
else:
    from forecast_service.geospatial_setup import extract_dem_profiles
    profili, diagnostica_estrazione, GEO_PREPROCESSING = extract_dem_profiles(
        DEM_SOURCE, GEO_TRANSECTS_SOURCE, profile_crs=CRS_PROFILI,
        geojson_crs="EPSG:4326" if DEMO else CRS_GEOJSON_TRANSETTI,
        id_field="TR_ID" if DEMO else COL_ID_GEOJSON, step_m=PASSO_CAMPIONAMENTO_M)
    profili[COL_Z] += CORREZIONE_QUOTE_DEM_M
    PROFILE_SOURCE = f"{GEO_TRANSECTS_SOURCE.name} + {DEM_SOURCE.name}"
    PROFILE_VERTICAL_REFERENCE = RIFERIMENTO_QUOTE
    PROFILE_Z_CORRECTION = CORREZIONE_QUOTE_DEM_M
    print("Passo dei profili:",GEO_PREPROCESSING["step_m"],"m")
PROFILE_DATA_MODE = "demo" if DEMO else "study"
diagnostica_estrazione.to_csv(WORK/"estrazione_transetti.csv", index=False)
display(diagnostica_estrazione)
PROFILE_CONFIG_SIGNATURE = json.dumps(GNSS_FIELDS,sort_keys=True)
print("I dati originali restano sorgenti locali. Verifica ora le impostazioni di pendenza.")

## 4. Pendenza dei transetti: imposta, calcola e controlla
**DEM + linee:** scegli Zmin/Zmax per delimitare la battigia, come nel notebook didattico già usato. Il fit si ferma prima del primo superamento di Zmax.

**GNSS:** i transetti sono già rilevati sulla battigia, quindi **si usano tutti i vertici XYZ senza Zmin/Zmax**. Con il ricalcolo, si regredisce la quota sulla distanza euclidea X/Y dal primo punto a mare; non si aggiungono punti né si campiona il DEM. Se scegli “Usa il campo pendenza”, il valore selezionato viene conservato come β e distinto dal ricalcolo. Il campo R² fornito resta un riferimento separato, non un filtro automatico.

Nel modulo qui sotto il GNSS mostra soltanto l’orientamento. Per profili corti, l’orientamento automatico confronta gruppi alle estremità senza sovrapporli. Controlla i grafici e le linee curve o con ritorni: non vengono rettificate automaticamente. Dopo le impostazioni esegui **Calcola le pendenze**, poi controlla ed eventualmente escludi i transetti non validi.

In [ ]:
#@title Imposta il calcolo della pendenza
from forecast_service.gnss_forms import ProfileFitEditor
Z_MIN = 0.0
Z_MAX = 1.0
ORIENTAMENTO = 'auto'
FIT_EDITOR = ProfileFitEditor(gnss=PROFILE_SOURCE_KIND=="GNSS (LineString XYZ)",z_min=Z_MIN,z_max=Z_MAX,orientation=ORIENTAMENTO)
display(FIT_EDITOR.view)
print("Conferma le impostazioni, poi esegui Calcola le pendenze.")

In [ ]:
#@title Calcola le pendenze dei transetti
FIT_SETTINGS = FIT_EDITOR.read()
Z_MIN,Z_MAX,ORIENTAMENTO = FIT_SETTINGS["z_min"],FIT_SETTINGS["z_max"],FIT_SETTINGS["orientation"]
if PROFILE_SOURCE_KIND=="GNSS (LineString XYZ)":
    if GNSS_FIELD_EDITOR.read()!=GNSS_FIELDS:
        raise ValueError("Campi GNSS cambiati: riesegui Leggi i profili prima di calcolare le pendenze.")
    from forecast_service.gnss_setup import analyze_gnss_transects,GNSS_OLS_METHOD,GNSS_ATTRIBUTE_METHOD
    riepilogo,preparati,selezioni=analyze_gnss_transects(profili,references=GEO_PREPROCESSING["references"],
        slope_source=GNSS_FIELDS["slope_source"],orientation=ORIENTAMENTO)
    PROFILE_FIT_METHOD=GNSS_OLS_METHOD if GNSS_FIELDS["slope_source"]=="xyz" else GNSS_ATTRIBUTE_METHOD
else:
    riepilogo,preparati,selezioni=analyze_transects(profili,Z_MIN,Z_MAX,
        id_col=COL_ID,distance_col=COL_DIST,elevation_col=COL_Z,x_col=COL_X,y_col=COL_Y,orientation=ORIENTAMENTO)
    PROFILE_FIT_METHOD=BEACHFACE_METHOD_VERSION
display(riepilogo)
ANALYSIS_SIGNATURE=(PROFILE_SOURCE_KIND,FIT_SETTINGS,PROFILE_CONFIG_SIGNATURE,COL_ID,COL_DIST,COL_Z,COL_X,COL_Y,
    hashlib.sha256(pd.util.hash_pandas_object(profili,index=True).values.tobytes()).hexdigest())
riepilogo.to_csv(WORK/"pendenze_transetti.csv",index=False)
validi=riepilogo[riepilogo.status=="valid"]
if validi.empty:raise ValueError("Nessun transetto valido: controlla i dati, i campi e l’orientamento.")
print("Metodo usato:",PROFILE_FIT_METHOD)
print("Pendenze valide:",validi.slope.min(),"—",validi.slope.max(),"m/m")
if validi.orientation_ambiguous.any():print("Orientamento automatico ambiguo per alcuni transetti: controlla i grafici o scegli manualmente il lato mare.")

In [ ]:
#@title Verifica un transetto nel grafico
ID_TRANSETTO_DA_VEDERE = '3' #@param {type:"string"}
identifier=ID_TRANSETTO_DA_VEDERE or str(validi.iloc[0].transect_id)
if DEMO and identifier not in preparati: identifier=str(validi.iloc[0].transect_id)
if identifier not in preparati:
    raise ValueError("Transetto non valido o inesistente; scegli un ID dalla tabella.")
p=preparati[identifier]; sel=selezioni[identifier]
info=validi[validi.transect_id==identifier].iloc[0]
fig,ax=plt.subplots(figsize=(10,4))
ax.plot(p.x_mare,p[COL_Z],color=".35",label="Profilo completo")
ax.scatter(p.loc[sel,"x_mare"],p.loc[sel,COL_Z],color="#d87832",label="Punti usati nella regressione",zorder=3)
xfit=p.loc[sel,"x_mare"]
ax.plot(xfit,info.slope*xfit+info.fit_intercept_m,color="#b54135",ls="--",label="Retta stimata" if PROFILE_FIT_METHOD!="gnss_slope_attribute_v1" else "Retta associata alla pendenza fornita")
if Z_MIN is not None:
    ax.axhspan(Z_MIN,Z_MAX,color="#d5e9ee",alpha=.35,label="Intervallo di quota")
if info.first_above_zmax_found:
    ax.axvline(p.loc[int(info.stop_index),"x_mare"],color="#8b633f",ls=":",label="Primo punto sopra Z_MAX")
ax.set(xlabel="Distanza euclidea dal primo punto a mare [m]",ylabel=f"Quota [m] · {PROFILE_VERTICAL_REFERENCE}",
       title=f"{'DEMO · ' if DEMO else ''}{identifier}: beta = {info.slope:.4f}, R² = {info.r_squared:.3f}")
ax.legend(fontsize=8); ax.grid(alpha=.2); fig.tight_layout()
fig.savefig(WORK/"profilo_selezionato.png",dpi=170)
PLOT_ANALYSIS_SIGNATURE=ANALYSIS_SIGNATURE
plt.show()

Controlla soprattutto che il primo punto sia dalla parte del mare e che i punti arancioni rappresentino la battigia. L’esclusione dietro il primo superamento di `Z_MAX` riprende il notebook didattico; non costituisce una simulazione della connessione idraulica o dell’inondazione. Con il GNSS i punti arancioni comprendono tutti i vertici; non si applica il filtro di quota del percorso DEM.

In [ ]:
#@title Conferma i transetti da includere
#@markdown Lascia vuoto per includere tutti i transetti validi. Per escluderne alcuni, scrivi gli ID separati da virgole.
TRANSETTI_ESCLUSI = '' #@param {type:"string"}
excluded={v.strip() for v in TRANSETTI_ESCLUSI.split(",") if v.strip()}
unknown=excluded-set(riepilogo.transect_id)
if unknown:
    raise ValueError(f"ID da escludere non riconosciuti: {unknown}")
scelti=riepilogo[~riepilogo.transect_id.isin(excluded)].copy()
if scelti.empty or (scelti.status!="valid").any():
    raise ValueError("Correggi i transetti non validi o escludili esplicitamente prima di esportare.")
PENDENZE=sorted(set(float(value) for value in scelti.slope))
SELECTION_ANALYSIS_SIGNATURE=ANALYSIS_SIGNATURE
print(f"Transetti inclusi: {len(scelti)}. Il campionamento delle pendenze si imposta dopo l’assegnazione ai settori.")

## 5. Settori costieri, nomi e variabilità delle pendenze
Esegui **Controlla l’assegnazione**. Ogni transetto è assegnato usando il centro del tratto di battigia impiegato per la pendenza. Se il punto è in più poligoni o sul confine, scegli il settore nel menu. Le opzioni sono limitate ai settori che intersecano il tratto usato per il fit; senza opzioni occorre correggere il poligono. Le linee non vengono tagliate.

Poi esegui **Assegna**, modifica i **nomi pubblici** nelle caselle e prosegui con **Conferma nomi e campiona**. Gli ID tecnici rimangono invariati. Senza poligoni compare una sola scheda per la spiaggia intera. Onde, livello marino, esposizione e dati storici restano comuni all’area.

In [ ]:
#@title Controlla l’assegnazione dei transetti
import copy
from forecast_service.sectors import resolve_sectors
from forecast_service.geospatial_setup import sector_assignment_choices, sectors_from_beachfaces, reproject_geometry
from forecast_service.sector_forms import SectorAssignmentEditor, SectorNameEditor
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass
ASSEGNAZIONI_EDITOR = None
if AREAS_SOURCE:
    proposte, nomi = sector_assignment_choices(AREAS_SOURCE, preparati, selezioni, scelti.transect_id,
        profile_crs=CRS_PROFILI,geojson_crs=CRS_GEOJSON_AREE,id_field=COL_ID_AREE,name_field=COL_NOME_AREE,x_col=COL_X,y_col=COL_Y)
    ASSEGNAZIONI_EDITOR = SectorAssignmentEditor(proposte, nomi)
    print(f"{sum(r['automatic'] is not None for r in proposte)} transetti assegnati automaticamente su {len(proposte)}.")
    display(ASSEGNAZIONI_EDITOR.view)
else:
    print("Spiaggia intera: nessuna assegnazione da scegliere. Esegui la cella successiva.")

In [ ]:
#@title Assegna i transetti ai settori costieri
SETTORI_SPIAGGIA = {}
ASSEGNAZIONI_MANUALI = ASSEGNAZIONI_EDITOR.read() if ASSEGNAZIONI_EDITOR else {}
if AREAS_SOURCE:
    SETTORI_SPIAGGIA, POLIGONI_SETTORI, assegnazioni = sectors_from_beachfaces(
        AREAS_SOURCE, preparati, selezioni, scelti.transect_id,
        profile_crs=CRS_PROFILI, geojson_crs=CRS_GEOJSON_AREE,
        id_field=COL_ID_AREE, name_field=COL_NOME_AREE,
        x_col=COL_X, y_col=COL_Y, manual=ASSEGNAZIONI_MANUALI)
    assegnazioni.to_csv(WORK/"assegnazioni_settori.csv", index=False)
    UNITA_COSTIERE = {key:value["name"] for key,value in SETTORI_SPIAGGIA.items()}
    display(assegnazioni.groupby("sector_id").size().rename("Numero di transetti").to_frame())
else:
    import rasterio
    from shapely.geometry import box, mapping
    with rasterio.open(DEM_SOURCE) as source_dem:
        POLIGONI_SETTORI = {"whole": mapping(reproject_geometry(box(*source_dem.bounds), source_dem.crs, "EPSG:4326"))}
    UNITA_COSTIERE = {"whole": NOME_SITO + " · spiaggia intera"}
NOMI_SETTORI_EDITOR = SectorNameEditor(UNITA_COSTIERE)
print("Modifica qui i nomi da mostrare sul sito, poi esegui la cella successiva.")
display(NOMI_SETTORI_EDITOR.view)
SECTOR_ANALYSIS_SIGNATURE = ANALYSIS_SIGNATURE

### Come entrano le pendenze nel runup
Ogni pendenza campionata viene applicata a ciascuna formula empirica, usando le stesse onde del settore. I transetti restano misure di riferimento, non previsioni indipendenti. Il campionamento rappresenta una scelta sulla variabilità della spiaggia: non attribuisce probabilità agli eventi o ai modelli. I percentili dei risultati restano indicatori di sensibilità.

L’ipotesi normale è un’opzione da controllare con gli istogrammi sotto. Media e deviazione standard (divisore N) sono quelle dei valori misurati, prima del troncamento; la media dei campioni può quindi differire. Rimangono disponibili l’uso diretto delle pendenze misurate e l’intervallo regolare.

In [ ]:
#@title Conferma nomi e campiona le pendenze per settore
#@markdown **Monte Carlo:** media e deviazione standard sono calcolate sulle pendenze misurate del singolo settore. La normale è troncata al suo minimo e massimo misurati. Con un solo valore distinto si usa una pendenza fissa.
MODALITA_PENDENZE = 'Monte Carlo · normale troncata' #@param ["Monte Carlo · normale troncata", "Valori dei transetti", "Intervallo minimo-massimo"]
NUM_CAMPIONI_MC = 200 #@param {type:"integer"}
SEME_MONTE_CARLO = 42 #@param {type:"integer"}
#@markdown `NUM_PENDENZE` è usato soltanto nell’alternativa “Intervallo minimo-massimo”.
NUM_PENDENZE = 5 #@param {type:"integer"}
from forecast_service.slope_sampling import sample_slopes
UNITA_COSTIERE = NOMI_SETTORI_EDITOR.read()
for key,sector in SETTORI_SPIAGGIA.items():sector["name"] = UNITA_COSTIERE[key]
method={"Monte Carlo · normale troncata":"truncated_normal","Valori dei transetti":"measured","Intervallo minimo-massimo":"linear_range"}[MODALITA_PENDENZE]
count=NUM_CAMPIONI_MC if method=="truncated_normal" else NUM_PENDENZE if method=="linear_range" else 200
PENDENZE, SLOPE_SAMPLING = sample_slopes(scelti.slope.tolist(),method=method,count=count,seed=SEME_MONTE_CARLO,unit_id="whole")
SLOPE_SAMPLING_BY_UNIT = {}; rows=[]; sampled_rows=[]
for unit,name in UNITA_COSTIERE.items():
    local = scelti if unit=="whole" else scelti[scelti.transect_id.isin(SETTORI_SPIAGGIA[unit]["transect_ids"])]
    values,metadata = sample_slopes(local.slope.tolist(),method=method,count=count,seed=SEME_MONTE_CARLO,unit_id=unit)
    SLOPE_SAMPLING_BY_UNIT[unit] = metadata
    if unit!="whole":SETTORI_SPIAGGIA[unit].update(slopes=values,slope_sampling=metadata)
    rows.append({"Settore":name,"Transetti":metadata["source_count"],"Media misurata β":metadata["source_mean"],
        "Deviazione standard β":metadata["source_std"],"Minimo β":metadata["source_min"],"Massimo β":metadata["source_max"],"Campioni usati":len(values)})
    sampled_rows.extend({"sector_id":unit,"slope":v} for v in values)
    fig,ax=plt.subplots(figsize=(7,3))
    ax.hist(local.slope,bins="auto",alpha=.55,label="Transetti misurati",density=True,color="#768b6e")
    ax.hist(values,bins="auto",histtype="step",linewidth=2,label="Pendenze per i modelli",density=True,color="#9b632d")
    ax.set(title=name,xlabel="Pendenza β = dislivello / distanza",ylabel="Densità relativa");ax.legend();display(fig);plt.close(fig)
display(pd.DataFrame(rows).set_index("Settore").T)
pd.DataFrame(sampled_rows).to_csv(WORK/"pendenze_campionate.csv",index=False)
(WORK/"campionamento_pendenze.json").write_text(json.dumps(SLOPE_SAMPLING_BY_UNIT,ensure_ascii=False,indent=2))
SLOPE_OPTIONS_SIGNATURE=(MODALITA_PENDENZE,NUM_CAMPIONI_MC,SEME_MONTE_CARLO,NUM_PENDENZE)
preview_location=dict(SITE_INPUT,slopes=PENDENZE,roughness_m=float(RUGOSITA_M),transects=scelti.to_dict(orient="records"),wave_exposure=ESPOSIZIONE,sectors=SETTORI_SPIAGGIA)
resolve_sectors(preview_location, thresholds)
print("Campioni salvati: verranno riutilizzati nelle previsioni, senza nuove estrazioni giornaliere. Non sono probabilità di allagamento.")

## 6. Strutture esposte — una scheda per settore costiero
Qui definisci **quali strutture sono presenti**, il loro nome, la quota e l’attività estiva o annuale. Deseleziona quelle assenti e usa “Aggiungi struttura” per altre strutture. Gli elenchi possono essere diversi tra settori costieri. Se la spiaggia non è suddivisa, compare una sola scheda.

Con un GeoJSON dei punti, il modulo parte soltanto dalle strutture mappate in ogni settore costiero e dalla media delle quote valide lette dal DEM originale. Minimo, massimo e numero di punti restano visibili; la media non rappresenta la quota del primo punto che potrebbe essere raggiunto. Senza punti, le quattro strutture iniziali restano esempi da verificare, non vincoli. Le quote devono usare lo stesso riferimento verticale del TWL. Dopo le modifiche esegui la cella di verifica; l’esportazione legge nuovamente il modulo aggiornato.

In [ ]:
#@title Quote delle strutture dai punti sul DEM (facoltativo)
ESCLUDI_PUNTI_STRUTTURE_NON_VALIDI = False #@param {type:"boolean"}
PROPOSTE_STRUTTURE = None
STRUCTURE_POINT_META = None
if STRUCTURES_SOURCE:
    from forecast_service.structure_points import thresholds_from_structure_points
    PROPOSTE_STRUTTURE, PUNTI_STRUTTURE_REPORT, STRUTTURE_QUOTE_RIEPILOGO, STRUCTURE_POINT_META = thresholds_from_structure_points(
        DEM_SOURCE, STRUCTURES_SOURCE, POLIGONI_SETTORI,
        geojson_crs=CRS_GEOJSON_STRUTTURE, label_field=COL_STRUTTURA,
        elevation_correction_m=CORREZIONE_QUOTE_DEM_M)
    PUNTI_STRUTTURE_REPORT.to_csv(WORK/"quote_strutture_punti.csv", index=False)
    STRUTTURE_QUOTE_RIEPILOGO.to_csv(WORK/"quote_strutture_riepilogo.csv", index=False)
    display(STRUTTURE_QUOTE_RIEPILOGO[["sector_id", "structure", "mean_elevation_m", "min_elevation_m", "max_elevation_m", "n_valid", "n_excluded"]].rename(columns={"sector_id":"Settore costiero", "structure":"Struttura", "mean_elevation_m":"Quota media (m)", "min_elevation_m":"Quota minima (m)", "max_elevation_m":"Quota massima (m)", "n_valid":"Punti validi", "n_excluded":"Punti esclusi"}))
    invalidi = PUNTI_STRUTTURE_REPORT[PUNTI_STRUTTURE_REPORT.status != "valid"]
    if len(invalidi):
        display(invalidi[["point_index", "feature_id", "structure", "sector_id", "status", "reason"]])
        if not ESCLUDI_PUNTI_STRUTTURE_NON_VALIDI:
            raise ValueError("Ci sono punti senza quota valida o con assegnazione ambigua: correggi il GeoJSON oppure conferma esplicitamente l’esclusione con la casella sopra e riesegui.")
    print("Quote proposte: media aritmetica dei punti validi per struttura e settore costiero.")
    print("Correzione verticale del DEM applicata:", CORREZIONE_QUOTE_DEM_M, "m; la stessa dovrà essere usata per il DEM della mappa.")
    print("Per i punti senza campo stagione è proposta l’attività annuale: verifica e modifica nel modulo.")
else:
    print("Nessun GeoJSON dei punti: compila manualmente le strutture come prima.")

In [ ]:
#@title Compila le strutture esposte
from forecast_service.structure_forms import StructureEditor
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass
STRUTTURE_EDITOR = StructureEditor(UNITA_COSTIERE, thresholds, by_unit=PROPOSTE_STRUTTURE)
display(STRUTTURE_EDITOR.view)

In [ ]:
#@title Verifica le strutture per ciascun settore costiero
STRUTTURE_PER_UNITA = STRUTTURE_EDITOR.read()
for key, values in STRUTTURE_PER_UNITA.items():
    print(UNITA_COSTIERE[key])
    display(pd.DataFrame(values, columns=["short_label", "elevation_m", "season"])[["short_label", "elevation_m", "season"]])
    if key != "whole": SETTORI_SPIAGGIA[key]["thresholds"] = values
thresholds = STRUTTURE_PER_UNITA.get("whole", next(iter(STRUTTURE_PER_UNITA.values())))

## 7. Prepara le mappe: tre moduli, poi il controllo
Esegui nell’ordine **7A Crediti**, **7B Quote**, **7C Risoluzione** e **Prepara i ritagli**. DEM e ortofoto eventualmente presente sono già caricati. Il preprocessing prepara un ritaglio per settore oppure per la spiaggia intera; gli originali restano invariati.

Con ortofoto si pubblica un’immagine a 0,5 m/pixel. Senza ortofoto si usa il DEM con legenda delle quote; NoData ed esterno del ritaglio sono trasparenti, con OpenStreetMap sotto. Qui controlli i dati cartografici; le previsioni dei quattro giorni vengono aggiunte dal servizio giornaliero.

Con transetti GNSS, gli avvii delle mappe vengono controllati con campioni del **DEM**, orientati secondo il rilievo GNSS. Questi campioni servono soltanto alla cartografia e non modificano le pendenze GNSS. Le quote GNSS non vengono confrontate direttamente con il DEM per assumere un raccordo verticale.

In [ ]:
#@title 7A · Autore, data e licenza del rilievo
#@markdown Indica chi ha prodotto i dati, quando sono stati acquisiti e con quale licenza possono essere pubblicati. Queste informazioni compariranno sul sito.
CREDITI_RILIEVI = '' #@param {type:"string"}
DATA_RILIEVI = '' #@param {type:"string"}
LICENZA_RILIEVI = '' #@param {type:"string"}

In [ ]:
#@title 7B · Riferimento verticale delle quote
#@markdown Specifica lo zero del DEM. La correzione in metri si **somma** al DEM della mappa. Conferma il raccordo soltanto dopo aver verificato che DEM e livello totale dell’acqua abbiano lo stesso riferimento.
RIFERIMENTO_VERTICALE_MAPPA = '' #@param {type:"string"}
CORREZIONE_DEM_MAPPA_M = 0.0 #@param {type:"number"}
RACCORDO_VERTICALE_CONFERMATO = False #@param {type:"boolean"}

In [ ]:
#@title 7C · Risoluzione e margine del ritaglio
#@markdown L’ortofoto usa il passo indicato in metri/pixel, se presente. Il margine mantiene terreno circostante nel calcolo della connessione; non allarga l’area visualizzata. La griglia DEM cartografica è leggera: verifica le opere strette nell’anteprima.
RISOLUZIONE_ORTOFOTO_M = 0.5 #@param {type:"number"}
MARGINE_CLIPPING_M = 100.0 #@param {type:"number"}
#@markdown **Controllo avanzato degli avvii:** se il primo punto cade su NoData, la ricerca prosegue soltanto sullo stesso transetto entro questi limiti di distanza e rialzo. Non modifica il DEM. Inizia con i valori proposti e controlla gli spostamenti nel riepilogo.
MAX_AVANZAMENTO_AVVIO_M = 5.0 #@param {type:"number"}
MAX_RIALZO_AVVIO_M = 0.5 #@param {type:"number"}
#@markdown Per un’eccezione manuale: seleziona una sola unità e indica punti WGS84 come `[[longitudine, latitudine], ...]`. Lascia entrambi i campi vuoti per la ricerca automatica.
UNITA_AVVII_MANUALI = '' #@param {type:"string"}
PUNTI_AVVIO_MANUALI_JSON = '' #@param {type:"string"}

In [ ]:
#@title Prepara i ritagli per il sito
from forecast_service.mapping_setup import prepare_mapping, SeedSelectionError
if STRUCTURE_POINT_META and not np.isclose(CORREZIONE_DEM_MAPPA_M, STRUCTURE_POINT_META["elevation_correction_m"], rtol=0, atol=1e-9):
    raise ValueError("Punti delle strutture e mappa usano lo stesso DEM: verificare e uniformare la correzione verticale prima di preparare i ritagli.")
PUNTI_MARE_MANUALI = {}
if UNITA_AVVII_MANUALI.strip() or PUNTI_AVVIO_MANUALI_JSON.strip():
    if UNITA_AVVII_MANUALI.strip() not in UNITA_COSTIERE:
        raise ValueError("Seleziona un ID di unità esistente per gli avvii manuali.")
    PUNTI_MARE_MANUALI[UNITA_AVVII_MANUALI.strip()] = json.loads(PUNTI_AVVIO_MANUALI_JSON)
    if not PUNTI_MARE_MANUALI[UNITA_AVVII_MANUALI.strip()]:raise ValueError("Inserisci almeno un punto manuale.")
MAP_PROFILES = preparati
if PROFILE_SOURCE_KIND=="GNSS (LineString XYZ)":
    from forecast_service.gnss_setup import gnss_map_profiles
    auto_ids=set()
    for unit in POLIGONI_SETTORI:
        if unit in PUNTI_MARE_MANUALI:continue
        auto_ids.update(SETTORI_SPIAGGIA[unit]["transect_ids"] if unit!="whole" else scelti.transect_id)
    MAP_PROFILES={}
    if auto_ids:
        MAP_PROFILES,map_report,map_metadata=gnss_map_profiles(DEM_SOURCE,GEO_TRANSECTS_SOURCE,
            {key:preparati[key] for key in sorted(auto_ids)},profile_crs=CRS_PROFILI,
            geojson_crs=GNSS_FIELDS["geojson_crs"],id_field=GNSS_FIELDS["id_field"])
        map_report.to_csv(WORK/"campioni_dem_solo_avvii_mappa.csv",index=False)
        print("Avvii cartografici: quote campionate dal DEM lungo le linee GNSS, separatamente dalle pendenze. Controlla i punti sulla mappa.")
to_wgs = Transformer.from_crs(CRS_PROFILI, 4326, always_xy=True)
CARTOGRAFIE = []
for unit, polygon in POLIGONI_SETTORI.items():
    ids = SETTORI_SPIAGGIA[unit]["transect_ids"] if unit != "whole" else list(scelti.transect_id)
    seeds = PUNTI_MARE_MANUALI.get(unit)
    seed_profiles = None
    if seeds is None:
        seed_profiles = {}
        for key in ids:
            # Already oriented sea → land; select only the initial coastal prefix.
            profile = MAP_PROFILES[key]
            prefix = profile.loc[profile["x_mare"] <= MAX_AVANZAMENTO_AVVIO_M+1e-8]
            samples = []
            for x, y, distance, source_z in prefix[[COL_X, COL_Y, "x_mare", COL_Z]].itertuples(index=False, name=None):
                lon, lat = to_wgs.transform(float(x), float(y))
                map_z = float(source_z)+CORREZIONE_DEM_MAPPA_M-(CORREZIONE_QUOTE_DEM_M if PROFILE_SOURCE_KIND=="DEM + linee" else 0.)
                samples.append([lon, lat, float(distance), map_z])
            seed_profiles[str(key)] = samples
    CARTOGRAFIE.append(dict(unit_id=unit, dem_path=str(DEM_SOURCE), orthophoto_path=str(ORTHO_SOURCE) if ORTHO_SOURCE else None,
        display_polygon=polygon, seed_points=seeds, seed_profiles=seed_profiles,
        seed_max_advance_m=MAX_AVANZAMENTO_AVVIO_M, seed_max_rise_m=MAX_RIALZO_AVVIO_M, survey_credit="DEMO sintetico" if DEMO else CREDITI_RILIEVI,
        survey_date="Dati sintetici" if DEMO else DATA_RILIEVI, license="Solo dimostrazione" if DEMO else LICENZA_RILIEVI,
        vertical_reference="Zero sintetico comune" if DEMO else RIFERIMENTO_VERTICALE_MAPPA,
        dem_offset_m=CORREZIONE_DEM_MAPPA_M,
        vertical_alignment_confirmed=True if DEMO else RACCORDO_VERTICALE_CONFERMATO,
        clip_to_polygon=True, clip_buffer_m=MARGINE_CLIPPING_M,
        orthophoto_resolution_m=RISOLUZIONE_ORTOFOTO_M))
MAPPING_CONFIGS = {}; CARTOGRAPHY_ASSETS = {}
for spec in CARTOGRAFIE:
    unit = spec["unit_id"]
    try:
        metadata, assets = prepare_mapping(output=WORK/"cartography"/unit, slug=ID_SITO, **spec)
    except SeedSelectionError as error:
        avvii = pd.DataFrame(error.report)
        avvii.to_csv(WORK/f"avvii_mare_{unit}.csv", index=False)
        display(avvii[["transect_id", "status", "advance_m", "grid_elevation_m"]])
        raise
    avvii = pd.DataFrame(metadata["seed_selection"])
    avvii.to_csv(WORK/f"avvii_mare_{unit}.csv", index=False)
    adjusted=avvii[avvii.advance_m>1e-8]
    display(pd.DataFrame({"Controllo":["Transetti verificati","Avvii avanzati sul transetto","Avanzamento massimo (m)"],
        "Valore":[len(avvii),len(adjusted),round(float(avvii.advance_m.max()),3)]}).set_index("Controllo"))
    print(UNITA_COSTIERE[unit], "· elenco completo degli avvii:")
    display(FileLink(str(WORK/f"avvii_mare_{unit}.csv")))
    MAPPING_CONFIGS[unit] = metadata; CARTOGRAPHY_ASSETS.update(assets)
    from forecast_service.inundation import load_grid
    grid, seeds, mask, bounds = load_grid(assets[f"cartography/{unit}/grid.npz"])
    import base64, html
    from forecast_service.terrain_background import terrain_background
    controllo=folium.Map(location=[(bounds[0][0]+bounds[1][0])/2,(bounds[0][1]+bounds[1][1])/2],tiles="OpenStreetMap")
    ortho_preparata=assets.get(f"cartography/{unit}/orthophoto.png")
    if ortho_preparata:
        folium.raster_layers.ImageOverlay(str(ortho_preparata),bounds=bounds,name="Ortofoto del rilievo",opacity=1).add_to(controllo)
    else:
        background,scale=terrain_background(grid,mask)
        folium.raster_layers.ImageOverlay("data:image/png;base64,"+base64.b64encode(background).decode(),bounds=bounds,name="DEM · quote del terreno",opacity=1).add_to(controllo)
        colors=",".join(scale["colors"])
        legend=f'<div style="position:fixed;bottom:25px;left:15px;z-index:9999;background:white;padding:8px">Quota DEM: {scale["min_m"]:.2f} m <span style="display:inline-block;width:100px;height:12px;background:linear-gradient(to right,{colors})"></span> {scale["max_m"]:.2f} m</div>'
        controllo.get_root().html.add_child(folium.Element(legend))
    avvii_layer=folium.FeatureGroup(name="Punti di avvio verso mare")
    for entry in metadata["seed_selection"]:
        lon,lat=entry.get("grid_cell_center",entry["selected_point"])
        folium.CircleMarker([lat,lon],radius=3,color="#bd2a22",fill=True,fill_opacity=1,
            tooltip=html.escape(str(entry["transect_id"]))+f" · avanzamento {entry['advance_m']:.2f} m").add_to(avvii_layer)
    avvii_layer.add_to(controllo)
    folium.LayerControl(collapsed=False).add_to(controllo)
    controllo.fit_bounds(bounds)
    print("Controllo cartografico ·",UNITA_COSTIERE[unit],"· non è una previsione")
    display(controllo)
MAP_ANALYSIS_SIGNATURE = ANALYSIS_SIGNATURE
CARTOGRAPHY_SIGNATURE = json.dumps(CARTOGRAFIE, sort_keys=True)
print("Ritagli pronti:", list(MAPPING_CONFIGS), "· sfondo:", "ortofoto" if ORTHO_SOURCE else "DEM a colori + OpenStreetMap")

### 7.1 Prova del limite sul DSM — sole linee

Questo passaggio facoltativo lavora **sulla rappresentazione del limite**, senza creare un DTM, riempire buchi o modificare il DSM. Le pendenze e le quote delle strutture restano quelle dei passaggi precedenti.

1. Scegli una **quota di prova**, nello stesso riferimento del DEM cartografico già corretto. È una prova geometrica, non una previsione.
2. Il calcolo originale trova le celle sotto quota connesse ai punti a mare.
3. Si estrae soltanto il fronte esterno fra celle osservate raggiunte e terreno esterno non raggiunto. I contorni degli isolotti interni, come quelli degli ombrelloni, vengono omessi; i bordi NoData e il limite del ritaglio non diventano falsi fronti di risalita.
4. Si regolarizza la linea con una tolleranza in metri. **0 mantiene la linea originale**; partire da valori piccoli confrontandoli con risoluzione e precisione del rilievo.

La mappa mostra linee sull’ortofoto, senza riempimenti. Puoi accendere il limite originale e confrontarlo con quello regolarizzato. La regolarizzazione può spostare il tratto disegnato entro la tolleranza: **non corregge la propagazione bloccata da false barriere del DSM** e non viene usata per soglie o misure di superficie.

Gli eventuali GeoJSON, PNG e metadati esportati sono **diagnostica di prova**: non attivano la regolarizzazione nelle previsioni pubblicate. I GeoTIFF e il calcolo di connessione restano invariati.

In [ ]:
#@title Quota di prova e regolarizzazione della linea
ESEGUI_PROVA_LINEA = False #@param {type:"boolean"}
QUOTA_TWL_PROVA_M = 1.4 #@param {type:"number"}
TOLLERANZA_LINEA_M = 1.0 #@param {type:"number"}
UNITA_LINEA_DA_VEDERE = '' #@param {type:"string"}
print("Unità disponibili:", list(MAPPING_CONFIGS))
print("Lascia vuota l’unità per vedere la prima; inserisci un ID per cambiare settore costiero.")
if not ESEGUI_PROVA_LINEA:
    print("Prova disattivata: puoi proseguire il setup senza modifiche.")

In [ ]:
#@title Confronta il limite originale e la linea regolarizzata
from forecast_service.contour_preview import preview_from_grid
LINE_PREVIEW_META = None
LINE_PREVIEW_ATTACHMENTS = {}
if ESEGUI_PROVA_LINEA:
    if not MAPPING_CONFIGS:
        raise ValueError("Prima prepara i ritagli cartografici.")
    unit = UNITA_LINEA_DA_VEDERE.strip() or next(iter(MAPPING_CONFIGS))
    if unit not in MAPPING_CONFIGS:
        raise ValueError("Unità sconosciuta: scegli un ID dalla lista della cella precedente.")
    grid_path = CARTOGRAPHY_ASSETS[f"cartography/{unit}/grid.npz"]
    original_line, display_line, line_meta = preview_from_grid(
        grid_path, level_m=QUOTA_TWL_PROVA_M, tolerance_m=TOLLERANZA_LINEA_M,
        display_polygon=POLIGONI_SETTORI[unit])
    line_meta["unit_id"] = unit
    line_meta["data_mode"] = "demo" if DEMO else "study"
    line_meta["vertical_reference"] = MAPPING_CONFIGS[unit]["vertical_reference"]
    line_meta["dem_offset_m"] = MAPPING_CONFIGS[unit]["dem_offset_m"]
    preview_dir = WORK / "prova_linee"
    preview_dir.mkdir(exist_ok=True)
    diagnostic_paths = {}
    for name, value in [(f"limite_originale_{unit}.geojson", original_line),
                        (f"limite_regolarizzato_{unit}.geojson", display_line),
                        (f"prova_linea_{unit}.json", line_meta)]:
        target = preview_dir / name
        target.write_text(json.dumps(value, ensure_ascii=False, allow_nan=False, indent=2)+"\n")
        diagnostic_paths[name] = target
    bounds = line_meta["bounds_wgs84"]
    preview_map = folium.Map(location=[(bounds[0][0]+bounds[1][0])/2,
                                       (bounds[0][1]+bounds[1][1])/2], tiles="OpenStreetMap")
    ortho_path = CARTOGRAPHY_ASSETS.get(f"cartography/{unit}/orthophoto.png")
    if ortho_path:
        folium.raster_layers.ImageOverlay(image=str(ortho_path), bounds=bounds,
            name="Ortofoto del rilievo", opacity=1).add_to(preview_map)
    if not ortho_path:
        import base64
        from forecast_service.terrain_background import terrain_background
        dem_grid,_,dem_mask,_=load_grid(grid_path)
        background,scale=terrain_background(dem_grid,dem_mask)
        folium.raster_layers.ImageOverlay("data:image/png;base64,"+base64.b64encode(background).decode(),
            bounds=bounds,name="DEM · quote del terreno",opacity=1).add_to(preview_map)
        colors=",".join(scale["colors"])
        legend=f'<div style="position:fixed;bottom:25px;left:15px;z-index:9999;background:white;padding:8px">Quota DEM: {scale["min_m"]:.2f} m <span style="display:inline-block;width:100px;height:12px;background:linear-gradient(to right,{colors})"></span> {scale["max_m"]:.2f} m</div>'
        preview_map.get_root().html.add_child(folium.Element(legend))
    if original_line["features"]:
        folium.GeoJson(original_line, name="Limite originale (stesso calcolo)", show=False,
            style_function=lambda f: {"color":"#555555", "weight":2, "dashArray":"4,4"}).add_to(preview_map)
    if display_line["features"]:
        folium.GeoJson(display_line, name="Linea regolarizzata — solo visualizzazione",
            style_function=lambda f: {"color":"#176ca4", "weight":3}).add_to(preview_map)
    folium.LayerControl(collapsed=False).add_to(preview_map)
    preview_map.fit_bounds(bounds)
    html_path = preview_dir / f"confronto_linee_{unit}.html"
    preview_map.save(str(html_path))
    display(preview_map)
    # Static comparison in the prepared map CRS; PNG is diagnostic only.
    from forecast_service.geospatial_setup import reproject_geometry
    from shapely.geometry import shape
    left, bottom, right, top = line_meta["bounds_3857"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), layout="constrained")
    def disegna_linee(ax, geometry, color):
        if geometry.is_empty:
            return
        if geometry.geom_type == "LineString":
            ax.plot(*geometry.xy, color=color, linewidth=1.4)
        elif hasattr(geometry, "geoms"):
            for part in geometry.geoms:
                disegna_linee(ax, part, color)
    for ax, collection, title, color in zip(axes, [original_line, display_line],
            ["Limite originale", "Linea regolarizzata (solo disegno)"], ["#555555", "#176ca4"]):
        if ortho_path:
            ax.imshow(plt.imread(ortho_path), extent=[left,right,bottom,top])
        else:
            terrain=ax.imshow(np.where(dem_mask,dem_grid,np.nan),extent=[left,right,bottom,top],cmap=LinearSegmentedColormap.from_list("quote_dem",scale["colors"]),vmin=scale["min_m"],vmax=scale["max_m"])
            fig.colorbar(terrain,ax=ax,label="Quota DEM (m)",shrink=.7)
        for feature in collection["features"]:
            disegna_linee(ax, reproject_geometry(shape(feature["geometry"]), 4326, 3857), color)
        ax.set(xlim=(left,right), ylim=(bottom,top), title=title)
        ax.set_aspect("equal"); ax.set_axis_off()
    fig.suptitle(f"{UNITA_COSTIERE[unit]} · quota di prova {QUOTA_TWL_PROVA_M:g} m · tolleranza {TOLLERANZA_LINEA_M:g} m")
    png_path = preview_dir / f"confronto_linee_{unit}.png"
    fig.savefig(png_path, dpi=140)
    display(fig); plt.close(fig)
    diagnostic_paths[png_path.name] = png_path
    LINE_PREVIEW_META = line_meta
    LINE_PREVIEW_ATTACHMENTS = diagnostic_paths
    display(pd.DataFrame([{"Quota di prova (m)":line_meta["level_m"],
        "Tolleranza (m)":line_meta["tolerance_m"], "Vertici originali":line_meta["original_vertices"],
        "Vertici regolarizzati":line_meta["simplified_vertices"],
        "Isolotti interni non disegnati":line_meta["internal_components_not_drawn"],
        "Celle riempite":line_meta["filled_cells"]}]))
    if line_meta["empty"]:
        print("Nessun fronte osservabile per questa quota: controlla punti a mare, quota, copertura e NoData. Il bordo del DEM non è un limite di risalita.")
    if line_meta["seeds_below_level"] == 0:
        print("La quota è sotto tutti i punti di avvio: la prova non raggiunge celle.")
    print("DSM, griglia originale, pendenze, soglie e maschera di connessione non modificati.")
    print("HTML interattivo salvato:", html_path)
    display(FileLink(str(html_path)))

## 8. Serie storica ERA5 — unica per tutta l’area
Per dati reali servono un account Climate Data Store e l’accettazione personale delle condizioni di [ERA5 time-series](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=download). La ricerca automatica della cella marina usa anche [ERA5 grigliato](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels?tab=download).

Puoi salvare il token nei **Secrets di Colab**, con nome `CDS_KEY` e accesso a questo notebook, oppure inserirlo nel campo protetto quando richiesto. Non scriverlo nei campi del notebook. La cache contiene soltanto richieste e dati: sarà riutilizzata se la sessione riparte dalla stessa cartella.

Il servizio ERA5 restituisce **periodo medio**, non Tp: lo salviamo come `tm_s`; non lo usiamo come periodo di picco per il runup. Il confronto storico usa Hs. Il confronto con tutte le ore storiche è distinto dal fit Gumbel dei massimi annuali per settore descritto sotto.

### Tempi di ritorno per direzione delle onde (non settori costieri)
La suddivisione statistica in settori di provenienza è distinta dall’esposizione fisica della spiaggia. Puoi scegliere l’ampiezza dei settori: nessuna direzione è privilegiata. I settori sono centrati su multipli dell’ampiezza scelta; per esempio, con 45°, quello centrato a 90° va da 67,5° incluso a 112,5° escluso. Le direzioni indicano **da dove arrivano** le onde.

Per ogni anno di calendario UTC completo si seleziona il massimo Hs **fra tutte le ore di ciascun settore**, non si suddivide il solo massimo annuale generale. Si usano soltanto anni con copertura oraria completa e direzione valida (la direzione può mancare nelle ore di calma con Hs=0). Se un settore non ha onde positive in qualche anno completo, il fit non viene mostrato: rimuovere quegli anni condizionerebbe la frequenza.

Il fit è una **Gumbel sui massimi annuali** (massima verosimiglianza), con banda bootstrap al 95% ricampionando gli anni. Questa prima versione richiede 30 anni completi; una serie lunga non dimostra da sola che Gumbel sia adeguata. Si assumono stazionarietà e indipendenza dei massimi annuali. Sul sito il picco Hs dei quattro giorni viene confrontato con il settore della sua direzione in quell’ora; si possono esaminare anche i picchi negli altri settori presenti.

Il tempo di ritorno riguarda Hs al largo per quel settore, non runup o allagamenti. La differenza fra ERA5 e Copernicus non è corretta e non rientra nella banda del fit. Non usare queste elaborazioni sperimentali per previsioni o scelte gestionali.

In [ ]:
#@title Impostazioni dei massimi annuali per settore
CALCOLA_ESTREMI_SETTORIALI = True #@param {type:"boolean"}
AMPIEZZA_SETTORI_ESTREMI = 45 #@param [30,45,60,90,120,180,360]
COL_DIREZIONE_ERA5 = 'direction_deg' #@param {type:"string"}
MIN_ANNI_ESTREMI = 30
REPLICHE_BOOTSTRAP = 400
print("Settori statistici:",AMPIEZZA_SETTORI_ESTREMI,"gradi; minimo",MIN_ANNI_ESTREMI,"anni completi.")

In [ ]:
#@title Sorgente e periodo storico
MODALITA_ERA5 = 'Scarica ERA5' #@param ["Scarica ERA5", "Importa CSV", "Salta per ora"]
ERA5_INIZIO = '1940-01-01' #@param {type:"date"}
ERA5_FINE = '2025-12-31' #@param {type:"date"}
CONSENSO_DOWNLOAD_ERA5 = True #@param {type:"boolean"}
CELLA_ERA5_NOTA = False #@param {type:"boolean"}
LAT_CELLA_ERA5 = 44.08 #@param {type:"number"}
LON_CELLA_ERA5 = 8.33 #@param {type:"number"}
FILE_CSV_ERA5 = '' #@param {type:"string"}
COL_TEMPO_ERA5 = 'time' #@param {type:"string"}
COL_HS_ERA5 = 'hs' #@param {type:"string"}
SCARTA_HS_MANCANTI = False #@param {type:"boolean"}
CLIMATE=None
CLIMATE_DATA_MODE="demo" if DEMO else "study"
climate_metadata={}
CACHE=BASE/"_cache_era5"
CACHE.mkdir(parents=True,exist_ok=True)
if DEMO:
    # Synthetic reference, clearly labelled; never presented as an ERA5 download.
    time=pd.date_range("2019-01-01", "2020-12-31 23:00",freq="h",tz="UTC")
    index=np.arange(len(time))
    hs=.25+1.1*(.5+.5*np.sin(index/179))**3+.3*(.5+.5*np.sin(index/37))
    era5_frame=pd.DataFrame({"time":time,"hs_m":hs,"direction_deg":(index*13)%360})
    history=WORK/"clima_dimostrativo.csv"; era5_frame.to_csv(history,index=False)
    CLIMATE=build_climate(history,"Esempio sintetico; non ERA5")
    climate_metadata={"mode":"demo","description":"Distribuzione sintetica per verificare il notebook."}
elif MODALITA_ERA5=="Scarica ERA5":
    if not CONSENSO_DOWNLOAD_ERA5:
        raise ValueError("Accetta le condizioni sul CDS e imposta CONSENSO_DOWNLOAD_ERA5=True per avviare la richiesta.")
    import os
    if not os.environ.get("CDS_KEY"):
        token=None
        try:
            from google.colab import userdata
            token=userdata.get("CDS_KEY")
        except Exception:
            pass
        if not token:
            from getpass import getpass
            token=getpass("Token CDS (input nascosto): ")
        if not token: raise ValueError("Token CDS non disponibile.")
        os.environ["CDS_KEY"]=token
        del token
    known_cell=(LON_CELLA_ERA5,LAT_CELLA_ERA5) if CELLA_ERA5_NOTA else None
    era5_frame,climate_metadata=download_series(SITE_INPUT,dt.date.fromisoformat(ERA5_INIZIO),dt.date.fromisoformat(ERA5_FINE),CACHE,cell=known_cell)
    history=WORK/"ERA5_hourly.csv"; era5_frame.to_csv(history,index=False)
    if climate_metadata["missing_hs_records"] and not SCARTA_HS_MANCANTI:
        raise ValueError("ERA5 ha ore mancanti: CSV conservato. Esamina i dati; puoi autorizzare lo scarto con SCARTA_HS_MANCANTI.")
    if SCARTA_HS_MANCANTI:
        climate_metadata["dropped_missing_rows"]=int(era5_frame.hs_m.isna().sum())
        era5_frame=era5_frame.dropna(subset=["hs_m"])
        history=WORK/"ERA5_hs_valide.csv"; era5_frame.to_csv(history,index=False)
    CLIMATE=build_climate(history,"ERA5")
elif MODALITA_ERA5=="Importa CSV":
    if not FILE_CSV_ERA5:
        from google.colab import files
        uploaded=files.upload()
        if len(uploaded)!=1: raise ValueError("Carica un solo CSV storico.")
        name,content=next(iter(uploaded.items()))
        history=WORK/Path(name).name; history.write_bytes(content)
    else:
        history=Path(FILE_CSV_ERA5)
    era5_frame=pd.read_csv(history)
    if SCARTA_HS_MANCANTI:
        climate_metadata["dropped_missing_rows"]=int(era5_frame[COL_HS_ERA5].isna().sum())
        era5_frame=era5_frame.dropna(subset=[COL_HS_ERA5])
        history=WORK/"ERA5_import_hs_valide.csv"; era5_frame.to_csv(history,index=False)
    CLIMATE=build_climate(history,"ERA5 (CSV dell’utente)",time_column=COL_TEMPO_ERA5,hs_column=COL_HS_ERA5)
else:
    print("Climatologia saltata: la località potrà essere esportata senza riferimento storico.")
if CLIMATE is not None:
    if CALCOLA_ESTREMI_SETTORIALI:
        imported=not DEMO and MODALITA_ERA5=="Importa CSV"
        CLIMATE["directional_extremes"]=build_directional_extremes(history,
            time_column=COL_TEMPO_ERA5 if imported else "time",
            hs_column=COL_HS_ERA5 if imported else "hs_m",
            direction_column=COL_DIREZIONE_ERA5 if imported else "direction_deg",
            sector_width_deg=int(AMPIEZZA_SETTORI_ESTREMI),min_years=MIN_ANNI_ESTREMI,
            bootstrap_replicates=REPLICHE_BOOTSTRAP)
        ext=CLIMATE["directional_extremes"]
        if ext["status"]=="available":
            display(pd.DataFrame([{"settore":f'{s["start_deg"]:g}° → {s["end_deg"]:g}°',
                "anni":s["fit"]["n_years"],"fit":s["fit"]["status"],"nota":s["fit"].get("reason","")}
                for s in ext["sectors"]]))
        else: print(ext["reason"])
    CLIMATE["download"]=climate_metadata
    if not DEMO: CLIMATE["doi"]="10.24381/1cf1ad76"
    (WORK/"climatology.json").write_text(json.dumps(CLIMATE,ensure_ascii=False,allow_nan=False,separators=(",",":")))
    display(pd.DataFrame([{k:CLIMATE[k] for k in ["source","start","end","n_records","p95_m","p99_m","max_hs_m","coverage_fraction"]}]))

In [ ]:
#@title Verifica la distribuzione e i massimi annuali
if CLIMATE is not None:
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    axes[0].hist(CLIMATE["hs_sorted_m"],bins=50,color="#527d78",density=True)
    for key,label,color in [("p95_m","P95","#b39245"),("p99_m","P99","#b65f44")]:
        axes[0].axvline(CLIMATE[key],ls="--",color=color,label=f"{label}: {CLIMATE[key]:.2f} m")
    axes[0].set(xlabel="Hs [m]",ylabel="Densità",title=CLIMATE["source"])
    axes[0].legend()
    annual=pd.DataFrame.from_dict(CLIMATE["annual_maxima"],orient="index")
    axes[1].bar(annual.index,annual.hs_m,color="#527d78")
    axes[1].tick_params(axis="x",labelrotation=60)
    axes[1].set(xlabel="Anno",ylabel="Hs massima osservata [m]",title="Massimi annuali osservati, non valori di ritorno")
    fig.tight_layout(); fig.savefig(WORK/"climatologia.png",dpi=150); plt.show()
    annual.to_csv(WORK/"massimi_annuali.csv",index_label="year")
    display(annual)
    if "cell_lat" in climate_metadata:
        m=folium.Map(location=[LAT_ONDE,LON_ONDE],zoom_start=8)
        folium.Marker([LAT_SITO,LON_SITO],tooltip="Sito").add_to(m)
        folium.CircleMarker([LAT_ONDE,LON_ONDE],radius=6,color="green",tooltip="Punto onde richiesto").add_to(m)
        folium.CircleMarker([climate_metadata["cell_lat"],climate_metadata["cell_lon"]],radius=7,color="orange",tooltip="Cella marina ERA5 usata").add_to(m)
        display(m)
else:
    print("Nessuna climatologia selezionata.")

In [ ]:
#@title Verifica le curve Gumbel dei settori
ext=CLIMATE.get("directional_extremes") if CLIMATE else None
if ext and ext["status"]=="available":
    valid=[s for s in ext["sectors"] if s["fit"]["status"]=="available"]
    if valid:
        fig,axes=plt.subplots((len(valid)+1)//2,2,figsize=(12,4*((len(valid)+1)//2)),squeeze=False)
        for ax,sector in zip(axes.flat,valid):
            fit=sector["fit"]; periods=fit["return_period_years"]
            ax.fill_between(periods,fit["ci95_low_m"],fit["ci95_high_m"],alpha=.2,color="#527d78",label="Bootstrap 95%")
            ax.plot(periods,fit["return_level_m"],color="#194e4c",label="Gumbel")
            ax.scatter([r["return_period_years"] for r in fit["empirical"]],[r["hs_m"] for r in fit["empirical"]],s=12,label="Massimi annuali")
            ax.set(xscale="log",xlabel="Tempo di ritorno [anni]",ylabel="Hs al largo [m]",title=f'{sector["start_deg"]:g}° → {sector["end_deg"]:g}° · {fit["n_years"]} anni')
            ax.set_ylim(bottom=0); ax.legend(fontsize=8)
        for ax in list(axes.flat)[len(valid):]: ax.set_visible(False)
        fig.tight_layout(); fig.savefig(WORK/"estremi_settoriali.png",dpi=150); plt.show()
    else: print("Nessun settore soddisfa i requisiti per il fit. Esamina le note nella tabella.")
else: print("Estremi settoriali non configurati.")

### Zero del livello marino: riferimento storico fisso Copernicus
Questa sezione è separata da ERA5: usa la **SSH del modello fisico Copernicus**, che comprende la marea e le variazioni atmosferiche e oceanografiche. Si calcola una volta la media di un anno UTC completo nella stessa cella usata per le previsioni, e la si conserva per gli aggiornamenti futuri. Non si sottrae la media dei giorni di previsione.

Per Borghetto è incluso il riferimento già verificato del **2025**: 8.760 ore, media SSH circa −0,4032 m. Puoi usarlo senza scaricare nuovamente dati o rifare ERA5. Il controllo rifiuta di applicarlo se cambi il punto del livello marino. Per un altro sito scegli **Calcola da Copernicus** e un anno completo disponibile nell’archivio: il download richiede l’account Copernicus Marine, distinto da quello CDS/ERA5. Le credenziali si leggono dai Secrets Colab o da input protetto e non sono esportate.

Lo zero così ottenuto è la **media del modello nell’anno dichiarato**, non una calibrazione del livello medio mare misurato sulla spiaggia. Il raccordo con le quote delle strutture rimane provvisorio. Eventuali correzioni locali devono restare esplicite nel campo `OFFSET_LIVELLO_M`.

In [ ]:
#@title Riferimento storico del livello marino
MODALITA_RIFERIMENTO_SSH = 'Usa riferimento incluso' #@param ["Usa riferimento incluso", "Calcola da Copernicus", "Importa JSON"]
ANNO_RIFERIMENTO_SSH = 2025 #@param {type:"integer"}
CONSENSO_DOWNLOAD_SSH = False #@param {type:"boolean"}
FILE_RIFERIMENTO_SSH = '' #@param {type:"string"}
SEA_LEVEL_REFERENCE=None
reference_location=dict(SITE_INPUT)
if not LIVELLO_STESSO_PUNTO:
    reference_location["sealevel_point"]=dict(latitude=LAT_LIVELLO,longitude=LON_LIVELLO)
if DEMO:
    print("Modalità demo: il livello marino resta sintetico; nessun riferimento reale viene usato.")
elif MODALITA_RIFERIMENTO_SSH=="Usa riferimento incluso":
    reference_file=RUNTIME/"config"/"study_sites"/ID_SITO/f"sea_level_reference_{ANNO_RIFERIMENTO_SSH}.json"
    if not reference_file.exists():
        raise ValueError("Nessun riferimento incluso per questo sito/anno: scegli Calcola da Copernicus o Importa JSON.")
    SEA_LEVEL_REFERENCE=json.loads(reference_file.read_text())
elif MODALITA_RIFERIMENTO_SSH=="Importa JSON":
    if FILE_RIFERIMENTO_SSH:
        SEA_LEVEL_REFERENCE=json.loads(Path(FILE_RIFERIMENTO_SSH).read_text())
    else:
        from google.colab import files
        uploaded=files.upload()
        if len(uploaded)!=1: raise ValueError("Carica un solo riferimento JSON.")
        SEA_LEVEL_REFERENCE=json.loads(next(iter(uploaded.values())))
else:
    if not CONSENSO_DOWNLOAD_SSH:
        raise ValueError("Per scaricare la SSH storica imposta CONSENSO_DOWNLOAD_SSH=True.")
    from getpass import getpass
    def account_secret(name,prompt):
        import os
        value=os.environ.get(name)
        if not value:
            try:
                from google.colab import userdata
                value=userdata.get(name)
            except Exception:
                pass
        return value or getpass(prompt)
    username=account_secret("COPERNICUSMARINE_USERNAME","Username Copernicus Marine: ")
    password=account_secret("COPERNICUSMARINE_PASSWORD","Password Copernicus Marine: ")
    try:
        SEA_LEVEL_REFERENCE=download_reference(reference_location,int(ANNO_RIFERIMENTO_SSH),username,password,cache=BASE/"_cache_ssh")
    finally:
        del username,password
if SEA_LEVEL_REFERENCE is not None:
    validate_reference(SEA_LEVEL_REFERENCE,location=reference_location)
    if SEA_LEVEL_REFERENCE["year"]!=ANNO_RIFERIMENTO_SSH:
        raise ValueError("L’anno del riferimento JSON non corrisponde a quello selezionato.")
    (WORK/f"sea_level_reference_{ANNO_RIFERIMENTO_SSH}.json").write_text(json.dumps(SEA_LEVEL_REFERENCE,ensure_ascii=False,allow_nan=False,indent=2))
    display(pd.DataFrame([{"riferimento":f'Media SSH Copernicus {SEA_LEVEL_REFERENCE["year"]}',"media SSH (m)":SEA_LEVEL_REFERENCE["mean_ssh_m"],"ore":SEA_LEVEL_REFERENCE["n_records"],"cella":SEA_LEVEL_REFERENCE["selected_point"],"versione dataset":SEA_LEVEL_REFERENCE["dataset_version"]}]))
    print("Questo valore resterà fisso nei futuri bollettini. Il raccordo con le quote locali è ancora da verificare.")

## 9. Rivedi ed esporta l’area
Il pacchetto contiene configurazione, soglie, pendenze e relativa diagnostica, profili normalizzati e climatologia quando disponibile. **Conserva la cache o il CSV storico in Drive**: il pacchetto riporta il riferimento statistico, non tutti i NetCDF delle richieste.

Un pacchetto prodotto in modalità demo è marcato come tale e l’importatore lo rifiuta per impostazione predefinita. Nessun token entra nel pacchetto. Nessun file del portale viene modificato da Colab.

In [ ]:
#@title Riepilogo prima dell’esportazione
from forecast_service.time_utils import iso
import hashlib
line_preview_metadata = None
if globals().get("ESEGUI_PROVA_LINEA", False):
    line_preview_metadata = globals().get("LINE_PREVIEW_META")
    preview_unit = UNITA_LINEA_DA_VEDERE.strip() or next(iter(MAPPING_CONFIGS))
    if not line_preview_metadata or preview_unit not in MAPPING_CONFIGS:
        raise ValueError("Riesegui il confronto delle linee prima di esportare la diagnostica.")
    preview_grid = CARTOGRAPHY_ASSETS[f"cartography/{preview_unit}/grid.npz"]
    if (line_preview_metadata["unit_id"] != preview_unit or
        line_preview_metadata["level_m"] != QUOTA_TWL_PROVA_M or
        line_preview_metadata["tolerance_m"] != TOLLERANZA_LINEA_M or
        line_preview_metadata["grid_sha256"] != hashlib.sha256(Path(preview_grid).read_bytes()).hexdigest()):
        raise ValueError("Quota, tolleranza, settore o griglia cambiati: riesegui il confronto delle linee.")
if NOMI_SETTORI_EDITOR.read()!=UNITA_COSTIERE or SLOPE_OPTIONS_SIGNATURE!=(MODALITA_PENDENZE,NUM_CAMPIONI_MC,SEME_MONTE_CARLO,NUM_PENDENZE):
    raise ValueError("Nomi o opzioni delle pendenze cambiati: riesegui Conferma nomi e campiona, poi le celle successive.")
if ASSEGNAZIONI_EDITOR and ASSEGNAZIONI_EDITOR.read()!=ASSEGNAZIONI_MANUALI:
    raise ValueError("Assegnazioni cambiate: riesegui Assegna i transetti e le celle successive.")
if PROFILE_SOURCE_KIND!=("DEM + linee" if DEMO else SORGENTE_PENDENZE):
    raise ValueError("Sorgente delle pendenze cambiata: riesegui lettura e analisi.")
if GNSS_FIELD_EDITOR:
    if GNSS_FIELD_EDITOR.read()!=GNSS_FIELDS or hashlib.sha256(GEO_TRANSECTS_SOURCE.read_bytes()).hexdigest()!=GEO_PREPROCESSING["transects_sha256"]:
        raise ValueError("File o campi GNSS cambiati: riesegui lettura e analisi prima di esportare.")
current_signature=(PROFILE_SOURCE_KIND,FIT_EDITOR.read(),PROFILE_CONFIG_SIGNATURE,COL_ID,COL_DIST,COL_Z,COL_X,COL_Y,
    hashlib.sha256(pd.util.hash_pandas_object(profili,index=True).values.tobytes()).hexdigest())
if PROFILE_DATA_MODE!=("demo" if DEMO else "study") or (CLIMATE is not None and CLIMATE_DATA_MODE!=("demo" if DEMO else "study")):
    raise ValueError("Modalità cambiata: riesegui caricamento profili, pendenze e climatologia prima di esportare.")
if current_signature!=ANALYSIS_SIGNATURE or SELECTION_ANALYSIS_SIGNATURE!=ANALYSIS_SIGNATURE or PLOT_ANALYSIS_SIGNATURE!=ANALYSIS_SIGNATURE:
    raise ValueError("Parametri o profili cambiati: riesegui calcolo, grafico e selezione delle pendenze.")
if STRUCTURE_POINT_META:
    if hashlib.sha256(STRUCTURES_SOURCE.read_bytes()).hexdigest()!=STRUCTURE_POINT_META["points_sha256"] or COL_STRUTTURA!=STRUCTURE_POINT_META["label_field"]:
        raise ValueError("Punti delle strutture cambiati: riesegui campionamento e modulo delle strutture.")
    if not np.isclose(CORREZIONE_QUOTE_DEM_M,STRUCTURE_POINT_META["elevation_correction_m"],rtol=0,atol=1e-9):
        raise ValueError("Correzione delle quote cambiata: riesegui i campionamenti.")
    if hashlib.sha256(json.dumps(POLIGONI_SETTORI,sort_keys=True).encode()).hexdigest()!=STRUCTURE_POINT_META["polygons_sha256"]:
        raise ValueError("Poligoni cambiati: riesegui le quote delle strutture.")
STRUTTURE_PER_UNITA = STRUTTURE_EDITOR.read()
if set(STRUTTURE_PER_UNITA) != set(UNITA_COSTIERE):
    raise ValueError("Unità cambiate: ricrea il modulo delle strutture esposte.")
thresholds = STRUTTURE_PER_UNITA.get("whole", next(iter(STRUTTURE_PER_UNITA.values())))
for key, values in STRUTTURE_PER_UNITA.items():
    if key != "whole": SETTORI_SPIAGGIA[key]["thresholds"] = values
normalized=profili.rename(columns={COL_ID:"TR_ID",COL_DIST:"distance",COL_Z:"SAMPLE_1",COL_X:"X",COL_Y:"Y"})[["TR_ID","distance","SAMPLE_1","X","Y"]]
normalized.to_csv(WORK/"profili_normalizzati.csv",index=False)
location=dict(SITE_INPUT,slopes=PENDENZE,slope_sampling=SLOPE_SAMPLING,roughness_m=float(RUGOSITA_M),parameters_status="provisional",
    wave_exposure=parse_exposure(SETTORI_PROVENIENZA,NOTE_ESPOSIZIONE),
    datum_offset_m=float(OFFSET_LIVELLO_M),vertical_reference_status="provisional",
    climatology_file=f"config/climatology/{ID_SITO}.json" if CLIMATE is not None else None,
    summer_season=dict(start_month_day=INIZIO_ESTATE or None,end_month_day=FINE_ESTATE or None,status="configured" if INIZIO_ESTATE else "to_be_defined"),
    transects=scelti.to_dict(orient="records"),notes=NOTE_SITO)
if SETTORI_SPIAGGIA:
    location["sectors"] = copy.deepcopy(SETTORI_SPIAGGIA)
resolve_sectors(location, thresholds)
if SECTOR_ANALYSIS_SIGNATURE != ANALYSIS_SIGNATURE or (MAPPING_CONFIGS and MAP_ANALYSIS_SIGNATURE != ANALYSIS_SIGNATURE):
    raise ValueError("Profili cambiati: riesegui assegnazione dei settori e preparazione delle mappe.")
if CARTOGRAPHY_SIGNATURE != json.dumps(CARTOGRAFIE, sort_keys=True):
    raise ValueError("Cartografia cambiata: riesegui la preparazione prima di esportare.")
for unit, metadata in MAPPING_CONFIGS.items():
    if unit == "whole":
        location["mapping"] = metadata
    elif unit in location.get("sectors", {}):
        location["sectors"][unit]["mapping"] = metadata
    else:
        raise ValueError("Settore rimosso: riesegui la preparazione della cartografia.")
if not LIVELLO_STESSO_PUNTO:
    location["sealevel_point"]=dict(latitude=LAT_LIVELLO,longitude=LON_LIVELLO)
if SEA_LEVEL_REFERENCE is not None:
    if SEA_LEVEL_REFERENCE["year"]!=ANNO_RIFERIMENTO_SSH:
        raise ValueError("Anno del riferimento cambiato: riesegui la sezione del livello marino prima di esportare.")
    location=apply_reference(location,SEA_LEVEL_REFERENCE)
payload=dict(schema_version=1,slug=ID_SITO,data_mode="demo" if DEMO else "study",location=location,thresholds=thresholds,
    provenance=dict(created_at=iso(dt.datetime.now(dt.timezone.utc)),slope_method=PROFILE_FIT_METHOD,
        reference_notebook="Stockdon_runup_soluzione.ipynb, celle 24–25" if PROFILE_SOURCE_KIND=="DEM + linee" else "GNSS: intera battigia; distanza euclidea e regressione oppure pendenza fornita nel campo selezionato",profile_source=PROFILE_SOURCE,
        profile_sha256=hashlib.sha256((WORK/"profili_normalizzati.csv").read_bytes()).hexdigest(),
        profile_data_mode=PROFILE_DATA_MODE,climate_data_mode=CLIMATE_DATA_MODE if CLIMATE else None,
        sea_level_reference_year=SEA_LEVEL_REFERENCE["year"] if SEA_LEVEL_REFERENCE else None,
        geospatial_preprocessing=GEO_PREPROCESSING,structure_point_preprocessing=STRUCTURE_POINT_META,
        cartographic_line_preview=line_preview_metadata,
        structure_point_exclusions_confirmed=bool(ESCLUDI_PUNTI_STRUTTURE_NON_VALIDI) if STRUCTURE_POINT_META else None,profile_crs=crs.to_string(),vertical_reference=PROFILE_VERTICAL_REFERENCE,z_correction_m=PROFILE_Z_CORRECTION,
        excluded_transects=sorted(excluded),slope_sampling=MODALITA_PENDENZE,slope_sampling_by_unit=SLOPE_SAMPLING_BY_UNIT,z_min_m=Z_MIN,z_max_m=Z_MAX))
display(pd.DataFrame([{"località":NOME_SITO,"modalità":payload["data_mode"],"transetti inclusi":len(scelti),"pendenze":PENDENZE,"ERA5 o riferimento":CLIMATE["source"] if CLIMATE else "Non configurato","esposizione":location["wave_exposure"]["sectors_deg"] if location["wave_exposure"] else "Da definire","stagione estiva":f"{INIZIO_ESTATE} / {FINE_ESTATE}" if INIZIO_ESTATE else "Da definire"}]))
print("Le soglie e il raccordo delle quote restano da validare, anche con profili misurati.")

In [ ]:
#@title Crea e scarica il pacchetto
ESPORTA_PACCHETTO = True #@param {type:"boolean"}
if ESPORTA_PACCHETTO:
    attachments={name:WORK/name for name in ["pendenze_transetti.csv","profili_normalizzati.csv","profilo_selezionato.png"]}
    for name in ["climatologia.png","massimi_annuali.csv","estremi_settoriali.png"]:
        if CLIMATE is not None and (WORK/name).exists(): attachments[name]=WORK/name
    for name in ["estrazione_transetti.csv", "assegnazioni_settori.csv", "quote_strutture_punti.csv", "quote_strutture_riepilogo.csv", "pendenze_campionate.csv", "campionamento_pendenze.json", "campioni_dem_solo_avvii_mappa.csv"]:
        if (WORK/name).exists(): attachments[name]=WORK/name
    for unit in MAPPING_CONFIGS:
        seed_report = WORK/f"avvii_mare_{unit}.csv"
        if seed_report.exists(): attachments[seed_report.name] = seed_report
    if line_preview_metadata:
        attachments.update(LINE_PREVIEW_ATTACHMENTS)
    if GEO_TRANSECTS_SOURCE: attachments["transetti.geojson"]=GEO_TRANSECTS_SOURCE
    if AREAS_SOURCE: attachments["settori.geojson"]=AREAS_SOURCE
    if STRUCTURES_SOURCE: attachments["strutture.geojson"]=STRUCTURES_SOURCE
    output=WORK/f"{ID_SITO}_{'DEMO' if DEMO else 'studio'}.zip"
    write_site_package(output,payload,climatology=CLIMATE,attachments=attachments,cartography=CARTOGRAPHY_ASSETS)
    print("Pacchetto creato:",output)
    try:
        from google.colab import files
        files.download(str(output))
    except ImportError:
        display(FileLink(str(output)))

## 10. Importa il pacchetto nella repository
Dopo il download, dal terminale nella cartella `Runup_GUI`, per aggiornare una località già presente:

```sh
python scripts/import_site_package.py /percorso/borghetto-santo-spirito_studio.zip --replace
```

Per una località nuova ometti `--replace`. Il controllo della configurazione viene eseguito prima di scrivere i file. Per una prova locale con un pacchetto demo serve anche `--allow-demo`; non usarlo per pubblicare parametri di esempio come dati reali.

Dopo l’importazione rigenera il bollettino e verifica il risultato nel portale. Il download ERA5 rimane un’attività di setup: gli aggiornamenti giornalieri leggono il riferimento già salvato.

### Riferimenti
- Metodo della pendenza: notebook didattico fornito dall’utente **Stockdon_runup_soluzione.ipynb**, celle 24–25.
- Stockdon, H. F., Holman, R. A., Howd, P. A., & Sallenger, A. H. (2006). *Empirical parameterization of setup, swash, and runup*. Coastal Engineering, 53(7), 573–588. https://doi.org/10.1016/j.coastaleng.2005.12.005
- ERA5 hourly time-series: https://doi.org/10.24381/1cf1ad76. Il downloader adatta lo script fornito dall’utente, copyright 2026 Alessio Rovere, MIT; la licenza è inclusa nella cella tecnica.
- [Colab: caricamento, salvataggio e condivisione dei notebook](https://research.google.com/colaboratory/faq.html).